# Agentic IFRS S1/S2 Report Generation Pipeline — Production final quality reconciliation engine

## Robust JSON handling

This version fixes a pipeline crash where the claims-register agent returned malformed or truncated JSON. The JSON parser now saves malformed raw outputs, attempts automatic repair with the strong model, and the claims-register builder has a deterministic fallback so the run can continue to deterministic gates or human review instead of stopping with `JSONDecodeError`.


## Strict evidence/scoring implementation

Added strict NaN/null/generic-field filtering, improved Strategy routing, missing-requirement audit flags, section-generation scores, and a final-report cleanliness block that prevents missing-data wording from entering the approved report.

In [1]:
# ============================================================
# CELL 1 — SETUP PATHS AND CONFIG
# Notebook expected location: /notebooks
# Style system expected at : /notebooks/gen_data/style/style_system
# ============================================================

import os
from pydoc import resolve
import re
import json
import time
import uuid
import shutil
import random
import urllib.request
import urllib.error
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple
from collections import defaultdict, Counter

import pandas as pd

try:
    from dotenv import load_dotenv, find_dotenv
except ImportError:
    raise ImportError("Install python-dotenv first: pip install python-dotenv")

try:
    load_dotenv(find_dotenv(usecwd=True), override=True)
except TypeError:
    load_dotenv(find_dotenv(), override=True)
except AssertionError:
    # Some non-interactive runners cannot inspect call frames for find_dotenv().
    load_dotenv(Path.cwd() / ".env", override=True)

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name == "notebooks":
    NOTEBOOK_DIR = CURRENT_DIR
elif (CURRENT_DIR / "notebooks").exists():
    NOTEBOOK_DIR = (CURRENT_DIR / "notebooks").resolve()
else:
    NOTEBOOK_DIR = CURRENT_DIR

GEN_DATA_DIR = NOTEBOOK_DIR / "gen_data"

# Input folders. Override with env vars if your structure differs.
PAYLOAD_DIR = Path(os.getenv("PAYLOAD_DIR", GEN_DATA_DIR / "payloads_risk")).resolve()
REQUIREMENTS_DIR = Path(os.getenv("IFRS_REQUIREMENTS_DIR", GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "section_by_section_requirements" / "json")).resolve()
STYLE_SYSTEM_DIR = Path(os.getenv("STYLE_SYSTEM_DIR", GEN_DATA_DIR / "style" / "style_system")).resolve()

# Output folder.
OUTPUT_DIR = Path(os.getenv("GENERATION_OUTPUT_DIR", GEN_DATA_DIR / "generated_reports" / "agentic_ifrs_report")).resolve()

# Pipeline controls.
PIPELINE_MODE = os.getenv("PIPELINE_MODE", "synthetic_demo")
FORBID_INVENTION = True  # hard invariant, not a configurable switch
ALLOW_PARTIAL_COVERAGE = os.getenv("ALLOW_PARTIAL_COVERAGE", "true").lower() == "true"
USE_FUZZY_EVIDENCE_MAPPER = os.getenv("USE_FUZZY_EVIDENCE_MAPPER", "false").lower() == "true"
MAX_REVISION_LOOPS = int(os.getenv("MAX_REVISION_LOOPS", "2"))

# Section order used by the final report.
SECTIONS = [
    "General Requirements",
    "Governance",
    "Strategy",
    "Risk Management",
    "Metrics and Targets",
]

SECTION_SLUGS = {
    "General Requirements": "general_requirements",
    "Governance": "governance",
    "Strategy": "strategy",
    "Risk Management": "risk_management",
    "Metrics and Targets": "metrics_and_targets",
}

# Output subfolders.
DIRS = {
    "evidence_maps": OUTPUT_DIR / "01_evidence_maps",
    "coverage": OUTPUT_DIR / "02_coverage",
    "missing_requirements": OUTPUT_DIR / "03_missing_requirements",
    "plans": OUTPUT_DIR / "04_disclosure_plans",
    "drafts": OUTPUT_DIR / "05_draft_sections",
    "claims": OUTPUT_DIR / "06_claims_registers",
    "gates": OUTPUT_DIR / "07_deterministic_gates",
    "judges": OUTPUT_DIR / "08_judge_results",
    "revisions": OUTPUT_DIR / "09_revised_sections",
    "approved": OUTPUT_DIR / "10_approved_sections",
    "connectivity": OUTPUT_DIR / "11_connectivity",
    "handoff": OUTPUT_DIR / "12_pdf_handoff",
    "audit_logs": OUTPUT_DIR / "audit_logs",
}

for folder in DIRS.values():
    folder.mkdir(parents=True, exist_ok=True)

print("Current working directory:", CURRENT_DIR)
print("Notebook directory:", NOTEBOOK_DIR)
print("Payload directory:", PAYLOAD_DIR)
print("Requirements directory:", REQUIREMENTS_DIR)
print("Style system directory:", STYLE_SYSTEM_DIR)
print("Output directory:", OUTPUT_DIR)
print("Pipeline mode:", PIPELINE_MODE)
print("Forbid invention:", FORBID_INVENTION)
print("Use fuzzy mapper:", USE_FUZZY_EVIDENCE_MAPPER)

Current working directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Notebook directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks
Payload directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads_risk
Requirements directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json
Style system directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\style\style_system
Output directory: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report
Pipeline mode: synthetic_demo
Forbid invention: True
Use fuzzy mapper: False


In [2]:
# ============================================================
# CELL 0B — LLM COST / LATENCY TELEMETRY (EVALUATION)
# ============================================================
# Why this cell exists (Efficiency metrics):
#   Every LLM call in the pipeline funnels through azure_chat(), which now calls
#   record_llm_call() with the provider usage block and measured latency. This
#   gives us tokens, call counts, per-tier cost, latency percentiles, and a
#   per-request-label breakdown for the evaluation scorecard, with no change to
#   agent logic. Pricing is configurable via env so cost figures track reality.
# ============================================================

# USD per 1K tokens, per model tier. Override with IFRS_PRICE_<TIER>_IN / _OUT.
_LLM_PRICE_PER_1K = {
    "strong": {
        "in": float(os.getenv("IFRS_PRICE_STRONG_IN", "0.0")),
        "out": float(os.getenv("IFRS_PRICE_STRONG_OUT", "0.0")),
    },
    "fast": {
        "in": float(os.getenv("IFRS_PRICE_FAST_IN", "0.0")),
        "out": float(os.getenv("IFRS_PRICE_FAST_OUT", "0.0")),
    },
}


class LLMTelemetry:
    """Accumulates per-call LLM usage for cost/latency evaluation metrics."""

    def __init__(self):
        self.calls = []

    def record(self, model_tier, request_label, usage, latency_s):
        prompt_tokens = completion_tokens = total_tokens = 0
        if isinstance(usage, dict):
            prompt_tokens = int(usage.get("prompt_tokens", 0) or 0)
            completion_tokens = int(usage.get("completion_tokens", 0) or 0)
            total_tokens = int(usage.get("total_tokens", prompt_tokens + completion_tokens) or 0)
        price = _LLM_PRICE_PER_1K.get(model_tier, {"in": 0.0, "out": 0.0})
        cost = (prompt_tokens / 1000.0) * price["in"] + (completion_tokens / 1000.0) * price["out"]
        self.calls.append({
            "model_tier": model_tier,
            "request_label": request_label,
            "prompt_tokens": prompt_tokens,
            "completion_tokens": completion_tokens,
            "total_tokens": total_tokens,
            "latency_s": round(float(latency_s), 3),
            "cost_usd": round(cost, 6),
        })

    def summary(self):
        n = len(self.calls)
        if n == 0:
            return {"llm_calls": 0}
        lat = sorted(c["latency_s"] for c in self.calls)

        def pct(p):
            if not lat:
                return 0.0
            k = min(len(lat) - 1, int(round((p / 100.0) * (len(lat) - 1))))
            return lat[k]

        by_tier, by_label = {}, {}
        for c in self.calls:
            for bucket, key in ((by_tier, c["model_tier"]), (by_label, c["request_label"])):
                b = bucket.setdefault(key, {"calls": 0, "total_tokens": 0, "cost_usd": 0.0, "latency_s": 0.0})
                b["calls"] += 1
                b["total_tokens"] += c["total_tokens"]
                b["cost_usd"] = round(b["cost_usd"] + c["cost_usd"], 6)
                b["latency_s"] = round(b["latency_s"] + c["latency_s"], 3)
        return {
            "llm_calls": n,
            "prompt_tokens": sum(c["prompt_tokens"] for c in self.calls),
            "completion_tokens": sum(c["completion_tokens"] for c in self.calls),
            "total_tokens": sum(c["total_tokens"] for c in self.calls),
            "total_cost_usd": round(sum(c["cost_usd"] for c in self.calls), 6),
            "total_latency_s": round(sum(c["latency_s"] for c in self.calls), 2),
            "latency_p50_s": pct(50),
            "latency_p95_s": pct(95),
            "by_model_tier": by_tier,
            "by_request_label": by_label,
        }

    def reset(self):
        self.calls = []


LLM_TELEMETRY = LLMTelemetry()


def record_llm_call(model_tier, request_label, usage, latency_s):
    LLM_TELEMETRY.record(model_tier, request_label, usage, latency_s)


print("LLM telemetry recorder ready (record_llm_call / LLM_TELEMETRY).")


LLM telemetry recorder ready (record_llm_call / LLM_TELEMETRY).


## Azure/OpenAI helper

This cell uses the same full deployment URL style as the working notebook you provided, but keeps this pipeline's model routing:

```env
AZURE_OPENAI_API_KEY=...

AZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full strong GPT-5.2 chat-completions URL>
AZURE_OPENAI_FAST_DEPLOYMENT_URL=<full fast chat-completions URL>
```

The notebook does **not** build or modify endpoint URLs from deployment names. It sends the full URL exactly as configured, after basic quote/markdown cleanup.


In [3]:
# ============================================================
# CELL 2 — LLM CLIENT
# Full deployment URL logic, matching the working REST style.
#
# Uses:
# - AZURE_OPENAI_GPT52_DEPLOYMENT_URL for strong agents
# - AZURE_OPENAI_FAST_DEPLOYMENT_URL for fast/light agents
#
# Important:
# This cell does NOT construct Azure URLs from endpoint + deployment.
# It sends the configured full deployment URL directly.
# ============================================================

import http.client

AZURE_OPENAI_API_KEY = (
    os.getenv("AZURE_OPENAI_API_KEY")
    or os.getenv("OPENAI_API_KEY")
)

# Full Azure / enterprise-gateway chat-completions URLs.
AZURE_OPENAI_GPT52_DEPLOYMENT_URL = os.getenv("AZURE_OPENAI_GPT52_DEPLOYMENT_URL")
AZURE_OPENAI_FAST_DEPLOYMENT_URL = os.getenv("AZURE_OPENAI_FAST_DEPLOYMENT_URL")


def _clean_url(value: Optional[str]) -> Optional[str]:
    """
    Basic cleanup for full deployment URLs.

    Keeps the full URL as provided; does not add/replace api-version.
    Handles:
    - surrounding quotes
    - accidental markdown link format: [label](https://...)
    - accidental copied bracket+url format
    """
    if not value:
        return None

    value = str(value).strip().strip('"').strip("'").strip()

    # Markdown link: [label](https://actual-url)
    md_match = re.search(r"\]\((https://[^)\s]+)\)", value)
    if md_match:
        value = md_match.group(1).strip()

    # Copied format that contains multiple https:// occurrences.
    # Keep the last URL-like occurrence, which is usually the actual href.
    https_positions = [m.start() for m in re.finditer(r"https://", value)]
    if https_positions:
        value = value[https_positions[-1]:]

    value = value.strip().strip("[]").strip()
    value = value.rstrip(").,;")

    return value


AZURE_OPENAI_GPT52_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_GPT52_DEPLOYMENT_URL)
AZURE_OPENAI_FAST_DEPLOYMENT_URL = _clean_url(AZURE_OPENAI_FAST_DEPLOYMENT_URL)

# Fast deployment falls back to strong if not configured.
if not AZURE_OPENAI_FAST_DEPLOYMENT_URL:
    AZURE_OPENAI_FAST_DEPLOYMENT_URL = AZURE_OPENAI_GPT52_DEPLOYMENT_URL


MODEL_CONFIG = {
    "fuzzy_evidence_mapper": "fast",
    "section_writer": "strong",
    "claims_register_builder": "strong",
    "ifrs_coverage_judge": "strong",
    "evidence_judge": "strong",
    "style_judge": "fast",
    "minimal_reviser": "strong",
    "whole_report_connectivity_judge": "strong",
}


def _mask_url_for_display(url: Optional[str]) -> str:
    """Mask full endpoint URL while keeping enough shape for diagnostics."""
    if not url:
        return "NOT CONFIGURED"

    try:
        import urllib.parse
        parsed = urllib.parse.urlparse(url)

        host = parsed.netloc
        if host:
            host_parts = host.split(".")
            if host_parts and len(host_parts[0]) > 6:
                host_parts[0] = host_parts[0][:3] + "***" + host_parts[0][-2:]
            host = ".".join(host_parts)

        path = parsed.path
        path = re.sub(
            r"(/deployments/)([^/]+)(/chat/completions)",
            lambda m: m.group(1) + m.group(2)[:2] + "***" + m.group(3),
            path,
        )

        # Avoid displaying the raw query because it can make notebook output messy.
        query = "..." if parsed.query else ""

        return urllib.parse.urlunparse((parsed.scheme, host, path, "", query, ""))

    except Exception:
        return "<configured URL, masking failed>"


def validate_llm_config() -> None:
    required = {
        "AZURE_OPENAI_API_KEY": AZURE_OPENAI_API_KEY,
        "AZURE_OPENAI_GPT52_DEPLOYMENT_URL": AZURE_OPENAI_GPT52_DEPLOYMENT_URL,
        "AZURE_OPENAI_FAST_DEPLOYMENT_URL": AZURE_OPENAI_FAST_DEPLOYMENT_URL,
    }

    missing = [name for name, value in required.items() if not value]

    if missing:
        flags = {
            "api_key_loaded": bool(AZURE_OPENAI_API_KEY),
            "gpt52_url_loaded": bool(AZURE_OPENAI_GPT52_DEPLOYMENT_URL),
            "fast_url_loaded": bool(AZURE_OPENAI_FAST_DEPLOYMENT_URL),
        }
        raise ValueError(
            "Missing Azure/OpenAI full-URL configuration values: "
            + ", ".join(missing)
            + "\n\nLoaded flags, keys are never printed:\n"
            + json.dumps(flags, indent=2)
            + "\n\nRequired .env:\n"
              "AZURE_OPENAI_API_KEY=<shared Azure resource key>\n"
              "AZURE_OPENAI_GPT52_DEPLOYMENT_URL=<full GPT-5.2 deployment URL>\n"
              "AZURE_OPENAI_FAST_DEPLOYMENT_URL=<full fast deployment URL>\n"
        )

    for name, url in {
        "AZURE_OPENAI_GPT52_DEPLOYMENT_URL": AZURE_OPENAI_GPT52_DEPLOYMENT_URL,
        "AZURE_OPENAI_FAST_DEPLOYMENT_URL": AZURE_OPENAI_FAST_DEPLOYMENT_URL,
    }.items():
        if not str(url).startswith("https://"):
            raise ValueError(f"{name} must be a full HTTPS deployment URL: {url!r}")

        if "/chat/completions" not in str(url):
            raise ValueError(
                f"{name} does not look like a chat-completions URL.\n"
                f"Configured URL shape: {_mask_url_for_display(url)}\n\n"
                "Expected a full URL ending with /chat/completions plus any required query string."
            )


validate_llm_config()

print("Azure/OpenAI full-URL configuration loaded")
print("Strong endpoint:", _mask_url_for_display(AZURE_OPENAI_GPT52_DEPLOYMENT_URL))
print("Fast endpoint:", _mask_url_for_display(AZURE_OPENAI_FAST_DEPLOYMENT_URL))
print("Model routing:", json.dumps(MODEL_CONFIG, indent=2))


def get_model_url(model_tier: str = "strong") -> str:
    model_tier = (model_tier or "strong").lower().strip()
    if model_tier == "fast":
        return AZURE_OPENAI_FAST_DEPLOYMENT_URL
    return AZURE_OPENAI_GPT52_DEPLOYMENT_URL


def _extract_message_content(data: Dict[str, Any]) -> str:
    try:
        content = data["choices"][0]["message"]["content"]
    except (KeyError, IndexError, TypeError) as exc:
        raise ValueError(
            "Unexpected Azure/OpenAI response structure:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        ) from exc

    if not content:
        raise ValueError(
            "Azure/OpenAI returned an empty message content. Response:\n"
            + json.dumps(data, indent=2, ensure_ascii=False)[:3000]
        )

    return content


def _azure_chat_completion(
    *,
    url: str,
    api_key: str,
    messages: List[Dict[str, str]],
    max_output_tokens: int,
    json_mode: bool = False,
    temperature: Optional[float] = None,
    timeout: int = 240,
    request_label: str = "LLM",
    max_attempts: int = 6,
) -> Dict[str, Any]:
    """
    Robust REST call for Azure/OpenAI-compatible enterprise gateways.

    Mirrors the working logic you provided:
    - Uses full deployment URL directly.
    - Retries transient 500/502/503/504 and connection errors.
    - Handles 429 Retry-After.
    - Tries max_completion_tokens first, then max_tokens for gateway compatibility.
    - Does not expose API keys in errors.
    """

    token_fields = ["max_completion_tokens", "max_tokens"]
    last_error = None

    for token_field in token_fields:
        payload = {
            "messages": messages,
            token_field: max_output_tokens,
        }

        if temperature is not None:
            payload["temperature"] = temperature

        if json_mode:
            payload["response_format"] = {"type": "json_object"}

        for attempt in range(1, max_attempts + 1):
            req = urllib.request.Request(
                url,
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "api-key": api_key,
                },
                method="POST",
            )

            try:
                with urllib.request.urlopen(req, timeout=timeout) as resp:
                    return json.loads(resp.read().decode("utf-8"))

            except urllib.error.HTTPError as exc:
                body = exc.read().decode(errors="replace")
                last_error = RuntimeError(
                    f"{request_label} HTTP error {exc.code}.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Token field used: {token_field}\n"
                    f"Response: {body[:3000]}"
                )

                rate_limited = exc.code == 429
                transient = exc.code in {500, 502, 503, 504}
                compatibility_candidate = (
                    token_field == "max_completion_tokens"
                    and exc.code in {400, 422, 500}
                )

                if rate_limited and attempt < max_attempts:
                    retry_after = None
                    try:
                        ra = exc.headers.get("Retry-After") if exc.headers else None
                        if ra is not None:
                            retry_after = float(str(ra).strip())
                    except (TypeError, ValueError):
                        retry_after = None

                    wait = retry_after if retry_after is not None else (2 ** attempt) * 2 + random.random()
                    wait = min(wait, 90)
                    print(
                        f"{request_label}: rate limited (429); retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if transient and attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: server error {exc.code}; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                if compatibility_candidate:
                    print(
                        f"{request_label}: gateway may not support "
                        "`max_completion_tokens`; retrying with `max_tokens`."
                    )
                    break

                if exc.code == 404:
                    raise RuntimeError(
                        f"{request_label} HTTP 404 Resource not found.\n"
                        f"Endpoint: {_mask_url_for_display(url)}\n\n"
                        "The notebook is now sending the configured full URL directly. "
                        "So a 404 means the URL itself is not accepted by the gateway, "
                        "or the deployment behind that URL is not accessible with this key.\n\n"
                        "Compare the exact .env value of AZURE_OPENAI_GPT52_DEPLOYMENT_URL "
                        "with the endpoint URL that works in your other notebook."
                    ) from exc

                raise last_error from exc

            except urllib.error.URLError as exc:
                last_error = RuntimeError(
                    f"{request_label} connection error.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Error: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection issue; retrying "
                        f"attempt {attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

            except (ConnectionError, TimeoutError, OSError, http.client.RemoteDisconnected) as exc:
                last_error = RuntimeError(
                    f"{request_label} connection reset/timeout.\n"
                    f"Endpoint: {_mask_url_for_display(url)}\n"
                    f"Error: {type(exc).__name__}: {exc}"
                )

                if attempt < max_attempts:
                    wait = min(2 ** (attempt - 1) + random.random(), 12)
                    print(
                        f"{request_label}: connection reset/timeout "
                        f"({type(exc).__name__}); retrying attempt "
                        f"{attempt + 1}/{max_attempts} in {wait:.1f}s..."
                    )
                    time.sleep(wait)
                    continue

                raise last_error from exc

    raise last_error or RuntimeError(f"{request_label} request failed for an unknown reason.")



def _strip_markdown_json_fence(text: str) -> str:
    """Remove common ```json fences without touching the JSON body."""
    text = str(text or "").strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()


def _extract_balanced_json_object(text: str) -> Optional[str]:
    """
    Return the first balanced JSON object found in text.

    This is safer than taking text[first_brace:last_brace] because model output can
    contain explanatory text, braces inside strings, or multiple JSON-looking blocks.
    If the object is truncated and never balances, return None so the caller can
    attempt LLM repair on the best candidate.
    """
    start = None
    depth = 0
    in_string = False
    escape = False

    for i, ch in enumerate(text):
        if start is None:
            if ch == "{":
                start = i
                depth = 1
            continue

        if escape:
            escape = False
            continue

        if ch == "\\":
            escape = True
            continue

        if ch == '"':
            in_string = not in_string
            continue

        if in_string:
            continue

        if ch == "{":
            depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0:
                return text[start:i + 1]

    return None


def _extract_json_object(text: str) -> str:
    """
    Extract the most likely JSON object from model output.

    Handles markdown fences and leading/trailing commentary. If the output appears
    truncated, returns the partial object candidate so the repair step can fix it.
    """
    text = _strip_markdown_json_fence(text)

    # Fast path: already a clean JSON object.
    if text.startswith("{") and text.endswith("}"):
        return text

    balanced = _extract_balanced_json_object(text)
    if balanced:
        return balanced

    first = text.find("{")
    last = text.rfind("}")
    if first >= 0 and last > first:
        return text[first:last + 1]
    if first >= 0:
        # Truncated object. Return from first brace onward for repair.
        return text[first:]

    return text


def _json_error_context(candidate: str, exc: json.JSONDecodeError, radius: int = 300) -> str:
    """Small excerpt around a JSONDecodeError location for debugging."""
    pos = getattr(exc, "pos", 0)
    left = max(0, pos - radius)
    right = min(len(candidate), pos + radius)
    excerpt = candidate[left:right]
    pointer = " " * max(0, pos - left) + "^"
    return excerpt + "\n" + pointer


def _safe_debug_filename(label: str) -> str:
    label = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(label or "llm_json"))
    return label.strip("_")[:80] or "llm_json"


def _write_llm_json_debug(raw: str, label: str = "malformed_json") -> Optional[Path]:
    """
    Persist malformed raw LLM output for inspection.
    Uses the notebook audit_logs folder when available.
    """
    try:
        base = DIRS.get("audit_logs", OUTPUT_DIR) if "DIRS" in globals() else Path.cwd()
        base = Path(base) / "llm_json_debug"
        base.mkdir(parents=True, exist_ok=True)
        path = base / f"{time.strftime('%Y%m%d_%H%M%S')}_{_safe_debug_filename(label)}_{uuid.uuid4().hex[:8]}.txt"
        path.write_text(str(raw), encoding="utf-8")
        return path
    except Exception:
        return None


def _repair_json_with_strong_model(malformed_content: str, request_label: str) -> Dict[str, Any]:
    repair_system = (
        "You repair malformed or truncated JSON. "
        "Return one complete valid JSON object only. "
        "Preserve the original meaning, scores, checklist values, issues, and fixes. "
        "Keep strings concise. Do not add markdown fences or commentary."
    )

    repair_user = f"""
Repair the following malformed or truncated output into one complete valid JSON object.

Requirements:
- Keep the same top-level fields when present.
- Finish incomplete strings and arrays conservatively.
- Fix missing commas, unescaped quotes, dangling keys, and truncated arrays.
- Limit each issue/fix/support note string to at most 35 words.
- If the object is a claims register, preserve as many claims as possible but cap at 60 claims.
- Return JSON only.

MALFORMED OUTPUT:
{str(malformed_content)[:70000]}
""".strip()

    data = _azure_chat_completion(
        url=AZURE_OPENAI_GPT52_DEPLOYMENT_URL,
        api_key=AZURE_OPENAI_API_KEY,
        messages=[
            {"role": "system", "content": repair_system},
            {"role": "user", "content": repair_user},
        ],
        max_output_tokens=int(os.getenv("JSON_REPAIR_MAX_TOKENS", "6000")),
        json_mode=True,
        temperature=0,
        request_label=f"{request_label} JSON repair",
    )

    repaired = _extract_message_content(data)
    candidate = _extract_json_object(repaired)
    return json.loads(candidate)


def _parse_or_repair_json(raw: str, request_label: str = "LLM output") -> Dict[str, Any]:
    """
    Parse JSON returned by an LLM. If parsing fails, save the raw output and ask
    the strong model to repair it. This prevents one malformed JSON response from
    crashing the full generation pipeline.
    """
    candidate = _extract_json_object(raw)

    try:
        return json.loads(candidate)
    except json.JSONDecodeError as exc:
        debug_path = _write_llm_json_debug(raw, request_label)
        print(
            f"{request_label}: invalid JSON at line {exc.lineno}, column {exc.colno}. "
            "Attempting JSON repair..."
        )
        if debug_path:
            print("Raw malformed output saved to:", debug_path)
        try:
            return _repair_json_with_strong_model(candidate, request_label=request_label)
        except Exception as repair_exc:
            detail = _json_error_context(candidate, exc)
            raise ValueError(
                f"{request_label}: failed to parse JSON and repair also failed.\n"
                f"Original JSON error: {exc}\n"
                f"Debug file: {debug_path}\n"
                f"Error context:\n{detail}"
            ) from repair_exc

def azure_chat(
    messages: List[Dict[str, str]],
    model_tier: str = "strong",
    temperature: float = 0,
    max_tokens: int = 4000,
    response_format: Optional[Dict[str, str]] = None,
    retries: int = 6,
    retry_sleep: int = 3,
) -> str:
    """
    Azure/OpenAI Chat Completions helper used by all LLM agents.

    Returns text content.
    If response_format={"type": "json_object"}, the model is asked for JSON mode.
    """
    url = get_model_url(model_tier)
    request_label = f"Azure {model_tier} agent"

    json_mode = bool(response_format and response_format.get("type") == "json_object")

    _llm_call_started = time.time()
    data = _azure_chat_completion(
        url=url,
        api_key=AZURE_OPENAI_API_KEY,
        messages=messages,
        max_output_tokens=max_tokens,
        json_mode=json_mode,
        temperature=temperature,
        request_label=request_label,
        max_attempts=retries,
    )

    # [Evaluation] Record cost/latency telemetry for every LLM call (see CELL 0B).
    if "record_llm_call" in globals():
        try:
            record_llm_call(
                model_tier=model_tier,
                request_label=request_label,
                usage=(data or {}).get("usage") if isinstance(data, dict) else None,
                latency_s=time.time() - _llm_call_started,
            )
        except Exception:
            pass

    return _extract_message_content(data)



def azure_chat_json(
    messages: List[Dict[str, str]],
    model_tier: str = "strong",
    temperature: float = 0,
    max_tokens: int = 4000,
    retries: int = 6,
    request_label: Optional[str] = None,
) -> Dict[str, Any]:
    """
    JSON-safe LLM call.
    First requests JSON mode. If the model returns malformed or truncated JSON,
    parse_json_response repairs it with the strong model.
    """
    label = request_label or f"Azure {model_tier} agent"
    content = azure_chat(
        messages=messages,
        model_tier=model_tier,
        temperature=temperature,
        max_tokens=max_tokens,
        response_format={"type": "json_object"},
        retries=retries,
    )
    return parse_json_response(content, request_label=label)


def parse_json_response(raw: str, request_label: str = "LLM output") -> Dict[str, Any]:
    """
    Backward-compatible parser for cells that call azure_chat(...json mode...).
    Now robust: strict parse first, then automatic repair instead of a hard crash.
    """
    return _parse_or_repair_json(raw, request_label=request_label)


def parse_json_safely(raw: str) -> Dict[str, Any]:
    try:
        return parse_json_response(raw)
    except Exception:
        return {
            "parse_error": True,
            "raw_output_preview": str(raw)[:2000],
        }


def test_llm_connection(model_tier: str = "strong") -> None:
    """Quick smoke test for a configured endpoint."""
    print(f"Testing {model_tier} endpoint:", _mask_url_for_display(get_model_url(model_tier)))

    content = azure_chat(
        [{"role": "user", "content": "Return exactly: OK"}],
        model_tier=model_tier,
        temperature=0,
        max_tokens=20,
    )

    print(f"{model_tier} response:", content)


print("Full-URL LLM helper functions ready")
print("Run test_llm_connection('strong') and test_llm_connection('fast') before running the full pipeline.")


Azure/OpenAI full-URL configuration loaded
Strong endpoint: https://eyq***or.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gp***/chat/completions
Fast endpoint: https://eyq***or.europe.fabric.ey.com/eyq/eu/api/openai/deployments/gp***/chat/completions
Model routing: {
  "fuzzy_evidence_mapper": "fast",
  "section_writer": "strong",
  "claims_register_builder": "strong",
  "ifrs_coverage_judge": "strong",
  "evidence_judge": "strong",
  "style_judge": "fast",
  "minimal_reviser": "strong",
  "whole_report_connectivity_judge": "strong"
}
Full-URL LLM helper functions ready
Run test_llm_connection('strong') and test_llm_connection('fast') before running the full pipeline.


In [4]:
# ============================================================
# CELL 3 — GENERAL UTILITIES
# ============================================================


def slugify(value: str) -> str:
    value = value.strip().lower()
    value = re.sub(r"[^a-z0-9]+", "_", value)
    return value.strip("_")


def read_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        if default is not None:
            return default
        raise FileNotFoundError(path)
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def write_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def read_text(path: Path, default: str = "") -> str:
    if not path.exists():
        return default
    return path.read_text(encoding="utf-8", errors="replace")


def write_text(text: str, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding="utf-8")


def flatten_json(obj: Any, prefix: str = "") -> Dict[str, Any]:
    """Flatten nested dict/list into path -> scalar/list/dict value."""
    out = {}

    if isinstance(obj, dict):
        for k, v in obj.items():
            new_prefix = f"{prefix}.{k}" if prefix else str(k)
            out.update(flatten_json(v, new_prefix))
    elif isinstance(obj, list):
        if not obj:
            out[prefix] = []
        else:
            for i, v in enumerate(obj):
                new_prefix = f"{prefix}[{i}]"
                out.update(flatten_json(v, new_prefix))
    else:
        out[prefix] = obj

    return out


def get_by_path(obj: Any, path: Any) -> Any:
    """
    Resolve payload paths like a.b[0].c.

    Robustness added:
    - If the LLM returns an evidence source as a dict, try common path keys.
    - If the source is not a string/path-like object, return None instead of crashing.
    - Accept paths copied with a leading "$.".
    """
    if path is None:
        return None

    if isinstance(path, dict):
        for key in ("payload_path", "path", "evidence_path", "source_path", "payloadPath"):
            value = path.get(key)
            if isinstance(value, str) and value.strip():
                path = value
                break
        else:
            return None

    if not isinstance(path, (str, bytes)):
        return None

    path = str(path).strip()
    if not path:
        return None

    if path.startswith("$."):
        path = path[2:]
    elif path.startswith("$"):
        path = path[1:].lstrip(".")

    cur = obj
    tokens = re.findall(r"([^\.\[\]]+)|(\[(\d+)\])", path)
    for name, _, idx in tokens:
        if name:
            if not isinstance(cur, dict) or name not in cur:
                return None
            cur = cur[name]
        elif idx:
            i = int(idx)
            if not isinstance(cur, list) or i >= len(cur):
                return None
            cur = cur[i]
    return cur


def value_preview(value: Any, limit: int = 260) -> str:
    text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else str(value)
    text = re.sub(r"\s+", " ", text).strip()
    return text[:limit] + ("..." if len(text) > limit else "")


def is_empty_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, str) and not value.strip():
        return True
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False


def tokens(text: str) -> List[str]:
    return [t.lower() for t in re.findall(r"[A-Za-z][A-Za-z0-9_\-]+", str(text)) if len(t) > 2]


def normalize_bool(value: Any) -> bool:
    if isinstance(value, bool):
        return value
    return str(value).strip().lower() in {"true", "1", "yes", "mandatory"}


def now_id() -> str:
    return uuid.uuid4().hex[:10]


In [5]:
# ============================================================
# CELL 5B — TWO-TIER LIMITATION-LANGUAGE POLICY + STRUCTURAL HYGIENE
# ============================================================
# Why this cell exists (Fix 1 + part of Fix 3):
#
# Earlier layers banned ALL absence/limitation wording at five points in the
# pipeline. That over-rotated: IFRS S1/S2 REQUIRES certain limitation statements
# (relief invocation for anticipated financial effects, the Scope 3 category
# basis, estimation/measurement uncertainty, assurance level). The blanket ban
# made the report silently omit disclosures it is obliged to make, and it made
# the retained `data_gaps` payload table impossible to surface.
#
# Policy: split limitation language into two tiers.
#   TIER 1 — pipeline-internal wording that must NEVER appear in the report
#            (payload/synthetic/placeholder/audit-only/template/debug ...).
#            This stays an absolute, unconditional block.
#   TIER 2 — IFRS-sanctioned limitation wording (not available / not reported /
#            excluded / data gap / estimation uncertainty ...). Allowed ONLY
#            inside an IFRS-sanctioned sentence context; blocked otherwise.
#
# A single scanner, scan_limitation_language(), is the source of truth for all
# gates so they cannot drift apart again.
#
# This cell also defines remove_empty_heading_blocks() (part of Fix 3): a
# deterministic pass that deletes any heading whose body was emptied by the
# sentence-removal sanitizers, which is what produced the orphaned
# "Operational emissions calculation integrity (fleet correction)" heading.
# ============================================================

# ---- TIER 1: pipeline-internal wording, unconditionally forbidden in the report ----
TIER1_PIPELINE_INTERNAL_PHRASES = [
    "payload", "synthetic", "audit-only", "audit only",
    "placeholder", "[insert", "template row", "generic template",
    "source content", "provided source content", "no source content",
    "missing requirement", "missing requirements",
    "human review", "human_review",
    "do not treat as verified",
    "correction was applied", "data-preparation", "data preparation step",
    "debug", "pre-computed", "precomputed",
]

# ---- TIER 2: limitation wording that IFRS may legitimately require ----
# Allowed ONLY when the surrounding sentence is an IFRS-sanctioned limitation
# statement (see IFRS_SANCTIONED_LIMITATION_PATTERNS); otherwise it is treated
# as an unsupported absence claim and flagged/removed exactly as before.
TIER2_LIMITATION_PHRASES = [
    "not available", "unavailable", "not reported", "not provided",
    "not disclosed", "not separately specified", "not separately tracked",
    "not separately disclosed", "data gap", "data gaps",
    "excluded", "not included", "incomplete data", "not quantified",
    "cannot be quantified", "not estimated",
]

# Sentence-level regexes that make a Tier-2 phrase legitimate. These map to
# specific IFRS S1/S2 requirements the report is obliged to satisfy.
IFRS_SANCTIONED_LIMITATION_PATTERNS = [
    # Relief for anticipated financial effects: IFRS S2 paragraphs 19-21 / S1 B38-B40.
    r"appl(?:y|ies|ied)\s+the\s+(?:relief|exemption)\s+in\s+paragraph\s+\d+",
    r"(?:relief|exemption)\s+(?:in|under)\s+(?:paragraph\s+\d+\s+of\s+)?IFRS\s+S[12]",
    r"impracticable\s+to\s+(?:provide|quantify|estimate)\b",
    r"without\s+undue\s+cost\s+or\s+effort",
    r"(?:reasonable|supportable)\s+information\s+(?:is\s+)?(?:not\s+)?available\s+without",
    # Scope 3 category basis: IFRS S2 paragraph 29(a)(iv).
    r"scope\s*3\s+categor(?:y|ies)\s+(?:included|excluded|comprise|are\s+limited\s+to|assessed)",
    r"(?:includes?|excludes?|limited\s+to)\s+scope\s*3\s+categor(?:y|ies)",
    r"(?:other|remaining)\s+scope\s*3\s+categor(?:y|ies)\s+(?:were|are|have\s+been)\s+assessed",
    # Estimation / measurement uncertainty: IFRS S1 estimation-uncertainty disclosure.
    r"(?:estimation|measurement)\s+uncertaint(?:y|ies)",
    r"sources?\s+of\s+(?:estimation|measurement)\s+uncertaint",
    r"subject\s+to\s+(?:estimation|measurement)\s+uncertaint",
    r"data\s+quality\s+(?:score|tier|mix|characteristics|register)",
    # Assurance context.
    r"(?:limited|reasonable)\s+assurance",
    r"(?:has|have)\s+not\s+been\s+(?:externally\s+)?assured",
    # Carbon-price scope negatives that were already whitelisted.
    r"does\s+not\s+apply\s+to\s+(?:financed\s+emissions|lending\s+decisions)",
]

_IFRS_SANCTIONED_LIMITATION_RE = [re.compile(p, re.IGNORECASE) for p in IFRS_SANCTIONED_LIMITATION_PATTERNS]


def is_sanctioned_limitation_sentence(sentence: str) -> bool:
    """True if a sentence's limitation wording is IFRS-sanctioned (Tier 2 allowed)."""
    s = str(sentence or "")
    return any(rx.search(s) for rx in _IFRS_SANCTIONED_LIMITATION_RE)


# Map each sanctioned pattern to the IFRS requirement family it satisfies. Used by
# the evaluation's required-limitation recall: a section "satisfies" a family if it
# contains a sentence matching that family, whether or not a Tier-2 phrase is present
# (e.g. "estimates carry measurement uncertainty" is a valid disclosure on its own).
SANCTIONED_LIMITATION_FAMILIES = {
    "relief_financial_effects": [
        r"appl(?:y|ies|ied)\s+the\s+(?:relief|exemption)\s+in\s+paragraph\s+\d+",
        r"(?:relief|exemption)\s+(?:in|under)\s+(?:paragraph\s+\d+\s+of\s+)?IFRS\s+S[12]",
        r"impracticable\s+to\s+(?:provide|quantify|estimate)\b",
        r"without\s+undue\s+cost\s+or\s+effort",
        r"(?:reasonable|supportable)\s+information\s+(?:is\s+)?(?:not\s+)?available\s+without",
    ],
    "scope3_category_basis": [
        r"scope\s*3\s+categor(?:y|ies)\s+(?:included|excluded|comprise|are\s+limited\s+to|assessed|quantified|reported)",
        r"(?:includes?|excludes?|limited\s+to|covers?)\s+scope\s*3\s+categor(?:y|ies)",
        r"(?:other|remaining)\s+scope\s*3\s+categor(?:y|ies)\s+(?:were|are|have\s+been)\s+(?:assessed|excluded|omitted)",
        # Heading- or boundary-style framing: "Scope of financed-emissions disclosures
        # (IFRS S2 paragraph 29(a)(iv))", "... boundary, categories included and
        # methodology", "selected Scope 3 categories quantified using ...".
        r"29\(a\)\(iv\)",
        r"boundary,\s*categories\s+included",
        r"categories\s+included\s+and\s+methodolog",
        r"scope\s+of\s+(?:financed[- ]emissions|scope\s*3)\s+disclosures?",
        r"selected\s+scope\s*3\s+categor(?:y|ies)",
        r"(?:disclosures?|reporting)\s+(?:in\s+this\s+section\s+)?covers?\s*:",
    ],
    "estimation_uncertainty": [
        r"(?:estimation|measurement)\s+uncertaint(?:y|ies)",
        r"sources?\s+of\s+(?:estimation|measurement)\s+uncertaint",
        r"subject\s+to\s+(?:estimation|measurement)\s+uncertaint",
        r"data\s+quality\s+(?:score|tier|mix|characteristics|register)",
    ],
    "assurance": [
        # "limited assurance", "limited external assurance", "reasonable independent assurance"
        r"(?:limited|reasonable)\s+(?:\w+\s+){0,2}assurance",
        # "subject to ... assurance", "assurance provided by", "assured by", ISAE 3000
        r"subject\s+to\s+(?:\w+\s+){0,3}assurance",
        r"assurance\s+(?:provided|performed|conducted)\s+by",
        r"(?:externally\s+)?assured\s+by",
        r"\bISAE\s?3000\b",
        r"(?:has|have)\s+not\s+been\s+(?:externally\s+)?assured",
    ],
}
_SANCTIONED_FAMILY_RE = {
    fam: [re.compile(p, re.IGNORECASE) for p in pats]
    for fam, pats in SANCTIONED_LIMITATION_FAMILIES.items()
}


def sanctioned_disclosure_families(text: str) -> set:
    """Set of IFRS limitation-disclosure families present anywhere in the text."""
    s = str(text or "")
    found = set()
    for fam, regexes in _SANCTIONED_FAMILY_RE.items():
        if any(rx.search(s) for rx in regexes):
            found.add(fam)
    return found


_SENTENCE_SPLIT_RE = re.compile(r"(?<=[.!?])\s+")


def _iter_sentences_with_spans(text: str):
    """Yield (sentence_text, is_table_or_heading_line) for coarse sentence scan."""
    for raw_line in str(text or "").splitlines():
        line = raw_line.strip()
        if not line:
            continue
        is_structural = line.startswith("#") or line.startswith("|")
        for sent in _SENTENCE_SPLIT_RE.split(line):
            sent = sent.strip()
            if sent:
                yield sent, is_structural


def scan_limitation_language(text: str) -> Dict[str, Any]:
    """Single source of truth for limitation-language policy.

    Returns:
      tier1_hits            : pipeline-internal phrases found (always violations)
      unsanctioned_tier2    : Tier-2 phrases in non-sanctioned sentences (violations)
      sanctioned_sentences  : sentences where Tier-2 wording is IFRS-legitimate (allowed)
    """
    lower = str(text or "").lower()
    tier1_hits = sorted({p for p in TIER1_PIPELINE_INTERNAL_PHRASES if p in lower})

    unsanctioned_tier2 = []
    sanctioned_sentences = []
    for sent, _is_structural in _iter_sentences_with_spans(text):
        sent_l = sent.lower()
        phrase_hits = [p for p in TIER2_LIMITATION_PHRASES if p in sent_l]
        if not phrase_hits:
            continue
        if is_sanctioned_limitation_sentence(sent):
            sanctioned_sentences.append({"sentence": sent[:300], "phrases": phrase_hits})
        else:
            unsanctioned_tier2.append({"sentence": sent[:300], "phrases": phrase_hits})

    return {
        "tier1_hits": tier1_hits,
        "unsanctioned_tier2": unsanctioned_tier2,
        "sanctioned_sentences": sanctioned_sentences,
    }


def limitation_policy_violations(text: str) -> List[Dict[str, Any]]:
    """Flat list of policy violations (Tier-1 always + unsanctioned Tier-2)."""
    scan = scan_limitation_language(text)
    issues: List[Dict[str, Any]] = []
    for hit in scan["tier1_hits"]:
        issues.append({"type": "tier1_pipeline_internal_language", "phrase": hit})
    for item in scan["unsanctioned_tier2"]:
        issues.append({
            "type": "unsanctioned_limitation_language",
            "phrases": item["phrases"],
            "sentence": item["sentence"],
            "required_fix": "Remove, rewrite positively, or place inside an IFRS-sanctioned limitation statement.",
        })
    return issues


def drop_unsanctioned_limitation_lines(text: str) -> str:
    """Remove report lines whose Tier-2 limitation wording is NOT IFRS-sanctioned.

    Tier-1 wording is removed line-wise as well. Sanctioned limitation sentences
    (relief invocation, Scope 3 basis, estimation uncertainty, assurance) are
    preserved because IFRS requires them.
    """
    kept = []
    for line in str(text or "").splitlines():
        low = line.lower()
        stripped = line.strip()
        # Structural lines (headings, table rows) are never dropped here.
        if stripped.startswith("#") or stripped.startswith("|"):
            kept.append(line)
            continue
        if any(p in low for p in TIER1_PIPELINE_INTERNAL_PHRASES):
            continue
        tier2 = [p for p in TIER2_LIMITATION_PHRASES if p in low]
        if tier2 and not is_sanctioned_limitation_sentence(line):
            continue
        kept.append(line)
    return "\n".join(kept)


# ---- Structural hygiene: remove empty heading blocks (Fix 3) ----
_HEADING_RE = re.compile(r"^(#{1,6})\s+(.*\S)\s*$")


def remove_empty_heading_blocks(markdown: str, max_passes: int = 5) -> str:
    """Remove any heading immediately followed by no body content.

    A heading is "empty" if everything between it and the next heading of the
    same-or-higher level (or end of document) is blank. This deletes orphan
    headings left behind when sentence-removal sanitizers strip a section body,
    e.g. the empty "Operational emissions calculation integrity (fleet
    correction)" heading in the BANK01 report. Runs iteratively so that
    nested empties collapse.
    """
    if not markdown:
        return markdown

    for _ in range(max_passes):
        lines = markdown.splitlines()
        headings = []  # (index, level)
        for idx, line in enumerate(lines):
            m = _HEADING_RE.match(line)
            if m:
                headings.append((idx, len(m.group(1))))

        drop_ranges = []
        for h_pos, (idx, level) in enumerate(headings):
            # find where this heading's block ends: next heading of level <= this one
            end = len(lines)
            for nxt_idx, nxt_level in headings[h_pos + 1:]:
                if nxt_level <= level:
                    end = nxt_idx
                    break
            body = [l for l in lines[idx + 1:end] if l.strip()]
            if not body:
                drop_ranges.append((idx, end))

        if not drop_ranges:
            break

        # Remove from the bottom up so indices stay valid.
        for start, end in sorted(drop_ranges, reverse=True):
            del lines[start:end]
        markdown = "\n".join(lines)

    markdown = re.sub(r"\n{3,}", "\n\n", markdown).strip() + "\n"
    return markdown


print("Loaded two-tier limitation-language policy and structural-hygiene helpers "
      "(scan_limitation_language, remove_empty_heading_blocks).")


Loaded two-tier limitation-language policy and structural-hygiene helpers (scan_limitation_language, remove_empty_heading_blocks).


## Load inputs

The notebook is tolerant of different file structures. Preferred structure:

```text
/notebooks/gen_data/payloads/
  payload_BANK01_general_requirements.json
  payload_BANK01_governance.json
  payload_BANK01_strategy.json
  payload_BANK01_risk_management.json
  payload_BANK01_metrics_targets.json

/notebooks/gen_data/ifrs_requirements/
  general_requirements_requirements.json
  governance_requirements.json
  strategy_requirements.json
  risk_management_requirements.json
  metrics_and_targets_requirements.json

/notebooks/gen_data/style/style_system/
  authoring/
  judging/
  rendering/
```

In [6]:
# ============================================================
# CELL 4 — LOAD STYLE ARTIFACTS
# ============================================================

AUTHORING_DIR = STYLE_SYSTEM_DIR / "authoring"
JUDGING_DIR = STYLE_SYSTEM_DIR / "judging"
RENDERING_DIR = STYLE_SYSTEM_DIR / "rendering"

# Backward-compatible fallbacks if final organized folders are not present.
if not AUTHORING_DIR.exists():
    AUTHORING_DIR = STYLE_SYSTEM_DIR
if not JUDGING_DIR.exists():
    JUDGING_DIR = STYLE_SYSTEM_DIR / "judging"
if not RENDERING_DIR.exists():
    RENDERING_DIR = STYLE_SYSTEM_DIR / "rendering"

GLOBAL_STYLE = read_json(AUTHORING_DIR / "global_style_guide.json", default={})
STYLE_RUBRIC = read_json(JUDGING_DIR / "style_compliance_rubric.json", default={})

NO_COPYING_RULES = read_text(
    AUTHORING_DIR / "language_rules" / "no_copying_rules.md",
    default=read_text(STYLE_SYSTEM_DIR / "language_rules" / "no_copying_rules.md", default="")
)

TABLE_PATTERNS = read_json(
    AUTHORING_DIR / "table_patterns" / "table_patterns.json",
    default=read_json(STYLE_SYSTEM_DIR / "table_patterns" / "table_patterns.json", default={})
)

FORBIDDEN_TERMS = read_json(
    AUTHORING_DIR / "language_rules" / "forbidden_reference_terms.json",
    default=read_json(STYLE_SYSTEM_DIR / "language_rules" / "forbidden_reference_terms.json", default=[])
)

# Hardcoded safety fallback in case forbidden_reference_terms.json is absent.
FORBIDDEN_TERMS = sorted(set(FORBIDDEN_TERMS + [
    "Emirates NBD", "Emirates NBD Group", "DenizBank", "Emirates Islamic",
    "Dubai", "UAE", "AED", "CBUAE", "Sustainalytics", "KPMG",
    "Microsoft Sustainability Manager"
]))


def load_section_style(section_name: str) -> Dict[str, Any]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        AUTHORING_DIR / "section_style_guides" / f"{slug}_style.json",
        AUTHORING_DIR / "section_style_guides" / f"{slug}.json",
        STYLE_SYSTEM_DIR / "section_style_guides" / f"{slug}_style.json",
        STYLE_SYSTEM_DIR / "section_style_guides" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return read_json(path, default={})
    return {}


def load_section_blueprint(section_name: str) -> Dict[str, Any]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        AUTHORING_DIR / "section_blueprints" / f"{slug}_blueprint.json",
        AUTHORING_DIR / "section_blueprints" / f"{slug}.json",
        STYLE_SYSTEM_DIR / "section_blueprints" / f"{slug}_blueprint.json",
        STYLE_SYSTEM_DIR / "section_blueprints" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return read_json(path, default={})
    return {}

print("Loaded global style:", bool(GLOBAL_STYLE))
print("Loaded table patterns:", bool(TABLE_PATTERNS))
print("Loaded no-copying rules:", bool(NO_COPYING_RULES))
print("Forbidden terms count:", len(FORBIDDEN_TERMS))

Loaded global style: True
Loaded table patterns: True
Loaded no-copying rules: True
Forbidden terms count: 31


In [7]:
# ============================================================
# CELL 5 — LOAD REQUIREMENTS
# REQUIREMENTS LOADER RULE: supports section JSON files nested by standard, e.g.
# {
#   "section_key": "governance",
#   "section_title": "Governance",
#   "row_count": 15,
#   "standards": {
#       "IFRS S1": {"row_count": 7, "requirements": [...]},
#       "IFRS S2": {"row_count": 8, "requirements": [...]}
#   }
# }
# ============================================================

METADATA_KEYS = {
    "section_key",
    "section_title",
    "source",
    "row_count",
    "standards",
    "created_at",
    "metadata",
    "notes",
}


def find_requirements_file(section_name: str) -> Optional[Path]:
    slug = SECTION_SLUGS[section_name]
    candidates = [
        REQUIREMENTS_DIR / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / f"{slug}.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / "json" / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / "json" / f"{slug}.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / f"{slug}_requirements.json",
        REQUIREMENTS_DIR / "section_by_section_requirements" / f"{slug}.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def find_combined_requirements_file() -> Optional[Path]:
    candidates = [
        REQUIREMENTS_DIR / "ifrs_s1_s2_generation_requirements.json",
        REQUIREMENTS_DIR / "generation_requirements.json",
        REQUIREMENTS_DIR / "ifrs_s1_s2_requirements_kb_final.json",
        REQUIREMENTS_DIR / "requirements.json",
        GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_generation_requirements.json",
        GEN_DATA_DIR / "IFRS" / "ifrs_requirements_kb_outputs_final" / "ifrs_s1_s2_requirements_kb_final.json",
        GEN_DATA_DIR / "ifrs_s1_s2_generation_requirements.json",
        GEN_DATA_DIR / "ifrs_s1_s2_requirements_kb_final.json",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def _norm_key(value: Any) -> str:
    return re.sub(r"[^a-z0-9]+", "", str(value).lower())


def _looks_like_requirement_dict(obj: Dict[str, Any]) -> bool:
    if not isinstance(obj, dict):
        return False
    keys = set(obj.keys())
    return bool(keys.intersection({
        "requirement_id",
        "clean_requirement_text",
        "requirement_text",
        "source_paragraph_text",
        "paragraph_id",
        "report_section",
        "clause_path",
    }))


def _row_has_requirement_text(row: Dict[str, Any]) -> bool:
    text = (
        row.get("clean_requirement_text")
        or row.get("requirement_text")
        or row.get("source_paragraph_text")
        or row.get("text")
        or row.get("paragraph_text")
    )
    return bool(str(text).strip())


def _is_real_requirement_row(row: Any) -> bool:
    if not isinstance(row, dict):
        return False

    rid = str(row.get("requirement_id", "")).strip()
    if not rid or rid in METADATA_KEYS:
        return False

    if not _row_has_requirement_text(row):
        return False

    rid_upper = rid.upper()
    return (
        rid_upper.startswith("IFRS_")
        or "paragraph_id" in row
        or "requirement_text" in row
        or "clean_requirement_text" in row
    )


def rows_from_requirements_object(obj: Any, section_name: str = "") -> List[Dict[str, Any]]:
    """
    Convert many possible JSON shapes into a list of REAL IFRS requirement rows.

    Critical fix:
    Your section requirement files store real rows under:
        obj["standards"]["IFRS S1"]["requirements"]
        obj["standards"]["IFRS S2"]["requirements"]

    The previous notebook iterated over metadata keys such as section_key,
    section_title and source. This function prevents that.
    """
    rows: List[Dict[str, Any]] = []

    # Case 1: already a list of rows.
    if isinstance(obj, list):
        rows = [r for r in obj if isinstance(r, dict)]

    elif isinstance(obj, dict):
        # Case 2: the actual format of your current files.
        if isinstance(obj.get("standards"), dict):
            for standard_name, standard_obj in obj["standards"].items():
                if not isinstance(standard_obj, dict):
                    continue

                reqs = standard_obj.get("requirements", [])
                if isinstance(reqs, dict):
                    reqs = list(reqs.values())

                if isinstance(reqs, list):
                    for req in reqs:
                        if isinstance(req, dict):
                            row = dict(req)
                            row.setdefault("standard", standard_name)
                            row.setdefault("report_section", obj.get("section_title", section_name))
                            rows.append(row)

        # Case 3: common direct list containers.
        elif isinstance(obj.get("requirements"), list):
            rows = [r for r in obj["requirements"] if isinstance(r, dict)]

        elif isinstance(obj.get("generation_requirements"), list):
            rows = [r for r in obj["generation_requirements"] if isinstance(r, dict)]

        elif isinstance(obj.get("items"), list):
            rows = [r for r in obj["items"] if isinstance(r, dict)]

        elif isinstance(obj.get("data"), list):
            rows = [r for r in obj["data"] if isinstance(r, dict)]

        # Case 4: single requirement object.
        elif _looks_like_requirement_dict(obj):
            rows = [obj]

        # Case 5: dict keyed by section names or requirement IDs.
        else:
            section_keys = {
                _norm_key(section_name),
                _norm_key(SECTION_SLUGS.get(section_name, "")),
                _norm_key(section_name.replace("and", "&")),
            }

            # First check whether a value is the matching section container.
            for key, value in obj.items():
                if key in METADATA_KEYS:
                    continue
                if _norm_key(key) in section_keys:
                    rows.extend(rows_from_requirements_object(value, section_name))

            # Otherwise, treat it as dict keyed by requirement_id.
            if not rows:
                for key, value in obj.items():
                    if key in METADATA_KEYS:
                        continue
                    if isinstance(value, dict):
                        row = dict(value)
                        row.setdefault("requirement_id", key)
                        rows.append(row)

    # Final cleanup: keep only real IFRS requirement rows.
    clean_rows = []
    seen = set()
    for row in rows:
        if not _is_real_requirement_row(row):
            continue

        rid = str(row.get("requirement_id", "")).strip()
        if rid in seen:
            continue
        seen.add(rid)
        clean_rows.append(row)

    return clean_rows


def normalize_requirement(row: Any, section_name: str) -> Dict[str, Any]:
    if not isinstance(row, dict):
        raise TypeError(f"Requirement row must be a dict after extraction, got: {type(row).__name__}")

    requirement_text = (
        row.get("clean_requirement_text")
        or row.get("requirement_text")
        or row.get("source_paragraph_text")
        or row.get("text")
        or row.get("paragraph_text")
        or ""
    )

    evidence_tags = row.get("evidence_tags", [])
    if isinstance(evidence_tags, str):
        try:
            parsed = json.loads(evidence_tags)
            evidence_tags = parsed if isinstance(parsed, list) else [parsed]
        except Exception:
            evidence_tags = [x.strip() for x in re.split(r"[,;|]", evidence_tags) if x.strip()]
    elif not isinstance(evidence_tags, list):
        evidence_tags = [str(evidence_tags)] if evidence_tags else []

    return {
        "requirement_id": str(row.get("requirement_id") or row.get("id") or now_id()),
        "standard": row.get("standard", ""),
        "paragraph_id": row.get("paragraph_id", row.get("paragraph", "")),
        "page": row.get("page", ""),
        "report_section": row.get("report_section", section_name),
        "requirement_text": str(requirement_text).strip(),
        "clause_path": row.get("clause_path", ""),
        "obligation_type": row.get("obligation_type", ""),
        "mandatory": normalize_bool(row.get("mandatory", True)),
        "evidence_tags": evidence_tags,
        "banking_relevance": row.get("banking_relevance", ""),
        "raw": row,
    }


def section_matches(row: Dict[str, Any], section_name: str) -> bool:
    sec = str(row.get("report_section", "")).strip()
    if not sec:
        return True
    return _norm_key(sec) == _norm_key(section_name)


def validate_loaded_requirements(section_name: str, rows: List[Dict[str, Any]], source_path: Path) -> None:
    bad_ids = {"section_key", "section_title", "source", "row_count", "standards"}
    ids = {str(r.get("requirement_id", "")) for r in rows}

    if ids.intersection(bad_ids):
        raise ValueError(
            f"Requirement loader bug for {section_name}: metadata keys were loaded as requirements: "
            f"{sorted(ids.intersection(bad_ids))}"
        )

    if not rows:
        raise ValueError(
            f"No real IFRS requirement rows extracted for {section_name} from {source_path}. "
            "Check that the JSON contains standards -> IFRS S1/IFRS S2 -> requirements."
        )


def load_requirements_for_section(section_name: str) -> List[Dict[str, Any]]:
    section_file = find_requirements_file(section_name)

    if section_file:
        obj = read_json(section_file)
        raw_rows = rows_from_requirements_object(obj, section_name)
        rows = [normalize_requirement(r, section_name) for r in raw_rows]
        rows = [r for r in rows if r["requirement_text"] and section_matches(r, section_name)]
        validate_loaded_requirements(section_name, rows, section_file)

        declared_count = obj.get("row_count") if isinstance(obj, dict) else None
        print(f"Loaded requirements for {section_name} from section file: {section_file}")
        print(f"  extracted real IFRS rows: {len(rows)}" + (f" / declared row_count: {declared_count}" if declared_count else ""))
        return rows

    combined_file = find_combined_requirements_file()
    if not combined_file:
        raise FileNotFoundError(
            "Could not find IFRS requirements. Place section JSON files in REQUIREMENTS_DIR "
            "or set IFRS_REQUIREMENTS_DIR in .env."
        )

    obj = read_json(combined_file)
    raw_rows = rows_from_requirements_object(obj, section_name)
    rows = []
    for r in raw_rows:
        nr = normalize_requirement(r, section_name)
        sec = str(nr.get("report_section", "")).strip()
        if _norm_key(sec) == _norm_key(section_name):
            rows.append(nr)

    validate_loaded_requirements(section_name, rows, combined_file)
    print(f"Loaded requirements for {section_name} from combined file: {combined_file}")
    print(f"  extracted real IFRS rows: {len(rows)}")
    return rows


requirements_by_section = {
    section: load_requirements_for_section(section)
    for section in SECTIONS
}

for section, reqs in requirements_by_section.items():
    sample_ids = [r["requirement_id"] for r in reqs[:3]]
    print(f"{section}: {len(reqs)} requirements | sample IDs: {sample_ids}")


Loaded requirements for General Requirements from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\general_requirements_requirements.json
  extracted real IFRS rows: 108 / declared row_count: 108
Loaded requirements for Governance from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\governance_requirements.json
  extracted real IFRS rows: 15 / declared row_count: 15
Loaded requirements for Strategy from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\IFRS\ifrs_requirements_kb_outputs_final\section_by_section_requirements\json\strategy_requirements.json
  extracted real IFRS rows: 70 / declared row_count: 70
Loaded requirements for Risk Management from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebo

In [8]:
# ============================================================
# CELL 6 — LOAD PAYLOADS
# UPDATED: also searches /notebooks/BANK01 and common project folders.
# ============================================================


def payload_search_dirs() -> List[Path]:
    candidates = [
        PAYLOAD_DIR,
        NOTEBOOK_DIR / "payloads",
        NOTEBOOK_DIR / "data",
        GEN_DATA_DIR / "payloads",
        GEN_DATA_DIR / "BANK01",
        GEN_DATA_DIR / "data",
        CURRENT_DIR / "BANK01",
        CURRENT_DIR / "data",
    ]

    # Keep unique existing-or-configured paths in order.
    out = []
    seen = set()
    for p in candidates:
        p = Path(p).resolve()
        if p not in seen:
            out.append(p)
            seen.add(p)
    return out


def find_payload_file(section_name: str) -> Optional[Path]:
    slug = SECTION_SLUGS[section_name]
    aliases = {
        "metrics_and_targets": ["metrics_targets", "metrics_and_targets", "metrics_targets"],
        "risk_management": ["risk_management", "risk"],
        "general_requirements": ["general_requirements", "general"],
        "governance": ["governance"],
        "strategy": ["strategy"],
    }[slug]

    filename_patterns = []
    for alias in aliases:
        filename_patterns.extend([
            f"payload_BANK01_{alias}.json",
            f"payload_BANK01_{alias}*.json",
            f"BANK01_{alias}.json",
            f"*{alias}*.json",
            f"{alias}.json",
        ])

    for folder in payload_search_dirs():
        if not folder.exists():
            continue
        for pattern in filename_patterns:
            matches = sorted(folder.glob(pattern))
            if matches:
                return matches[0]

    return None


def find_combined_payload_file() -> Optional[Path]:
    filename_patterns = [
        "payload_BANK01.json",
        "payload_BANK01*.json",
        "BANK01.json",
        "payload.json",
        "*payload*.json",
    ]

    for folder in payload_search_dirs():
        if not folder.exists():
            continue
        for pattern in filename_patterns:
            matches = sorted(folder.glob(pattern))
            if matches:
                # Avoid selecting section payload if a combined one exists later.
                section_hint = matches[0].name.lower()
                if any(x in section_hint for x in ["governance", "strategy", "risk_management", "metrics", "general_requirements"]):
                    continue
                return matches[0]

    return None


def load_payload_for_section(section_name: str) -> Dict[str, Any]:
    section_file = find_payload_file(section_name)
    if section_file:
        print(f"Loaded payload for {section_name} from section file: {section_file}")
        return read_json(section_file)

    combined_file = find_combined_payload_file()
    if combined_file:
        print(f"Loaded payload for {section_name} from combined file: {combined_file}")
        combined = read_json(combined_file)
        slug = SECTION_SLUGS[section_name]
        possible_keys = [
            slug,
            slug.replace("metrics_and_targets", "metrics_targets"),
            section_name,
            section_name.lower(),
            section_name.replace(" ", "_").lower(),
        ]
        for key in possible_keys:
            if isinstance(combined, dict) and key in combined:
                return combined[key]
        return combined

    searched = "\n".join([f"- {p}" for p in payload_search_dirs()])
    raise FileNotFoundError(
        "Could not find payload files.\n\n"
        "Searched these folders:\n"
        f"{searched}\n\n"
        "Expected examples:\n"
        "- payload_BANK01_governance.json\n"
        "- payload_BANK01_strategy.json\n"
        "- payload_BANK01_risk_management.json\n"
        "- payload_BANK01_metrics_targets.json\n"
        "- payload_BANK01_general_requirements.json\n"
        "- payload_BANK01.json"
    )


payloads_by_section = {
    section: load_payload_for_section(section)
    for section in SECTIONS
}

for section, payload in payloads_by_section.items():
    flat_count = len(flatten_json(payload))
    print(f"{section}: payload fields={flat_count}")


Loaded payload for General Requirements from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads_risk\payload_BANK01_general_requirements.json
Loaded payload for Governance from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads_risk\payload_BANK01_governance.json
Loaded payload for Strategy from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads_risk\payload_BANK01_strategy.json
Loaded payload for Risk Management from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads_risk\payload_BANK01_risk_management.json
Loaded payload for Metrics and Targets from section file: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\payloads_risk\payload_BANK01_metrics_targets.json
General Requirements: payload fields=734
Governance: payload fields=939
Strategy: payload fields=1372
Risk Management: payload fie

In [9]:
# ============================================================
# CELL 9C — DETERMINISTIC ARITHMETIC / COHERENCE VALIDATOR
# ============================================================
# Why this cell exists (Fix 2):
#
# The fact-lock gate only checks whether a number EXISTS somewhere in the
# payload. It cannot catch contradictions where both sides are individually
# payload-supported, e.g.:
#   - an intensity target of 20.0 tCO2e/EURm while financed emissions / loans
#     recomputes to ~1,154.8;
#   - a Scope 1+2 baseline of 137,849 tCO2e against a reported current figure of
#     ~5,695 (a ~24x mismatch);
#   - an all-scopes baseline (120,000) that is SMALLER than one of its own
#     subset baselines (137,849);
#   - "37.5% climate expertise" on a 10-person board (= 3.75 directors).
#
# This validator recomputes these relationships deterministically. It is fuzzy
# about key names (it discovers candidate fields in the flattened payload) so it
# works across BANK01-BANK05 without hard-coding schema paths. It runs at two
# levels:
#   payload_arithmetic_issues()      — against the payload (fail fast, pre-write)
#   report_numeric_consistency_issues() — against the assembled report prose
#                                          (precision drift + labelled-metric conflicts)
#
# Behaviour is controlled by IFRS_ARITHMETIC_STRICT (default: warn, don't raise).
# ============================================================

from decimal import Decimal, InvalidOperation

IFRS_ARITHMETIC_STRICT = os.getenv("IFRS_ARITHMETIC_STRICT", "0").strip().lower() in {"1", "true", "yes", "on"}
IFRS_ARITHMETIC_REL_TOL = float(os.getenv("IFRS_ARITHMETIC_REL_TOL", "0.05"))       # 5% for ratio recomputation
IFRS_ARITHMETIC_MAGNITUDE_TOL = float(os.getenv("IFRS_ARITHMETIC_MAGNITUDE_TOL", "5.0"))  # baseline vs current ratio


def _arith_to_decimal(value: Any) -> Optional[Decimal]:
    if value is None or isinstance(value, bool):
        return None
    if isinstance(value, Decimal):
        return value if value.is_finite() else None
    if isinstance(value, (int, float)):
        try:
            d = Decimal(str(value))
            return d if d.is_finite() else None
        except (InvalidOperation, ValueError):
            return None
    if isinstance(value, str):
        # Grouped form (1,234,567.8) requires >=1 separator group; otherwise plain.
        m = re.search(r"-?\d{1,3}(?:[,\s]\d{3})+(?:\.\d+)?|-?\d+(?:\.\d+)?", value)
        if not m:
            return None
        try:
            d = Decimal(m.group(0).replace(",", "").replace(" ", ""))
            return d if d.is_finite() else None
        except (InvalidOperation, ValueError):
            return None
    return None


def _arith_leaf(path: str) -> str:
    """Last dotted/bracketed segment of a flattened path, lowercased."""
    p = re.sub(r"\[\d+\]", "", str(path))
    return p.rsplit(".", 1)[-1].lower()


def _arith_flat_numeric_index(payload: Dict[str, Any]) -> Dict[str, Decimal]:
    """Flatten payload to {lowercased_path: Decimal} for numeric fields only."""
    flat = flatten_json(payload)
    out = {}
    for path, value in flat.items():
        d = _arith_to_decimal(value)
        if d is not None:
            out[str(path).lower()] = d
    return out


def _arith_flat_string_index(payload: Dict[str, Any]) -> Dict[str, str]:
    """Flatten payload to {lowercased_path: str} for string fields (e.g. a target's
    `scope`/`metric`), used by R3 to read the scope coverage and unit of a baseline
    from its sibling fields."""
    flat = flatten_json(payload)
    out = {}
    for path, value in flat.items():
        if isinstance(value, str):
            out[str(path).lower()] = value
    return out


def _arith_find(index: Dict[str, Decimal], must_all: List[str], must_any: Optional[List[str]] = None,
                exclude: Optional[List[str]] = None) -> List[Tuple[str, Decimal]]:
    """Find (path, value) pairs whose LEAF field name contains all `must_all`
    tokens and at least one `must_any` token, excluding any `exclude` tokens.

    Matching is leaf-scoped so that a parent segment (e.g. the container
    `financed_emissions`) does not pollute matching/exclusion of a leaf such as
    `lending_book_outstanding_meur`.
    """
    must_any = must_any or []
    exclude = exclude or []
    hits = []
    for path, val in index.items():
        leaf = _arith_leaf(path)
        if all(tok in leaf for tok in must_all) and not any(x in leaf for x in exclude):
            if not must_any or any(tok in leaf for tok in must_any):
                hits.append((path, val))
    return hits


def _rel_diff(a: Decimal, b: Decimal) -> float:
    if a == 0 and b == 0:
        return 0.0
    denom = max(abs(a), abs(b))
    if denom == 0:
        return 0.0
    return float(abs(a - b) / denom)


def _arith_parent(path: str) -> str:
    """Object parent of a flattened path (drops the final field and any [idx])."""
    p = re.sub(r"\[\d+\]", "", str(path))
    return p.rsplit(".", 1)[0] if "." in p else ""


def _arith_share_parent(path_a: str, path_b: str, levels: int = 1) -> bool:
    """True if two paths share their top `levels` parent segment(s).

    Prevents pairing a percentage from one object (e.g. board governance) with a
    headcount from an unrelated object (e.g. workforce), which is what caused the
    combinatorial explosion when all sections were merged into one index.
    """
    def prefix(p):
        p = re.sub(r"\[\d+\]", "", str(p))
        return ".".join(p.split(".")[:levels])
    pa, pb = prefix(path_a), prefix(path_b)
    return bool(pa) and pa == pb


def _section_arithmetic_issues(index: Dict[str, Decimal], scope_label: str,
                               string_index: Optional[Dict[str, str]] = None) -> List[Dict[str, Any]]:
    """All deterministic coherence rules for ONE section's numeric index.

    Pairing rules (R2, R4) are constrained to fields that share an object parent,
    and each rule emits at most a bounded, deduplicated set of findings, so the
    output is a usable audit signal rather than a full cross-product.
    """
    issues: List[Dict[str, Any]] = []

    # R1 — intensity == financed emissions / loan (or lending) book, within tolerance.
    intensities = _arith_find(index, ["intensity"], exclude=["target", "baseline"])
    fin_em = _arith_find(index, ["financed"], must_any=["emission", "tco2e", "ghg"], exclude=["intensity", "target", "per_"])
    loans = _arith_find(index, [], must_any=["loan", "lending", "outstanding", "exposure"],
                        exclude=["intensity", "emission", "tco2e", "count", "number"])
    if intensities and fin_em and loans:
        fe = max(fin_em, key=lambda kv: abs(kv[1]))[1]
        lb = max(loans, key=lambda kv: abs(kv[1]))[1]
        if lb != 0:
            recomputed = fe / lb
            # Compare only the single most plausible stated intensity, not every match.
            best = min(intensities, key=lambda kv: _rel_diff(kv[1], recomputed) if kv[1] != 0 else 1e9)
            path, stated = best
            if stated != 0 and IFRS_ARITHMETIC_REL_TOL < _rel_diff(stated, recomputed) < 100:
                issues.append({
                    "type": "intensity_recompute_mismatch",
                    "scope": scope_label,
                    "stated_path": path,
                    "stated_value": float(stated),
                    "recomputed_from": "financed_emissions / lending_book",
                    "recomputed_value": float(recomputed),
                    "relative_difference": round(_rel_diff(stated, recomputed), 4),
                    "note": "Stated intensity does not match financed emissions divided by the lending book.",
                })

    # R2 — subset baselines must be <= superset (all-scopes) baseline (same object only).
    baselines = _arith_find(index, ["baseline"], exclude=["intensity", "year", "date"])
    all_scope_baselines = [kv for kv in baselines if "all" in kv[0] and "scope" in kv[0]]
    subset_baselines = [kv for kv in baselines if kv not in all_scope_baselines]
    seen_r2 = set()
    for a_path, a_val in all_scope_baselines:
        for s_path, s_val in subset_baselines:
            if not _arith_share_parent(a_path, s_path):
                continue
            if s_val > a_val and a_val > 0 and (a_path, s_path) not in seen_r2:
                seen_r2.add((a_path, s_path))
                issues.append({
                    "type": "baseline_subset_exceeds_superset",
                    "scope": scope_label,
                    "all_scopes_baseline_path": a_path,
                    "all_scopes_baseline_value": float(a_val),
                    "subset_baseline_path": s_path,
                    "subset_baseline_value": float(s_val),
                    "note": "A subset (e.g. Scope 1+2) baseline exceeds the all-scopes baseline.",
                })

    # R3 — target baseline vs reported current emissions: same order of magnitude.
    # Compared LIKE-FOR-LIKE: a baseline is only checked against current emissions
    # of the SAME scope family (scope1<->scope1, financed<->financed, ...). The old
    # "nearest value" pairing compared a financed-emissions baseline (tens of
    # millions) against operational Scope 2 (thousands) and against metadata/
    # reconciliation fields, producing spurious 500x-5000x ratios.
    def _scope_family(path: str) -> Optional[str]:
        p = str(path).lower()
        if "financed" in p:
            return "financed"
        if "scope1" in p or "scope_1" in p:
            return "scope1"
        if "scope2" in p or "scope_2" in p:
            return "scope2"
        if "scope3" in p or "scope_3" in p:
            return "scope3"
        if "total_ghg" in p or "total_emissions" in p:
            return "total"
        return None

    def _family_from_scope_string(scope_str: str) -> Optional[str]:
        p = str(scope_str).lower()
        if "financed" in p or "cat15" in p or "category_15" in p or "cat_15" in p:
            return "financed"
        if "all_scope" in p or "all scopes" in p or "all-scope" in p:
            return "allscopes"
        # scope 1 and 2 combined ("scope_1_and_2", "scope 1 & 2", "scope1and2",
        # "scope1_and_2"). The "2" may not carry its own "scope" prefix.
        if re.search(r"scope[_\s]?1[_\s]?(?:and|&|\+|,)[_\s]?2", p) or \
           re.search(r"1_and_2|1and2|1\s*&\s*2|1\s+and\s+2", p):
            return "scope12"
        has_1 = bool(re.search(r"scope[_\s]?1", p))
        has_2 = bool(re.search(r"scope[_\s]?2", p))
        if has_1 and has_2:
            return "scope12"
        if has_1:
            return "scope1"
        if has_2:
            return "scope2"
        if re.search(r"scope[_\s]?3", p):
            return "scope3"
        return None

    def _baseline_context(b_path: str) -> Dict[str, Any]:
        # Read sibling fields (scope, metric unit) of the target object that owns this
        # baseline from the STRING index, so we know its scope coverage and whether it
        # is an absolute or intensity target.
        parent = re.sub(r"\.baseline_value$", "", b_path)
        ctx: Dict[str, Any] = {}
        for src_map in (string_index or {}, index):
            for k, v in src_map.items():
                if k.startswith(parent + "."):
                    ctx.setdefault(k[len(parent) + 1:], v)
        return ctx

    # Current absolute emissions by family (sum sibling components where needed).
    current_by_family: Dict[str, float] = {}
    for path, val in index.items():
        if val is None or float(val) <= 0:
            continue
        pl = str(path).lower()
        if any(x in pl for x in ("baseline", "target", "per_", "pct", "percent",
                                  "reconciliation", "metadata", "rec_", "adjustment",
                                  "sovereign", "national_")):
            continue
        fam = _scope_family(path)
        if not fam:
            continue
        # Keep the largest plausible current value per family (the headline figure).
        current_by_family[fam] = max(current_by_family.get(fam, 0.0), float(val))
    # Derive a scope1+2 combined current value if both are present.
    if "scope1" in current_by_family and "scope2" in current_by_family:
        current_by_family["scope12"] = current_by_family["scope1"] + current_by_family["scope2"]

    # Map target scope strings (from the raw payloads) to families, per section.
    def _target_scope_family(b_path: str) -> Optional[str]:
        parent = re.sub(r"\.baseline_value$", "", b_path)
        # Look up the scope string in the raw payload via the flattened index keys.
        for key_suffix in ("scope", "scope_coverage"):
            k = f"{parent}.{key_suffix}"
            # index only holds numerics; scope strings are not there, so re-read raw.
        return None  # resolved below from raw payload

    for b_path, b_val in baselines:
        if b_val <= 0:
            continue
        ctx = _baseline_context(b_path)
        # Skip intensity-unit targets: their baseline is per-EURm, a different unit
        # space from absolute current emissions, so a magnitude ratio is meaningless.
        metric_unit = str(ctx.get("metric", "")).lower() if ctx else ""
        if "per_meur" in metric_unit or "intensity" in metric_unit or "per_eur" in metric_unit:
            continue
        b_family = _family_from_scope_string(str(ctx.get("scope", ctx.get("scope_coverage", "")))) or _scope_family(b_path)
        if not b_family:
            continue
        # allscopes baselines compare against the financed/total headline (dominant).
        lookup_family = b_family
        if b_family == "allscopes":
            lookup_family = "financed" if "financed" in current_by_family else "total"
        c_val = current_by_family.get(lookup_family)
        candidates = [(lookup_family, Decimal(str(c_val)))] if c_val else []
        if not candidates:
            continue
        fam_name, c_dec = candidates[0]
        c_val_f = float(c_dec)
        if c_val_f > 0:
            ratio = float(max(float(b_val), c_val_f) / min(float(b_val), c_val_f))
            if ratio > IFRS_ARITHMETIC_MAGNITUDE_TOL:
                issues.append({
                    "type": "baseline_vs_current_magnitude_mismatch",
                    "scope": scope_label,
                    "scope_family": b_family,
                    "baseline_path": b_path,
                    "baseline_value": float(b_val),
                    "current_family": fam_name,
                    "current_value": c_val_f,
                    "ratio": round(ratio, 2),
                    "note": f"Target baseline ({b_family}) and current {fam_name} emissions differ by ~{ratio:.0f}x.",
                })

    # R4 — a percentage that is SEMANTICALLY ABOUT a population should imply a whole
    # number of people. Only percentages whose field name refers to the population
    # itself (e.g. "board_climate_expertise_pct", "independent_directors_pct",
    # "% of directors ...") are tested. Compensation/remuneration/agenda/pay ratios
    # (ceo_esg_compensation_pct, all_exec_climate_remuneration_pct,
    # climate_on_board_agenda_pct) are NOT percentages of headcount -- multiplying
    # them by board size is meaningless -- so they are excluded. This removed a large
    # block of false positives.
    _POPULATION_PCT_TOKENS = ("expertise", "independent_directors", "directors_with",
                              "board_members", "women", "gender", "diversity",
                              "female", "male", "membership")
    _NON_POPULATION_PCT_TOKENS = ("compensation", "remuneration", "pay", "salary",
                                  "bonus", "agenda", "coverage", "attendance",
                                  "budget", "revenue", "capex", "opex", "return")
    headcounts = _arith_find(index, [], must_any=["board_size", "num_directors", "n_directors",
                                                  "headcount", "num_employees", "n_employees",
                                                  "board_member_count"],
                             exclude=["pct", "percent", "ratio"])

    def _is_population_percentage(path: str) -> bool:
        pl = str(path).lower()
        if any(t in pl for t in _NON_POPULATION_PCT_TOKENS):
            return False
        return any(t in pl for t in _POPULATION_PCT_TOKENS)

    percentages = [kv for kv in _arith_find(index, [], must_any=["pct", "percent", "expertise"],
                                            exclude=["amount", "meur", "tco2e"])
                   if _is_population_percentage(kv[0])]
    seen_r4 = set()
    for p_path, p_val in percentages:
        if not (0 < float(p_val) <= 100):
            continue
        # Pair against headcounts in the same object; normalise array indices so
        # sibling elements (governance[0]/[1]/[2]) are not treated as separate
        # objects and do not multiply the same finding.
        local_heads = [kv for kv in headcounts if _arith_share_parent(p_path, kv[0])]
        # De-index the percentage path so the dedup key collapses array siblings.
        p_key = re.sub(r"\[\d+\]", "[]", p_path)
        for h_path, h_val in local_heads:
            if h_val <= 0 or h_val > 1000:
                continue
            h_key = re.sub(r"\[\d+\]", "[]", h_path)
            product = float(p_val) / 100.0 * float(h_val)
            if abs(product - round(product)) > 0.05 and (p_key, h_key, float(p_val)) not in seen_r4:
                seen_r4.add((p_key, h_key, float(p_val)))
                issues.append({
                    "type": "percentage_of_headcount_not_integer",
                    "scope": scope_label,
                    "percentage_path": p_path,
                    "percentage_value": float(p_val),
                    "headcount_path": h_path,
                    "headcount_value": float(h_val),
                    "implied_count": round(product, 2),
                    "note": "A population percentage does not yield a whole number of people.",
                })

    # R5 — distributions/mix fields that should sum to ~100.
    mix_groups: Dict[str, List[Tuple[str, Decimal]]] = defaultdict(list)
    for path, val in index.items():
        if any(tok in path for tok in ["_pct", "percent", "share", "mix", "distribution"]) and 0 <= float(val) <= 100:
            mix_groups[_arith_parent(path)].append((path, val))
    for parent, members in mix_groups.items():
        if len(members) >= 3:
            total = float(sum(v for _, v in members))
            if 90.0 < total < 110.0 and abs(total - 100.0) > 1.0:
                issues.append({
                    "type": "distribution_does_not_sum_to_100",
                    "scope": scope_label,
                    "group": parent,
                    "member_count": len(members),
                    "sum": round(total, 2),
                    "note": "A percentage distribution sums close to but not exactly 100.",
                })

    # R6 — progress percentages must be within 0-100.
    for path, val in _arith_find(index, ["progress"], exclude=["amount", "meur"]):
        if not (0 <= float(val) <= 100):
            issues.append({
                "type": "progress_out_of_range",
                "scope": scope_label,
                "path": path,
                "value": float(val),
                "note": "A progress percentage falls outside 0-100.",
            })

    return issues


def payload_arithmetic_issues(section_name: Optional[str] = None) -> List[Dict[str, Any]]:
    """Deterministic coherence checks over the payload(s).

    Runs the rule set PER SECTION (never on a merged cross-section index), because
    merging different banks/sections into one flat index destroys the object context
    that makes a numeric relationship meaningful and produces a combinatorial blow-up
    of meaningless field pairings. Findings are deduplicated across sections.
    """
    sections = [section_name] if section_name is not None else list(SECTIONS)
    all_issues: List[Dict[str, Any]] = []
    seen: Dict[Tuple, Dict[str, Any]] = {}
    for sec in sections:
        if sec not in payloads_by_section:
            continue
        index = _arith_flat_numeric_index(payloads_by_section[sec])
        string_index = _arith_flat_string_index(payloads_by_section[sec])
        for issue in _section_arithmetic_issues(index, sec, string_index):
            # Dedup on the finding's DATA identity (type + the paths/values it
            # references), NOT on the scope. Section payloads each carry the shared
            # governance/targets context, so the same governance finding surfaces
            # under every section; keying on scope let all five copies through.
            path_id = (issue.get("stated_path") or issue.get("subset_baseline_path")
                       or issue.get("baseline_path") or issue.get("percentage_path")
                       or issue.get("group") or issue.get("path"))
            secondary = (issue.get("current_path") or issue.get("headcount_path")
                         or issue.get("all_scopes_baseline_path") or "")
            key = (issue.get("type"), path_id, secondary)
            if key in seen:
                # Record additional sections where the same finding appears, but do
                # not emit a duplicate issue.
                seen[key].setdefault("also_in_sections", [])
                if sec not in seen[key]["also_in_sections"] and sec != seen[key].get("scope"):
                    seen[key]["also_in_sections"].append(sec)
                continue
            seen[key] = issue
            all_issues.append(issue)
    return all_issues


# ---- Report-level numeric consistency (precision drift + labelled-metric conflicts) ----

_LABELLED_METRIC_TOKENS = {
    "financed_emissions": ["financed emission"],
    "financed_emissions_intensity": ["financed emissions intensity", "intensity"],
    "tier1_capital": ["tier 1 capital", "cet1", "common equity tier 1"],
    "climate_capex": ["climate capex", "climate capital expenditure", "green capex"],
    "scope1": ["scope 1"],
    "scope2": ["scope 2"],
}


_METRIC_LABEL_ORDINAL_RE = re.compile(
    # "Scope 1", "Scopes 1 and 2", "Scope 1-2", "Scopes 1 & 2", "Category 15", ...
    r"\b(?:tiers?|scopes?|categor(?:y|ies)|cet|phases?|stages?)\s*"
    r"\d+(?:\s*(?:[-\u2013&,]|and|to)\s*\d+)*\b", re.IGNORECASE)


def _report_numbers_in_sentence(sentence: str) -> List[Decimal]:
    # Remove the metric label's own ordinal (e.g. the "1" in "Tier 1", the "2"
    # in "Scope 2") so it is not mistaken for a reported value.
    sentence = _METRIC_LABEL_ORDINAL_RE.sub(" ", str(sentence or ""))
    nums = []
    for m in re.finditer(r"-?\d{1,3}(?:,\d{3})+(?:\.\d+)?|-?\d+(?:\.\d+)?", sentence):
        d = _arith_to_decimal(m.group(0))
        if d is not None:
            nums.append(d)
    return nums


def report_numeric_consistency_issues(sections: Dict[str, str]) -> List[Dict[str, Any]]:
    """Detect (a) the same value rendered at different precisions across the
    report, and (b) conflicting values reported for the same labelled metric."""
    issues: List[Dict[str, Any]] = []

    # (a) precision drift: values that are equal within 0.05% but printed differently.
    seen_tokens: Dict[str, str] = {}   # normalized value -> first raw rendering
    for section, md in sections.items():
        for m in re.finditer(r"-?\d{1,3}(?:,\d{3})+(?:\.\d+)?|-?\d+\.\d+", str(md or "")):
            raw = m.group(0)
            d = _arith_to_decimal(raw)
            if d is None or abs(d) < 10:
                continue
            key = f"{float(d):.4g}"
            if key in seen_tokens and seen_tokens[key] != raw:
                issues.append({
                    "type": "precision_inconsistency",
                    "value_a": seen_tokens[key],
                    "value_b": raw,
                    "section": section,
                    "note": "Same underlying value rendered with different precision across the report.",
                })
            else:
                seen_tokens.setdefault(key, raw)

    # (b) labelled-metric conflicts: same metric label, materially different numbers.
    #
    # Attribution rule: a number is assigned to the metric whose label is NEAREST to
    # it in the sentence, not to every metric mentioned in the sentence. The old
    # "any label present -> all numbers" rule caused one value (e.g. 202.0) to be
    # attributed to every metric a multi-metric sentence mentioned, polluting the
    # conflict lists. We also require the number to sit within a small character
    # window of the label, so unrelated figures elsewhere in a long sentence are not
    # captured.
    # Role/unit context tokens: numbers are only compared WITHIN the same role, so a
    # baseline is never flagged as conflicting with an interim milestone, a target
    # ambition percentage, opex, or a projection. This prevents false positives on
    # well-structured IFRS target disclosures (baseline -> milestones -> target),
    # which are legitimate multi-value series, not contradictions.
    _ROLE_TOKENS = {
        "baseline": ("baseline",),
        "target": ("target ambition", "target value", "reduction", "% reduction", "ambition"),
        "milestone": ("milestone", "interim", "2025", "2026", "2027", "2028", "2029", "2030"),
        "opex": ("opex", "operating expenditure"),
        "projection": ("projected", "projection", "scenario", "forecast", "estimated future"),
    }

    def _role_of(sentence_lower: str, num_pos_in_sentence: int) -> str:
        # A number's role is inferred from nearby role tokens; default "reported".
        window = sentence_lower[max(0, num_pos_in_sentence - 70):num_pos_in_sentence + 20]
        for role, toks in _ROLE_TOKENS.items():
            if any(t in window for t in toks):
                return role
        return "reported"

    # A year immediately attached to a value (e.g. "EUR 322.2 million (2022)") marks
    # it as one point in a time series -> exempt from conflict comparison.
    _YEAR_NEAR_RE = re.compile(r"\((?:19|20)\d{2}\)|\b(?:19|20)\d{2}\b")

    metric_values: Dict[Tuple[str, str, str], List[Tuple[str, Decimal]]] = defaultdict(list)
    max_window = int(os.getenv("IFRS_METRIC_LABEL_WINDOW_CHARS", "60"))

    def _analysis_units(markdown: str) -> List[str]:
        """Split report text into attribution units.

        Markdown TABLE ROWS are emitted as individual units. Previously pipes were
        replaced with spaces and the whole table was sentence-split, which merged
        rows: a "Scope 2 (market-based)" row inherited the "location-based"
        qualifier from the row above, and a carbon-price cell was read as an
        emissions value from a neighbouring row. Each row is now self-contained.
        """
        units: List[str] = []
        prose_buffer: List[str] = []
        for line in str(markdown or "").splitlines():
            stripped = line.strip()
            if stripped.startswith("|") and stripped.count("|") >= 2:
                # Flush any pending prose, then emit this row on its own.
                if prose_buffer:
                    units.extend(_SENTENCE_SPLIT_RE.split(" ".join(prose_buffer)))
                    prose_buffer = []
                # Skip separator rows (|---|---|).
                if re.fullmatch(r"\|[\s\-:|]+\|", stripped):
                    continue
                units.append(re.sub(r"\|", "  ", stripped))
            else:
                prose_buffer.append(line)
        if prose_buffer:
            units.extend(_SENTENCE_SPLIT_RE.split(" ".join(prose_buffer)))
        return [u for u in units if u and u.strip()]

    for section, md in sections.items():
        for sent in _analysis_units(md):
            sl = sent.lower()
            # Record each label's start AND end position, so a number is measured to
            # the nearest EDGE of the label (a value right after "climate capex ..."
            # is close to that label's end, not its start).
            label_spans: List[Tuple[int, int, str]] = []
            for metric, tokens in _LABELLED_METRIC_TOKENS.items():
                for tok in tokens:
                    start = 0
                    while True:
                        idx = sl.find(tok, start)
                        if idx == -1:
                            break
                        label_spans.append((idx, idx + len(tok), metric))
                        start = idx + len(tok)
            if not label_spans:
                continue
            # Mask label ordinals ("Tier 1"/"Scope 2") AND chemical-formula unit
            # tokens ("tCO2e", "CO2e", "CO2") so their embedded digits are not read
            # as reported values.
            masked = _METRIC_LABEL_ORDINAL_RE.sub(lambda m: " " * len(m.group(0)), sent)
            masked = re.sub(r"(?i)t?co2e?|tco2e|co\u2082e?", lambda m: " " * len(m.group(0)), masked)

            def _distance_to_label(num_pos: int, span: Tuple[int, int, str]) -> Tuple[int, int]:
                start_i, end_i, _ = span
                # Prefer labels whose END precedes the number (metric named before its
                # value). Distance is 0 inside the label, else gap to nearest edge.
                if num_pos >= end_i:
                    return (0, num_pos - end_i)        # number after label: preferred
                if num_pos < start_i:
                    return (1, start_i - num_pos)       # number before label: deprioritised
                return (0, 0)

            for m in re.finditer(r"-?\d{1,3}(?:,\d{3})+(?:\.\d+)?|-?\d+(?:\.\d+)?", masked):
                d = _arith_to_decimal(m.group(0))
                if d is None or abs(d) < 1:
                    continue
                # Skip bare 4-digit calendar years: "the Bank reports 2024 fleet
                # Scope 1" is a period reference, not a Scope 1 quantity.
                if re.fullmatch(r"(?:19|20)\d{2}(?:\.0)?", m.group(0)) and 1900 <= float(d) <= 2100:
                    continue
                num_pos = m.start()
                nearest = min(label_spans, key=lambda sp: _distance_to_label(num_pos, sp))
                _, gap = _distance_to_label(num_pos, nearest)
                if gap > max_window:
                    continue
                # Enumeration / list guard. In "net interest income of X, net profit
                # of Y, Tier 1 capital of Z" each number belongs to its OWN clause,
                # but only "Tier 1 capital" may be a recognised metric label. To avoid
                # attributing X and Y (other metrics' values) to that one label, we
                # require the number and the label to be in the SAME clause: no comma
                # or semicolon may sit between them on either side. A comma marks a new
                # list item / metric, so a value across a comma is a different metric.
                n_start, n_end, _n_metric = nearest
                lo, hi = (n_end, num_pos) if num_pos >= n_end else (num_pos, n_start)
                between = sent[lo:hi]
                # Ignore digit-grouping commas (1,234) when testing for a clause break.
                clause_break = re.sub(r"(?<=\d),(?=\d{3}\b)", "", between)
                if "," in clause_break or ";" in clause_break:
                    continue
                # Another recognised label between this label and the number also means
                # the number belongs to that other clause.
                if any(lo <= sp_start < hi for (sp_start, sp_end, _mk) in label_spans
                       if (sp_start, sp_end) != (n_start, n_end)):
                    continue
                # Skip values that ARE years (e.g. "the Bank reports 2024 fleet
                # Scope 1"): a year is a period label, not a measurement.
                if d == d.to_integral_value() and 1900 <= int(d) <= 2100 and "." not in m.group(0):
                    continue
                # Skip values that are part of an explicit time series (year attached
                # nearby): a trend of yearly figures is legitimate, not a conflict.
                # The trailing window is generous enough to catch "322.2 million (2022)"
                # where a unit word separates the value from its year.
                nearby = sent[max(0, num_pos - 15):num_pos + 40]
                if _YEAR_NEAR_RE.search(nearby):
                    continue
                # Unit guard: a per-unit PRICE ("EUR 42.73/tCO2e"), a data-quality
                # SCORE ("PCAF data quality score: 2"), or an attribution-factor style
                # figure is a different quantity from the metric total, even when the
                # metric label appears in the same sentence. Skip these so a carbon
                # price applied "to ... financed emissions" is not read as a financed-
                # emissions value.
                trailing = sent[m.end():m.end() + 12].lower()
                leading = sent[max(0, num_pos - 30):num_pos].lower()
                if re.search(r"^\s*/\s*t?co\u2082?e|^\s*per\s+tco\u2082?e", trailing):
                    continue  # per-tonne price
                if "data quality score" in leading or "dq score" in leading or "pcaf" in leading[-14:]:
                    continue  # data-quality score
                # Reject implausibly small values for emissions-quantity metrics:
                # a bare "1"/"2" adjacent to a Scope label is a PCAF data-quality
                # score or a scope ordinal, not an emissions figure.
                if nearest[2] in ("scope1", "scope2", "scope3", "financed_emissions") and abs(d) < 10:
                    continue
                role = _role_of(sl, num_pos)
                # Accounting-method qualifier near the number (location/market-based,
                # gross/net, etc.). Values under DIFFERENT qualifiers are distinct
                # required disclosures, not conflicts.
                qual_window = sent[max(0, num_pos - 80):num_pos + 20].lower()
                if "location-based" in qual_window or "location based" in qual_window:
                    qualifier = "location"
                elif "market-based" in qual_window or "market based" in qual_window:
                    qualifier = "market"
                elif "gross" in qual_window:
                    qualifier = "gross"
                elif "net" in qual_window:
                    qualifier = "net"
                else:
                    qualifier = ""
                metric_values[(nearest[2], role, qualifier)].append((section, d))
    for (metric, role, qualifier), vals in metric_values.items():
        # Only "reported" current values are expected to be single-valued across the
        # report. Baselines, targets, milestones, opex and projections are allowed to
        # differ by design, so they are recorded but not flagged as conflicts.
        if role != "reported":
            continue
        uniq = {}
        for section, num in vals:
            uniq.setdefault(f"{float(num):.4g}", (section, num))
        distinct = list(uniq.values())
        if len(distinct) < 2:
            continue
        nums_only = [float(n) for _, n in distinct]

        # Materiality filter: drop values that are negligible relative to the metric's
        # dominant (largest) value. A stray "2.0" or "29.0" sitting next to a
        # 35,973,167.7 financed-emissions figure is not a competing measurement of the
        # same quantity -- it is ordinal/table noise from an adjacent cell. Keep only
        # values within a plausible band (>= 1% of the max) so genuine conflicts of
        # comparable magnitude are still caught.
        dominant = max(abs(v) for v in nums_only) or 1.0
        material = [(sec, n) for sec, n in distinct if abs(float(n)) >= 0.01 * dominant]
        if len(material) < 2:
            continue
        mat_nums = [float(n) for _, n in material]
        spread = (max(mat_nums) - min(mat_nums)) / max(abs(max(mat_nums)), 1e-9)
        if spread > IFRS_ARITHMETIC_REL_TOL:
            issues.append({
                "type": "labelled_metric_conflict",
                "metric": metric,
                "role": role,
                "method_qualifier": qualifier or None,
                "values": [{"section": s, "value": float(n)} for s, n in material][:8],
                "note": "The same reported (current-value) metric is given materially different values across the report.",
            })

    return issues


def run_payload_arithmetic_audit() -> Dict[str, Any]:
    """Run payload-level arithmetic checks now and persist an audit artifact.

    payload_arithmetic_issues(None) already iterates every section and dedups, so
    it IS the full audit. The previous version also summed per-section results on
    top of it, double-counting every issue. We report the deduplicated set once and
    additionally group it by section for readability.
    """
    all_issues = payload_arithmetic_issues(None)
    per_section: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
    for issue in all_issues:
        per_section[issue.get("scope", "unknown")].append(issue)
    total = len(all_issues)
    audit = {
        "strict_mode": IFRS_ARITHMETIC_STRICT,
        "issues": all_issues,
        "issues_by_section": dict(per_section),
        "total_issue_count": total,
    }
    write_json(audit, DIRS["audit_logs"] / "payload_arithmetic_audit.json")
    print(f"Payload arithmetic audit: {total} issue(s). See {DIRS['audit_logs'] / 'payload_arithmetic_audit.json'}")
    if total and IFRS_ARITHMETIC_STRICT:
        raise ValueError(
            f"Payload arithmetic validator found {total} coherence issue(s) with IFRS_ARITHMETIC_STRICT=1. "
            f"See {DIRS['audit_logs'] / 'payload_arithmetic_audit.json'}"
        )
    return audit


payload_arithmetic_audit = run_payload_arithmetic_audit()


Payload arithmetic audit: 1 issue(s). See C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\audit_logs\payload_arithmetic_audit.json


## Deterministic evidence mapping and coverage

The evidence mapper is code-first. It maps requirements to actual payload paths. The optional LLM fuzzy mapper can suggest matches, but the path must still resolve to the payload.

Missing requirements are written to JSON audit outputs and are **not passed to the Writer as report content**.

In [10]:
# ============================================================
# CELL 7 — DETERMINISTIC EVIDENCE MAPPER
# PAYLOAD-AWARE RULE: payload-aware mapping.
#
# Why:
# - Your requirement files are nested by standard and now load correctly.
# - Your payloads are rich section payloads with recurring metadata, bank,
#   reporting_kpis and section-specific tables.
# - Pure lexical matching can over-map broad IFRS requirements to weak fields.
#
# This mapper still remains deterministic/code-first, but adds:
# - section root routing
# - evidence tag routing
# - phrase-to-field boosts
# - metadata noise filtering
# ============================================================

STOPWORDS = {
    "the", "and", "for", "with", "that", "this", "from", "into", "about", "their",
    "shall", "should", "must", "entity", "entities", "information", "disclose",
    "disclosure", "disclosed", "disclosures", "related", "sustainability", "climate",
    "risks", "risk", "opportunities", "opportunity", "reporting", "period",
    "including", "describe", "explain", "enable", "users", "general", "purpose",
    "financial", "reports", "understand", "specific", "specifically", "current",
    "anticipated", "effects", "used", "uses", "use", "accordance", "paragraph",
    "paragraphs", "standard", "standards", "ifrs", "prepare", "preparing",
}

# Section-level allowed roots. These match the top-level tables/objects in your
# BANK01 payload files.
SECTION_ROOTS = {
    "General Requirements": {
        "metadata", "bank", "financial_summary", "general_requirements_context",
        "targets", "scope1", "scope2", "scope3_travel", "financed_emissions",
        "reporting_kpis",
    },
    "Governance": {
        "bank", "governance", "board_minutes", "climate_risk_register",
        "reporting_kpis",
    },
    "Strategy": {
        "metadata", "bank", "financial_summary", "climate_scenarios",
        "climate_risk_register", "value_chain_map", "climate_opportunities",
        "targets", "transition_plan", "resilience_assessment",
        "climate_financial_effects", "reporting_kpis",
    },
    "Risk Management": {
        "metadata", "bank", "climate_risk_register", "physical_risk_exposures",
        "value_chain_map", "governance", "climate_financial_effects",
        "reporting_kpis",
    },
    "Metrics and Targets": {
        "metadata", "bank", "financial_summary", "scope1", "scope2",
        "scope3_travel", "financed_emissions", "financed_emissions_equity",
        "financed_emissions_sovereign", "targets", "carbon_credits",
        "internal_carbon_price", "scope3_categories", "ghg_methodology",
        "scope12_consolidation", "reporting_kpis",
    },
}

# Route IFRS evidence tags to likely payload roots.
TAG_ROOTS = {
    "governance_body": {"governance", "board_minutes"},
    "management_role": {"governance", "board_minutes"},
    "remuneration": {"governance", "board_minutes"},
    "risk_process": {"climate_risk_register", "physical_risk_exposures", "governance", "climate_financial_effects"},
    "scenario_analysis": {"climate_scenarios", "climate_risk_register", "physical_risk_exposures", "resilience_assessment"},
    "business_model_value_chain": {"value_chain_map", "climate_scenarios", "climate_risk_register", "climate_financial_effects"},
    "strategy_decision_making": {"transition_plan", "targets", "climate_opportunities", "climate_scenarios", "climate_risk_register"},
    "financial_effects": {"financial_summary", "climate_financial_effects", "climate_scenarios", "reporting_kpis"},
    "metrics": {
        "scope1", "scope2", "scope3_travel", "financed_emissions",
        "targets", "financial_summary", "reporting_kpis", "ghg_methodology",
        "scope12_consolidation", "scope3_categories", "internal_carbon_price",
        "carbon_credits", "financed_emissions_equity", "financed_emissions_sovereign",
    },
    "targets": {"targets", "reporting_kpis", "governance"},
    "ghg_emissions": {"scope1", "scope2", "scope3_travel", "financed_emissions", "ghg_methodology", "scope12_consolidation", "scope3_categories"},
    "scope_1": {"scope1", "scope12_consolidation"},
    "scope_2": {"scope2", "scope12_consolidation"},
    "scope_3": {"scope3_travel", "scope3_categories", "financed_emissions"},
    "materiality": {"general_requirements_context", "metadata", "climate_risk_register", "value_chain_map"},
    "connected_information": {"general_requirements_context", "financial_summary", "bank", "metadata"},
    "source_guidance": {"general_requirements_context", "metadata", "ghg_methodology"},
}

# Phrase rules connect common IFRS wording to expected payload roots and fields.
# Each rule: (phrases in requirement text, preferred roots, path/value hints)
PHRASE_RULES = [
    (["governance body", "body", "board", "committee", "charged with governance"], ["governance", "board_minutes"], ["board", "committee", "governance", "members_present", "meeting"]),
    (["skills", "competencies", "competence"], ["governance"], ["skill", "expertise", "training", "development", "competenc"]),
    (["how often", "informed"], ["governance", "board_minutes"], ["frequency", "meeting", "minutes", "agenda", "reporting_to_board", "climate_risk_reporting"]),
    (["major transactions", "trade-offs", "trade offs"], ["governance", "board_minutes", "transition_plan"], ["major_transactions", "decision", "trade", "transition_plan"]),
    (["targets", "progress"], ["targets", "governance", "reporting_kpis"], ["target", "progress", "baseline", "remuneration"]),
    (["remuneration", "compensation"], ["governance"], ["compensation", "remuneration", "ceo", "exec"]),
    (["management", "controls", "procedures"], ["governance", "climate_risk_register"], ["management", "committee", "erm", "control", "integrated"]),
    (["identify", "assess", "prioritise", "prioritize", "monitor"], ["climate_risk_register", "physical_risk_exposures"], ["risk_rating", "likelihood", "severity", "monitoring_frequency", "risk_name", "risk_category", "risk_description", "high_risk_flag"]),
    (["scenario analysis"], ["climate_scenarios", "climate_risk_register", "resilience_assessment"], ["scenario", "scenario_analysis", "framework", "horizon", "methodology", "resilience"]),
    (["changed", "previous reporting period"], ["climate_risk_register"], ["changed_since_prior_period"]),
    (["integrated", "overall risk management"], ["climate_risk_register", "governance"], ["erm_integrated", "erm_integration", "management"]),
    (["business model", "value chain"], ["value_chain_map"], ["value_chain", "node", "upstream", "downstream", "business_model"]),
    (["financial position", "financial performance", "cash flows", "financial effects"], ["climate_financial_effects", "financial_summary"], ["affected_statement", "line_item", "quantitative_effect", "financial", "cash", "performance", "revenue", "profit"]),
    (["resilience", "climate resilience"], ["resilience_assessment", "climate_scenarios"], ["resilience", "capacity", "scenario", "uncertainties"]),
    (["transition plan"], ["transition_plan", "climate_scenarios", "targets"], ["transition_plan", "net_zero", "dependencies", "resourcing", "assumptions"]),
    (["greenhouse gas", "ghg", "emissions", "co2"], ["scope1", "scope2", "scope3_travel", "financed_emissions", "ghg_methodology"], ["scope", "emissions", "tco2e", "ghg"]),
    (["scope 1"], ["scope1", "scope12_consolidation"], ["scope1"]),
    (["scope 2"], ["scope2", "scope12_consolidation"], ["scope2", "market", "location"]),
    (["scope 3", "financed emissions"], ["scope3_travel", "financed_emissions", "scope3_categories", "financed_emissions_equity", "financed_emissions_sovereign"], ["scope3", "financed", "category", "attributed"]),
    (["carbon price", "internal carbon"], ["internal_carbon_price", "climate_scenarios"], ["carbon_price", "internal_carbon"]),
    (["capital deployment", "capital expenditure", "financing", "investment deployed"], ["financial_summary", "reporting_kpis"], ["capex", "opex", "climate_capex", "investment", "financing"]),
    (["comparative", "revised comparative", "redefines", "replaces", "estimate"], ["financial_summary", "scope1", "scope2", "financed_emissions", "metadata"], ["2022", "2023", "comparative", "estimate", "data_gaps"]),
    (["data source", "inputs", "parameters", "measurement approach", "method"], ["metadata", "ghg_methodology", "scope12_consolidation", "climate_scenarios", "physical_risk_exposures"], ["method", "source", "data_source", "input", "assumption", "basis", "scope"]),
    (["reporting entity", "same reporting entity", "financial statements", "currency", "reporting period"], ["bank", "general_requirements_context", "financial_summary"], ["reporting", "currency", "entity", "period", "fiscal", "boundary"]),
    (["material"], ["general_requirements_context", "metadata", "climate_risk_register"], ["materiality", "material", "risk_rating", "high_risk"]),
]

NOISE_PATH_FRAGMENTS = [
    "coherence_fixes_applied",
]


def root_of_path(path: str) -> str:
    return re.split(r"[.\[]", str(path), maxsplit=1)[0]


def requirement_text_blob(req: Dict[str, Any]) -> str:
    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        tags = [tags]
    return " ".join([
        str(req.get("requirement_text", "")),
        str(req.get("clause_path", "")),
        " ".join([str(t).replace("_", " ") for t in tags]),
    ]).lower()


def allowed_roots_for_requirement(section_name: str, req: Dict[str, Any]) -> set:
    roots = set(SECTION_ROOTS.get(section_name, set()))

    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        tags = [tags]

    for tag in tags:
        roots |= TAG_ROOTS.get(str(tag), set())

    text = requirement_text_blob(req)
    for phrases, preferred_roots, _path_hints in PHRASE_RULES:
        if any(phrase in text for phrase in phrases):
            roots |= set(preferred_roots)

    return roots


def requirement_keywords(req: Dict[str, Any]) -> List[str]:
    parts = [
        req.get("requirement_text", ""),
        req.get("clause_path", ""),
        req.get("obligation_type", ""),
        req.get("banking_relevance", ""),
    ]

    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        try:
            tags = json.loads(tags)
        except Exception:
            tags = re.split(r"[,;|]", tags)
    elif not isinstance(tags, list):
        tags = [tags] if tags else []

    parts.extend([str(t).replace("_", " ") for t in tags])

    kws = []
    for part in parts:
        kws.extend(tokens(str(part).replace("_", " ")))

    return sorted(set([k for k in kws if k not in STOPWORDS and len(k) > 2]))


def field_keywords(path: str, value: Any) -> List[str]:
    text = str(path).replace("_", " ").replace(".", " ")
    if isinstance(value, (str, int, float, bool)):
        text += " " + str(value).replace("_", " ")
    elif isinstance(value, (dict, list)):
        text += " " + value_preview(value, limit=500).replace("_", " ")
    return [t for t in tokens(text) if t not in STOPWORDS and len(t) > 2]


def phrase_path_boost(req: Dict[str, Any], path: str, value: Any) -> Tuple[int, List[str]]:
    text = requirement_text_blob(req)
    path_text = str(path).lower().replace("_", " ")
    value_text = str(value).lower().replace("_", " ") if isinstance(value, (str, int, float, bool)) else ""
    root = root_of_path(path)

    total = 0
    hits = []

    for phrases, preferred_roots, path_hints in PHRASE_RULES:
        if not any(phrase in text for phrase in phrases):
            continue

        matched_hints = [
            hint for hint in path_hints
            if hint.replace("_", " ") in path_text or hint.replace("_", " ") in value_text
        ]

        if matched_hints:
            total += 4
            hits.extend(matched_hints[:4])
        elif root in preferred_roots:
            total += 2

    return total, sorted(set(hits))


def metadata_allowed_for_requirement(req: Dict[str, Any], path: str) -> bool:
    """
    Metadata is valuable for data gaps, methodology, reporting basis and assumptions,
    but should not dominate every requirement.
    """
    text = requirement_text_blob(req)
    relevant_terms = [
        "data", "comparative", "estimate", "measurement", "method", "source",
        "unavailable", "gap", "limitation", "scope", "basis", "assumption",
        "currency", "period", "reporting entity", "financial statements",
        "guidance",
    ]

    if "metadata.data_gaps" in path:
        return any(term in text for term in relevant_terms)

    if "metadata.pcaf_methodology" in path:
        return any(term in text for term in ["method", "source", "emission", "financed", "scope 3", "data", "estimate"])

    return True


def evidence_score(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Tuple[int, List[str], str]:
    root = root_of_path(path)

    if any(fragment in str(path) for fragment in NOISE_PATH_FRAGMENTS):
        return -999, [], "excluded_noise_path"

    if root == "metadata" and not metadata_allowed_for_requirement(req, str(path)):
        return -999, [], "metadata_not_relevant_to_requirement"

    allowed_roots = allowed_roots_for_requirement(section_name, req)

    req_kws = set(requirement_keywords(req))
    f_kws = set(field_keywords(path, value))
    overlap = sorted(req_kws.intersection(f_kws))

    score = len(overlap)

    path_lower = str(path).lower()
    for kw in req_kws:
        if len(kw) > 3 and kw in path_lower:
            score += 1

    route_reason = "lexical"

    if allowed_roots:
        if root in allowed_roots:
            score += 3
            route_reason = "payload_root_routing+lexical"
        else:
            score -= 3
            route_reason = "outside_expected_payload_root"

    boost, phrase_hits = phrase_path_boost(req, path, value)
    if boost:
        score += boost
        route_reason = "payload_root_routing+phrase_boost+lexical"

    # Mild recency/context boost; never sufficient alone.
    if "reporting_year" in path_lower or "2024" in str(value):
        score += 1

    return score, sorted(set(overlap + phrase_hits)), route_reason


def build_evidence_map_for_section(
    section_name: str,
    requirements: List[Dict[str, Any]],
    payload: Dict[str, Any],
    top_k: int = 8,
    min_score: int = 5,
) -> List[Dict[str, Any]]:
    flat = flatten_json(payload)
    non_empty_items = [(p, v) for p, v in flat.items() if not is_empty_value(v)]
    mapped = []

    for req in requirements:
        candidates = []
        for path, value in non_empty_items:
            score, matched_terms, reason = evidence_score(req, section_name, path, value)

            if score >= min_score:
                candidates.append({
                    "payload_path": path,
                    "payload_root": root_of_path(path),
                    "value_preview": value_preview(value),
                    "value_type": type(value).__name__,
                    "match_score": score,
                    "matched_keywords": matched_terms,
                    "mapping_reason": reason,
                })

        candidates = sorted(
            candidates,
            key=lambda x: (x["match_score"], len(x.get("matched_keywords", []))),
            reverse=True,
        )[:top_k]

        mapped.append({
            "requirement_id": req["requirement_id"],
            "section_name": section_name,
            "requirement_text": req["requirement_text"],
            "mandatory": req["mandatory"],
            "evidence_candidates": candidates,
            "mapping_method": "deterministic_payload_aware",
        })

    return mapped


def summarize_evidence_map(section_name: str, evidence_map: List[Dict[str, Any]]) -> Dict[str, Any]:
    roots = Counter()
    candidate_counts = []
    for row in evidence_map:
        candidate_counts.append(len(row.get("evidence_candidates", [])))
        for c in row.get("evidence_candidates", []):
            roots[c.get("payload_root") or root_of_path(c.get("payload_path", ""))] += 1

    covered = sum(1 for x in candidate_counts if x > 0)
    return {
        "section_name": section_name,
        "requirements_total": len(evidence_map),
        "requirements_with_candidates": covered,
        "requirements_without_candidates": len(evidence_map) - covered,
        "candidate_root_distribution": dict(roots.most_common()),
    }



# Performance cache: prevents recomputing requirement/path tokens for every candidate pair.
_REQUIREMENT_TEXT_BLOB_CACHE = {}
_REQUIREMENT_KEYWORDS_CACHE = {}
_FIELD_KEYWORDS_CACHE = {}


def requirement_text_blob(req: Dict[str, Any]) -> str:  # noqa: F811 - intentional cached override
    rid = str(req.get("requirement_id", id(req)))
    if rid in _REQUIREMENT_TEXT_BLOB_CACHE:
        return _REQUIREMENT_TEXT_BLOB_CACHE[rid]
    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        tags = [tags]
    text = " ".join([
        str(req.get("requirement_text", "")),
        str(req.get("clause_path", "")),
        " ".join([str(t).replace("_", " ") for t in tags]),
    ]).lower()
    _REQUIREMENT_TEXT_BLOB_CACHE[rid] = text
    return text


def requirement_keywords(req: Dict[str, Any]) -> List[str]:  # noqa: F811 - intentional cached override
    rid = str(req.get("requirement_id", id(req)))
    if rid in _REQUIREMENT_KEYWORDS_CACHE:
        return _REQUIREMENT_KEYWORDS_CACHE[rid]
    parts = [
        req.get("requirement_text", ""),
        req.get("clause_path", ""),
        req.get("obligation_type", ""),
        req.get("banking_relevance", ""),
    ]
    tags = req.get("evidence_tags", [])
    if isinstance(tags, str):
        try:
            tags = json.loads(tags)
        except Exception:
            tags = re.split(r"[,;|]", tags)
    elif not isinstance(tags, list):
        tags = [tags] if tags else []
    parts.extend([str(t).replace("_", " ") for t in tags])
    kws = []
    for part in parts:
        kws.extend(tokens(str(part).replace("_", " ")))
    result = sorted(set([k for k in kws if k not in STOPWORDS and len(k) > 2]))
    _REQUIREMENT_KEYWORDS_CACHE[rid] = result
    return result


def field_keywords(path: str, value: Any) -> List[str]:  # noqa: F811 - intentional cached override
    # Path is unique inside a flattened payload. Include a short value preview to avoid collisions across sections.
    key = (str(path), type(value).__name__, value_preview(value, limit=200) if not isinstance(value, (int, float, bool)) else str(value))
    if key in _FIELD_KEYWORDS_CACHE:
        return _FIELD_KEYWORDS_CACHE[key]
    text = str(path).replace("_", " ").replace(".", " ")
    if isinstance(value, (str, int, float, bool)):
        text += " " + str(value).replace("_", " ")
    elif isinstance(value, (dict, list)):
        text += " " + value_preview(value, limit=500).replace("_", " ")
    result = [t for t in tokens(text) if t not in STOPWORDS and len(t) > 2]
    _FIELD_KEYWORDS_CACHE[key] = result
    return result


# Build and save evidence maps.
evidence_maps_by_section = {}
evidence_map_summaries = {}

for section in SECTIONS:
    evidence_map = build_evidence_map_for_section(
        section,
        requirements_by_section[section],
        payloads_by_section[section],
    )
    evidence_maps_by_section[section] = evidence_map

    summary = summarize_evidence_map(section, evidence_map)
    evidence_map_summaries[section] = summary

    slug = SECTION_SLUGS[section]
    path = DIRS["evidence_maps"] / f"evidence_map_{slug}.json"
    write_json(evidence_map, path)
    write_json(summary, DIRS["evidence_maps"] / f"evidence_map_summary_{slug}.json")

    print(
        section,
        "mapped", len(evidence_map), "requirements |",
        "with candidates:", summary["requirements_with_candidates"],
        "| without:", summary["requirements_without_candidates"],
        "->", path
    )

General Requirements mapped 108 requirements | with candidates: 100 | without: 8 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_general_requirements.json
Governance mapped 15 requirements | with candidates: 15 | without: 0 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_governance.json
Strategy mapped 70 requirements | with candidates: 66 | without: 4 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_strategy.json
Risk Management mapped 17 requirements | with candidates: 17 | without: 0 -> C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\01_evidence_maps\evidence_map_risk_management.json
Metrics and Targets mapped 151 requirements | with candidate

In [11]:
# ============================================================
# CELL 8 — OPTIONAL FUZZY EVIDENCE MAPPER LLM FALLBACK
# Use only for unresolved requirements. The path still must exist.
# ============================================================


def fuzzy_map_unresolved_requirements(
    section_name: str,
    requirements: List[Dict[str, Any]],
    payload: Dict[str, Any],
    evidence_map: List[Dict[str, Any]],
    max_unresolved: int = 20,
) -> List[Dict[str, Any]]:
    unresolved = [m for m in evidence_map if not m["evidence_candidates"]]
    if not unresolved or not USE_FUZZY_EVIDENCE_MAPPER:
        return evidence_map

    unresolved = unresolved[:max_unresolved]
    flat = flatten_json(payload)
    payload_catalog = [
        {"payload_path": p, "value_preview": value_preview(v, 120)}
        for p, v in flat.items()
        if not is_empty_value(v)
    ][:300]

    prompt = {
        "section_name": section_name,
        "task": "Suggest payload paths that may support unresolved IFRS requirements. Only use paths from payload_catalog.",
        "rules": [
            "Do not invent payload paths.",
            "Return an empty list if no path supports a requirement.",
            "A suggested path must be semantically relevant, not merely same section.",
        ],
        "unresolved_requirements": [
            {
                "requirement_id": m["requirement_id"],
                "requirement_text": m["requirement_text"],
                "mandatory": m["mandatory"],
            }
            for m in unresolved
        ],
        "payload_catalog": payload_catalog,
    }

    messages = [
        {"role": "system", "content": "You are a precise evidence mapping assistant. Return JSON only."},
        {"role": "user", "content": json.dumps(prompt, ensure_ascii=False)},
    ]
    raw = azure_chat(
        messages,
        model_tier=MODEL_CONFIG["fuzzy_evidence_mapper"],
        temperature=0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )
    obj = parse_json_response(raw, request_label=f"fuzzy_evidence_mapper_{SECTION_SLUGS[section_name]}")
    suggestions = obj.get("suggestions", [])

    existing_paths = set(flat.keys())
    by_req = defaultdict(list)
    for s in suggestions:
        rid = s.get("requirement_id")
        for path in s.get("payload_paths", []):
            if path in existing_paths:
                by_req[rid].append({
                    "payload_path": path,
                    "value_preview": value_preview(flat[path]),
                    "value_type": type(flat[path]).__name__,
                    "match_score": int(s.get("confidence", 1)),
                    "matched_keywords": ["llm_fuzzy_match"],
                    "llm_reason": s.get("reason", ""),
                })

    for m in evidence_map:
        if not m["evidence_candidates"] and m["requirement_id"] in by_req:
            m["evidence_candidates"] = by_req[m["requirement_id"]]
            m["mapping_method"] = "llm_fuzzy_verified_path"

    return evidence_map

if USE_FUZZY_EVIDENCE_MAPPER:
    for section in SECTIONS:
        updated = fuzzy_map_unresolved_requirements(
            section,
            requirements_by_section[section],
            payloads_by_section[section],
            evidence_maps_by_section[section],
        )
        evidence_maps_by_section[section] = updated
        write_json(updated, DIRS["evidence_maps"] / f"evidence_map_{SECTION_SLUGS[section]}.json")
        print("Fuzzy mapping completed for", section)
else:
    print("Fuzzy evidence mapper disabled.")

Fuzzy evidence mapper disabled.


In [12]:
# ============================================================
# CELL 9 — [flattened] LEGACY COVERAGE BUILD REMOVED
# ============================================================
# The initial classify_requirement_coverage / build_coverage_and_missing_register
# implementations and their execution loop were fully superseded by the strict
# evidence implementation two cells below, which redefines both functions and
# rebuilds coverage_by_section / missing_registers_by_section from scratch.
# Keeping one implementation avoids order-dependent shadowing.
print("Coverage build is performed by the strict-evidence implementation (next cells).")


Coverage build is performed by the strict-evidence implementation (next cells).


## Strict evidence, audit flagging, and section scoring implementation

This implementation keeps missing requirements out of report prose. Missing requirements are written only to audit JSON/Markdown outputs and are used for scoring and human review decisions.

In [13]:
# ============================================================
# CELL 9B — STRICT EVIDENCE, COVERAGE, MISSING FLAGS + SCORING IMPLEMENTATION strict evidence layer
# ============================================================
# Fast post-processor over CELL 7 maps:
# - removes null/NaN/generic evidence candidates
# - adds targeted Strategy/General routes for known high-level clauses
# - recalculates coverage using strong vs medium evidence
# - writes missing-requirement flags as audit-only outputs
# ============================================================

MISSING_LIKE_STRINGS = {
    "", "nan", "none", "null", "na", "n/a", "not applicable", "not_applicable"
}

GENERIC_CONTEXT_LEAVES = {
    "reporting_year", "bank_id", "summary_id", "id", "country", "lei_code",
    "fiscal_year_end", "boundary_type", "reporting_currency", "established_year",
    "headcount", "in_scope_esg_flag", "regulatory_regime"
}

GENERIC_ALLOWED_TERMS = {
    "reporting period", "reporting year", "same reporting", "reporting entity",
    "financial statements", "presentation currency", "currency", "fiscal",
    "comparative", "preceding period", "prior period", "boundary", "general purpose financial reports",
    "same time", "period covered", "longer or shorter than 12 months"
}


# STRICT EVIDENCE RULE — audit-only evidence may help scoring/flagging but must never be
# passed to section writers or used as strong support for "covered".
# Fix 1: `data_gaps` / `data_gap` are NO LONGER audit-only. The estimation- and
# measurement-uncertainty disclosure required by IFRS S1 is built from these
# fields, so the writer must be able to see them. They surface in the report
# only inside IFRS-sanctioned limitation statements (enforced by the two-tier
# scanner); pipeline-internal registers stay audit-only via the leaves below.
AUDIT_ONLY_PATH_FRAGMENTS = {
    "missing_requirement",
    "missing_requirements",
}

AUDIT_ONLY_LEAVES = {
    "sovereign_bonds_with_data_gaps",
    "listed_equity_emissions_are_proxy",
}

def is_audit_only_evidence_path(path: str) -> bool:
    p = str(path).lower()
    leaf = path_leaf(path) if "path_leaf" in globals() else re.split(r"[.\[\]]+", p)[-1]
    return any(fragment in p for fragment in AUDIT_ONLY_PATH_FRAGMENTS) or leaf in AUDIT_ONLY_LEAVES

def writer_evidence_path_allowed(path: str) -> bool:
    """Evidence safety gate for disclosure plans and LLM writer context."""
    return not is_audit_only_evidence_path(path)

STRICT_EXTRA_PHRASE_RULES = [
    (["risks and opportunities that could reasonably be expected", "risks and opportunities", "affect the entity's prospects", "affect the entity’s prospects"],
     ["climate_risk_register", "climate_opportunities", "value_chain_map"],
     ["risk_name", "risk_description", "risk_category", "opportunity", "description", "time_horizon", "materiality"]),
    (["strategy and decision-making", "strategy and decision making", "responded to", "plans to respond", "strategic response"],
     ["transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"],
     ["transition_plan", "resilience", "scenario", "target", "progress", "opportunity", "financial_effect", "mitigation"]),
    (["fair presentation", "complete, neutral and accurate", "faithful representation", "statement of compliance", "apply this standard"],
     ["general_requirements_context", "metadata", "bank"],
     ["standards_basis", "assurance", "reporting", "regulatory_regime", "source_systems"]),
    (["judgements", "approximations", "assumptions", "measurement uncertainty", "sources of measurement uncertainty"],
     ["general_requirements_context", "metadata", "ghg_methodology", "scope12_consolidation", "financial_summary", "reporting_kpis"],
     ["methodology", "assumption", "estimate", "data_gaps", "quality", "source", "pcaf", "scope2_rec_reconciliation"]),
]

_existing_rule_keys = {tuple(r[0]) for r in PHRASE_RULES}
for _rule in reversed(STRICT_EXTRA_PHRASE_RULES):
    if tuple(_rule[0]) not in _existing_rule_keys:
        PHRASE_RULES.insert(0, _rule)

TAG_ROOTS.update({
    "reporting_basis": {"general_requirements_context", "metadata", "bank"},
    "compliance_basis": {"general_requirements_context", "metadata", "bank"},
    "measurement_uncertainty": {"general_requirements_context", "metadata", "ghg_methodology", "scope12_consolidation", "reporting_kpis"},
    "strategy_response": {"transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"},
    "risks_opportunities": {"climate_risk_register", "climate_opportunities", "value_chain_map"},
})

REQUIREMENT_ID_ROUTE_HINTS = {
    "IFRS_S1_29_C01": ({"climate_risk_register", "climate_opportunities"}, {"risk_name", "risk_description", "risk_category", "description", "opportunity_name", "time_horizon"}),
    "IFRS_S1_30_C01": ({"climate_risk_register", "climate_opportunities"}, {"risk_name", "risk_description", "risk_category", "description", "opportunity_name", "time_horizon"}),
    "IFRS_S2_9_C01": ({"climate_risk_register", "climate_opportunities", "climate_scenarios"}, {"risk_name", "risk_description", "risk_category", "scenario", "description", "time_horizon"}),
    "IFRS_S2_10_C01": ({"climate_risk_register", "climate_opportunities", "climate_scenarios"}, {"risk_name", "risk_description", "risk_category", "scenario", "description", "time_horizon"}),
    "IFRS_S2_10_C02": ({"climate_risk_register"}, {"risk_category", "risk_name", "risk_description", "physical", "transition"}),
    "IFRS_S1_29_C03": ({"transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"}, {"transition_plan", "resilience", "methodology_notes", "target", "progress", "opportunity", "mitigation_actions"}),
    "IFRS_S1_33_C01": ({"transition_plan", "climate_scenarios", "climate_opportunities", "targets", "climate_financial_effects"}, {"transition_plan", "resilience", "methodology_notes", "target", "progress", "opportunity", "mitigation_actions"}),
    "IFRS_S2_9_C03": ({"transition_plan", "climate_scenarios", "targets", "climate_financial_effects"}, {"transition_plan", "resilience", "methodology_notes", "target", "progress"}),
    "IFRS_S1_5_C01": ({"general_requirements_context", "metadata", "bank"}, {"standards_basis", "regulatory_regime", "source_systems"}),
    "IFRS_S1_11_C01": ({"general_requirements_context", "metadata"}, {"standards_basis", "source_systems", "assurance", "risk_rating_methodology"}),
    "IFRS_S1_13_C01": ({"general_requirements_context", "metadata"}, {"standards_basis", "source_systems", "assurance", "risk_rating_methodology"}),
    "IFRS_S1_15_C01": ({"general_requirements_context", "metadata", "reporting_kpis"}, {"assurance", "source_systems", "emissions_data_quality", "data_quality"}),
    "IFRS_S1_21_C01": ({"general_requirements_context", "metadata", "financial_summary", "reporting_kpis"}, {"source_systems", "reporting", "financial", "data_gaps"}),
    "IFRS_S1_B39_C01": ({"general_requirements_context", "metadata", "financial_summary", "reporting_kpis"}, {"source_systems", "reporting", "financial", "data_gaps"}),
    "IFRS_S1_25_C02": ({"climate_scenarios", "transition_plan", "climate_opportunities", "targets", "financial_summary"}, {"transition_plan", "resilience", "scenario", "target", "opportunity", "climate_capex"}),
}


def path_leaf(path: str) -> str:
    parts = re.split(r"[.\[\]]+", str(path))
    return next((p for p in reversed(parts) if p and not p.isdigit()), "")


def is_missing_like_value(value: Any) -> bool:
    if value is None:
        return True
    if isinstance(value, float):
        try:
            if pd.isna(value):
                return True
        except Exception:
            pass
    if isinstance(value, str):
        return value.strip().lower() in MISSING_LIKE_STRINGS
    try:
        if not isinstance(value, (list, dict, tuple, set)) and pd.isna(value):
            return True
    except Exception:
        pass
    if isinstance(value, (list, dict)) and len(value) == 0:
        return True
    return False


def is_empty_value(value: Any) -> bool:  # noqa: F811 - intentional notebook override
    return is_missing_like_value(value)


def requirement_allows_generic_context(req: Dict[str, Any], path: str) -> bool:
    leaf = path_leaf(path)
    if leaf not in GENERIC_CONTEXT_LEAVES:
        return True
    text = requirement_text_blob(req)
    return any(term in text for term in GENERIC_ALLOWED_TERMS)


def manual_route_bonus(req: Dict[str, Any], path: str) -> Tuple[int, List[str], bool]:
    rid = str(req.get("requirement_id", ""))
    if rid not in REQUIREMENT_ID_ROUTE_HINTS:
        return 0, [], False
    roots, hints = REQUIREMENT_ID_ROUTE_HINTS[rid]
    root = root_of_path(path)
    path_text = str(path).lower()
    matched = sorted([h for h in hints if h.lower() in path_text])
    if root in roots and matched:
        return 7, matched[:5], True
    if root in roots:
        return 3, [], True
    return -2, [], False


def evidence_candidate_allowed(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Tuple[bool, str]:
    path = str(path)
    if any(fragment in path for fragment in NOISE_PATH_FRAGMENTS):
        return False, "excluded_noise_path"
    if is_missing_like_value(value):
        return False, "excluded_missing_like_value"
    if not requirement_allows_generic_context(req, path):
        return False, "excluded_generic_context_field"
    if root_of_path(path) == "metadata" and not metadata_allowed_for_requirement(req, path):
        return False, "excluded_metadata_not_relevant"
    return True, "allowed"


def evidence_score(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Tuple[int, List[str], str]:  # noqa: F811
    allowed, reason = evidence_candidate_allowed(req, section_name, path, value)
    if not allowed:
        return -999, [], reason

    root = root_of_path(path)
    allowed_roots = allowed_roots_for_requirement(section_name, req)
    manual_bonus, manual_hits, manual_routed = manual_route_bonus(req, path)
    if manual_routed:
        allowed_roots = set(allowed_roots) | {root}

    req_kws = set(requirement_keywords(req))
    f_kws = set(field_keywords(path, value))
    overlap = sorted(req_kws.intersection(f_kws))

    score = len(overlap)
    path_lower = str(path).lower()
    for kw in req_kws:
        if len(kw) > 3 and kw in path_lower:
            score += 1

    route_reason = "lexical"
    if allowed_roots:
        if root in allowed_roots:
            score += 3
            route_reason = "payload_root_routing+lexical"
        else:
            score -= 4
            route_reason = "outside_expected_payload_root"

    boost, phrase_hits = phrase_path_boost(req, path, value)
    if boost:
        score += boost
        route_reason = "payload_root_routing+phrase_boost+lexical"

    if manual_bonus:
        score += manual_bonus
        route_reason = "manual_requirement_route+" + route_reason

    # STRICT EVIDENCE RULE: do not boost generic reporting-year values.
    # Reporting-period fields can be contextual evidence but should not create
    # false "covered" decisions for broader disclosure clauses.

    matched = sorted(set(overlap + phrase_hits + manual_hits))
    return score, matched, route_reason


def evidence_strength(req: Dict[str, Any], candidate: Dict[str, Any]) -> str:
    score = int(candidate.get("match_score", 0) or 0)
    matched = candidate.get("matched_keywords", []) or []
    reason = str(candidate.get("mapping_reason", ""))

    # STRICT EVIDENCE RULE:
    # - Generic context fields and audit-only paths can support context/scoring,
    #   but they are not enough to mark a requirement as fully covered.
    # - This prevents paths like reporting_year, summary_id, boundary_type or
    #   metadata.data_gaps[*] from upgrading coverage to "covered".
    if candidate.get("audit_only_evidence") or candidate.get("generic_context_field"):
        return "medium" if score >= 6 else "weak"

    if score >= 10 and (len(matched) >= 2 or "manual_requirement_route" in reason or "phrase_boost" in reason):
        return "strong"
    if score >= 6:
        return "medium"
    return "weak"


def make_candidate(req: Dict[str, Any], section_name: str, path: str, value: Any) -> Optional[Dict[str, Any]]:
    score, matched_terms, reason = evidence_score(req, section_name, path, value)
    if score < 6:
        return None
    candidate = {
        "payload_path": path,
        "payload_root": root_of_path(path),
        "value_preview": value_preview(value),
        "value_type": type(value).__name__,
        "match_score": score,
        "matched_keywords": matched_terms,
        "mapping_reason": reason,
        "generic_context_field": path_leaf(path) in GENERIC_CONTEXT_LEAVES,
        "audit_only_evidence": is_audit_only_evidence_path(path),
        "writer_safe": writer_evidence_path_allowed(path),
        "missing_like_value": False,
    }
    candidate["evidence_strength"] = evidence_strength(req, candidate)
    return candidate


def targeted_candidate_paths(req: Dict[str, Any], flat_payload: Dict[str, Any]) -> List[str]:
    rid = str(req.get("requirement_id", ""))
    if rid not in REQUIREMENT_ID_ROUTE_HINTS:
        return []
    roots, hints = REQUIREMENT_ID_ROUTE_HINTS[rid]
    out = []
    for path, value in flat_payload.items():
        if root_of_path(path) not in roots:
            continue
        if is_missing_like_value(value):
            continue
        path_lower = path.lower()
        if any(h.lower() in path_lower for h in hints):
            out.append(path)
    return out[:80]


def clean_and_enrich_evidence_map(section_name: str, evidence_map: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    reqs_by_id = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    payload = payloads_by_section[section_name]
    flat = flatten_json(payload)
    cleaned_map = []

    for row in evidence_map:
        req = reqs_by_id[row["requirement_id"]]
        by_path = {}

        # Re-score old candidates under strict rules.
        for c in row.get("evidence_candidates", []):
            path = c.get("payload_path")
            value = get_by_path(payload, path)
            candidate = make_candidate(req, section_name, path, value) if path else None
            if candidate:
                by_path[path] = candidate

        # Targeted enrichment for high-level clauses that lexical matching often misses.
        for path in targeted_candidate_paths(req, flat):
            if path in by_path:
                continue
            candidate = make_candidate(req, section_name, path, flat[path])
            if candidate:
                by_path[path] = candidate

        candidates = sorted(
            by_path.values(),
            key=lambda x: (
                2 if x.get("evidence_strength") == "strong" else 1 if x.get("evidence_strength") == "medium" else 0,
                x["match_score"],
                len(x.get("matched_keywords", [])),
                not x.get("generic_context_field", False),
            ),
            reverse=True,
        )[:8]

        cleaned_row = dict(row)
        cleaned_row["evidence_candidates"] = candidates
        cleaned_row["mapping_method"] = "deterministic_payload_aware_strict_writer_safe_postprocessed"
        cleaned_map.append(cleaned_row)

    return cleaned_map


def classify_requirement_coverage(req: Dict[str, Any], mapping: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    candidates = mapping.get("evidence_candidates", [])
    mandatory = req.get("mandatory", True)
    strong_candidates = [c for c in candidates if c.get("evidence_strength") == "strong"]
    medium_candidates = [c for c in candidates if c.get("evidence_strength") == "medium"]
    max_score = max([c.get("match_score", 0) for c in candidates] or [0])

    if strong_candidates:
        status = "covered"
        selected = strong_candidates[:8]
    elif medium_candidates:
        status = "partially_covered"
        selected = medium_candidates[:8]
    else:
        selected = []
        if mandatory:
            status = "not_available_in_payload"
        else:
            raw_text = json.dumps(req.get("raw", {}), ensure_ascii=False).lower()
            status = "not_applicable" if any(x in raw_text for x in ["if applicable", "when applicable", "where applicable", "conditional"]) else "not_available_in_payload"

    return {
        "requirement_id": req["requirement_id"],
        "standard": req.get("standard", ""),
        "paragraph_id": req.get("paragraph_id", ""),
        "report_section": req.get("report_section", ""),
        "requirement_text": req.get("requirement_text", ""),
        "mandatory": mandatory,
        "coverage_status": status,
        "evidence_count": len(selected),
        "raw_candidate_count": len(candidates),
        "strong_candidate_count": len(strong_candidates),
        "medium_candidate_count": len(medium_candidates),
        "evidence_confidence_score": max_score,
        "evidence_paths": [c["payload_path"] for c in selected],
        "coverage_quality_note": "Strict strict evidence layer coverage: covered requires at least one strong, non-null, non-generic, writer-safe evidence candidate.",
        "not_applicable_justification": "Conditional/non-mandatory requirement with no relevant synthetic payload evidence." if status == "not_applicable" else "",
    }


def build_coverage_and_missing_register(section_name: str) -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:  # noqa: F811
    reqs = requirements_by_section[section_name]
    maps = {m["requirement_id"]: m for m in evidence_maps_by_section[section_name]}
    coverage = []
    missing = []

    for req in reqs:
        cov = classify_requirement_coverage(req, maps[req["requirement_id"]])
        coverage.append(cov)
        if cov["coverage_status"] == "not_available_in_payload":
            missing.append({
                "flag_type": "missing_requirement",
                "requirement_id": cov["requirement_id"],
                "standard": cov["standard"],
                "paragraph_id": cov["paragraph_id"],
                "report_section": section_name,
                "mandatory": cov["mandatory"],
                "requirement_text": cov["requirement_text"],
                "coverage_status": "not_available_in_payload",
                "reason": "No sufficiently strong, non-null, non-generic payload evidence was identified for this requirement under strict strict evidence layer coverage rules.",
                "action_needed": "Add real evidence to the section payload, improve deterministic routing, or manually map an existing evidence path after review.",
                "report_instruction": "Do not mention this missing requirement or missing data in the generated report. Keep it only in audit outputs.",
            })

    counts = Counter([c["coverage_status"] for c in coverage])
    section_readiness_score = round(
        100 * (counts.get("covered", 0) + 0.5 * counts.get("partially_covered", 0)) / max(1, len(coverage)),
        2,
    )
    missing_register = {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "policy": "Missing requirements are recorded here and are excluded from report prose.",
        "missing_requirements_count": len(missing),
        "missing_requirement_ids": [m["requirement_id"] for m in missing],
        "section_readiness_score_0_to_100": section_readiness_score,
        "missing_requirements": missing,
    }
    return coverage, missing_register


def rebuild_strict_evidence_coverage_outputs() -> None:
    global evidence_maps_by_section, evidence_map_summaries, coverage_by_section, missing_registers_by_section

    evidence_map_summaries = {}
    coverage_by_section = {}
    missing_registers_by_section = {}

    for section in SECTIONS:
        evidence_maps_by_section[section] = clean_and_enrich_evidence_map(section, evidence_maps_by_section[section])
        summary = summarize_evidence_map(section, evidence_maps_by_section[section])
        evidence_map_summaries[section] = summary
        slug = SECTION_SLUGS[section]
        write_json(evidence_maps_by_section[section], DIRS["evidence_maps"] / f"evidence_map_{slug}.json")
        write_json(summary, DIRS["evidence_maps"] / f"evidence_map_summary_{slug}.json")

        coverage, missing_register = build_coverage_and_missing_register(section)
        coverage_by_section[section] = coverage
        missing_registers_by_section[section] = missing_register
        write_json(coverage, DIRS["coverage"] / f"coverage_matrix_{slug}.json")
        write_json(missing_register, DIRS["missing_requirements"] / f"missing_requirements_{slug}.json")
        counts = Counter([c["coverage_status"] for c in coverage])
        print(
            section,
            "strict candidates:", summary["requirements_with_candidates"], "/", summary["requirements_total"],
            "| coverage:", dict(counts),
            "| missing:", len(missing_register["missing_requirements"]),
            "| readiness:", missing_register["section_readiness_score_0_to_100"],
        )

    combined_missing = {
        "pipeline_mode": PIPELINE_MODE,
        "policy": "The report contains only evidence-supported disclosures. Missing requirements are stored here, used for scoring/flagging, and never included in report prose.",
        "total_missing_requirements": sum(len(m.get("missing_requirements", [])) for m in missing_registers_by_section.values()),
        "sections": missing_registers_by_section,
    }
    write_json(combined_missing, DIRS["missing_requirements"] / "missing_requirements_all_sections.json")


rebuild_strict_evidence_coverage_outputs()


General Requirements strict candidates: 102 / 108 | coverage: {'partially_covered': 59, 'covered': 43, 'not_available_in_payload': 6} | missing: 6 | readiness: 67.13
Governance strict candidates: 15 / 15 | coverage: {'covered': 15} | missing: 0 | readiness: 100.0
Strategy strict candidates: 70 / 70 | coverage: {'covered': 60, 'partially_covered': 10} | missing: 0 | readiness: 92.86
Risk Management strict candidates: 17 / 17 | coverage: {'covered': 10, 'partially_covered': 7} | missing: 0 | readiness: 79.41
Metrics and Targets strict candidates: 139 / 151 | coverage: {'partially_covered': 11, 'covered': 128, 'not_available_in_payload': 12} | missing: 12 | readiness: 88.41


## Deterministic section planner

The planner is code-first. It builds a disclosure plan from covered requirements, available evidence, section blueprints, and table patterns. Missing requirements are excluded from the plan and kept only in JSON audit files.

In [14]:
# ============================================================
# CELL 10 — DETERMINISTIC SECTION PLANNER
# ============================================================

DEFAULT_SECTION_SUBSECTIONS = {
    "General Requirements": [
        {"heading": "Basis of preparation", "keywords": ["basis", "preparation", "compliance", "standard"]},
        {"heading": "Reporting boundary and connected information", "keywords": ["boundary", "entity", "connected", "financial"]},
        {"heading": "Materiality and judgement", "keywords": ["material", "judgement", "estimate", "assumption"]},
    ],
    "Governance": [
        {"heading": "Governance oversight", "keywords": ["board", "committee", "oversight", "governance"]},
        {"heading": "Roles, responsibilities and escalation", "keywords": ["responsibility", "role", "management", "escalation", "report"]},
        {"heading": "Skills, controls and monitoring", "keywords": ["skill", "competence", "control", "monitor", "training"]},
    ],
    "Strategy": [
        {"heading": "Business model and value chain", "keywords": ["business", "model", "value", "chain", "upstream", "downstream"]},
        {"heading": "Sustainability-related risks and opportunities", "keywords": ["risk", "opportunity", "material", "impact"]},
        {"heading": "Time horizons and financial effects", "keywords": ["time", "horizon", "financial", "cash", "performance"]},
        {"heading": "Resilience and strategic response", "keywords": ["resilience", "strategy", "response", "scenario"]},
    ],
    "Risk Management": [
        {"heading": "Risk identification and assessment", "keywords": ["identify", "assessment", "assess", "risk"]},
        {"heading": "Risk management processes and controls", "keywords": ["manage", "process", "control", "mitigation"]},
        {"heading": "Monitoring, reporting and integration", "keywords": ["monitor", "report", "integrat", "escalation"]},
    ],
    "Metrics and Targets": [
        {"heading": "Metrics register", "keywords": ["metric", "value", "unit", "measure"]},
        {"heading": "Targets and progress", "keywords": ["target", "baseline", "progress", "goal"]},
        {"heading": "Methodology and source traceability", "keywords": ["method", "source", "boundary", "definition"]},
    ],
}


def choose_subsection(section_name: str, requirement_text: str) -> str:
    req_tokens = set(tokens(requirement_text))
    candidates = DEFAULT_SECTION_SUBSECTIONS[section_name]
    scored = []
    for sub in candidates:
        score = sum(1 for kw in sub["keywords"] if any(kw in t for t in req_tokens))
        scored.append((score, sub["heading"]))
    scored.sort(reverse=True)
    return scored[0][1] if scored and scored[0][0] > 0 else candidates[0]["heading"]



# final/sanitized disclosure planning layer IMPLEMENTATION: aggressively sanitize authoring blueprints before they enter disclosure plans
# or writer context. Blueprint templates can contain generic layout guidance such
# as "missing data protocol" or "not currently available"; those are useful for
# generic reporting templates but must not reach this report because missing
# requirements are audit/scoring-only.
BLUEPRINT_PROSE_LEAKAGE_PATTERNS = [
    # Explicit missing-data / audit leakage
    "missing requirement",
    "missing data",
    "missing/not applicable",
    "data gap",
    "data gaps",
    "unavailable",
    "not available",
    "not currently available",
    "not reported",
    "not yet covered",
    "not applicable",
    "not material",
    "not currently reported",
    "no data",
    "no available data",
    "not enough data",
    "insufficient data",
    "insufficient evidence",
    "report_instruction",
    "not_available_in_payload",
    "metadata.data_gaps",
    "payload fields",

    # Generic blueprint phrasing that tends to make the writer add gap/limitation prose
    # even when the evidence pack is otherwise clean. These concepts stay audit-only
    # unless explicitly supported by a writer-safe evidence path and a normal requirement.
    "material gaps",
    "scope gaps",
    "gap or area",
    "gaps or areas",
    "current limitations",
    "limitations disclosure",
    "limitations and enhancement",
    "limitations and planned",
    "limitations and next",
    "scope/limitations",
    "key limitations",
    "limitations (",
    "limitation",
    "improvement plans",
    "planned enhancements",
    "future enhancements",
    "continuous improvement",
    "enhancement roadmap",
    "do not leave blanks",
    "status labels",
    "standardized status labels",
    "standardised status labels",
    "data/method constraints",
    "method constraints",
    "data constraints",
]

def blueprint_text_is_writer_safe(text: str) -> bool:
    lower = str(text).lower()
    return not any(pattern in lower for pattern in BLUEPRINT_PROSE_LEAKAGE_PATTERNS)

def sanitize_blueprint_for_report(obj: Any) -> Any:
    """Recursively remove blueprint instructions that could make the writer
    mention missing data, missing requirements, unavailable data, or audit-only
    information in report prose."""
    if isinstance(obj, dict):
        cleaned = {}
        for key, value in obj.items():
            # Remove entire key-value pairs when the key itself is unsafe.
            if not blueprint_text_is_writer_safe(key):
                continue
            cleaned_value = sanitize_blueprint_for_report(value)
            # Drop empty strings/lists/dicts created by filtering.
            if cleaned_value in ({}, [], ""):
                continue
            cleaned[key] = cleaned_value
        return cleaned
    if isinstance(obj, list):
        cleaned = []
        for item in obj:
            # If an item is a prose string and unsafe, remove it.
            if isinstance(item, str):
                if blueprint_text_is_writer_safe(item):
                    cleaned.append(item)
                continue
            # If a dict/list item serializes to unsafe prose, sanitize inside rather than
            # dropping the whole object unless it becomes empty.
            cleaned_item = sanitize_blueprint_for_report(item)
            if cleaned_item not in ({}, [], ""):
                # Defensive second check for fully textual small objects.
                serialized = json.dumps(cleaned_item, ensure_ascii=False)
                if blueprint_text_is_writer_safe(serialized):
                    cleaned.append(cleaned_item)
                else:
                    # Keep only if recursive cleaning removed explicit unsafe pieces;
                    # otherwise drop the object to avoid leakage.
                    if isinstance(cleaned_item, (dict, list)):
                        # If it still contains unsafe text after cleaning, skip.
                        continue
                    cleaned.append(cleaned_item)
        return cleaned
    if isinstance(obj, str):
        return obj if blueprint_text_is_writer_safe(obj) else ""
    return obj

def load_writer_safe_section_blueprint(section_name: str) -> Dict[str, Any]:
    return sanitize_blueprint_for_report(load_section_blueprint(section_name) or {})


def build_disclosure_plan(section_name: str) -> Dict[str, Any]:
    coverage = coverage_by_section[section_name]
    reqs_by_id = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    maps_by_id = {m["requirement_id"]: m for m in evidence_maps_by_section[section_name]}

    include_statuses = {"covered"}
    if ALLOW_PARTIAL_COVERAGE:
        include_statuses.add("partially_covered")

    supported = [c for c in coverage if c["coverage_status"] in include_statuses and c["evidence_count"] > 0]

    subsections = []
    subsection_map = defaultdict(lambda: {
        "heading": "",
        "purpose": "",
        "requirement_ids": [],
        "evidence_paths": [],
        "recommended_format": "short narrative",
    })

    for cov in supported:
        req = reqs_by_id[cov["requirement_id"]]
        heading = choose_subsection(section_name, req["requirement_text"])
        item = subsection_map[heading]
        item["heading"] = heading
        item["purpose"] = f"Address evidence-supported {section_name.lower()} disclosure requirements related to {heading.lower()}."
        item["requirement_ids"].append(cov["requirement_id"])
        # STRICT EVIDENCE RULE: do not pass audit-only evidence, such as metadata.data_gaps,
        # to disclosure plans or the section writer. Missing-data details stay in
        # audit/scoring JSON only.
        item["evidence_paths"].extend([p for p in cov["evidence_paths"] if writer_evidence_path_allowed(p)])

    for heading, item in subsection_map.items():
        item["requirement_ids"] = sorted(set(item["requirement_ids"]))
        item["evidence_paths"] = sorted(set(item["evidence_paths"]))
        if section_name == "Metrics and Targets":
            item["recommended_format"] = "table-first with brief narrative"
        elif section_name in {"Governance", "Risk Management"}:
            item["recommended_format"] = "narrative plus responsibility/process table if evidence supports it"
        elif section_name == "Strategy":
            item["recommended_format"] = "structured narrative plus value-chain/time-horizon table if evidence supports it"
        subsections.append(dict(item))

    if not subsections:
        subsections = [{
            "heading": section_name,
            "purpose": "No evidence-supported requirements were available for report drafting.",
            "requirement_ids": [],
            "evidence_paths": [],
            "recommended_format": "omit section content or mark for human review",
        }]

    # Recommended tables from style table patterns.
    recommended_tables = []
    recommended_columns = TABLE_PATTERNS.get("recommended_columns_by_table_type", {}) if isinstance(TABLE_PATTERNS, dict) else {}
    for table_name, cols in recommended_columns.items():
        t = table_name.lower()
        if section_name.lower().split()[0] in t or (
            section_name == "Metrics and Targets" and "metrics" in t
        ) or (
            section_name == "Risk Management" and "risk" in t
        ):
            recommended_tables.append({"table_name": table_name, "columns": cols})

    plan = {
        "section_name": section_name,
        "pipeline_mode": PIPELINE_MODE,
        "policy": "Plan includes only covered/partially covered evidence-supported requirements. Audit-only evidence paths are excluded from report content.",
        "subsections": subsections,
        # sanitized disclosure planning layer IMPLEMENTATION: sanitize style-derived recommended_tables too.
        # These templates can contain optional columns such as "Notes on scope/limitations",
        # which can make the writer add limitation/missing-data prose even when the
        # evidence pack is clean.
        "recommended_tables": sanitize_blueprint_for_report(recommended_tables)[:4],
        "section_blueprint": load_writer_safe_section_blueprint(section_name),
    }
    return plan

plans_by_section = {}
for section in SECTIONS:
    plan = build_disclosure_plan(section)
    plans_by_section[section] = plan
    write_json(plan, DIRS["plans"] / f"disclosure_plan_{SECTION_SLUGS[section]}.json")
    print(section, "subsections:", len(plan["subsections"]), "recommended tables:", len(plan["recommended_tables"]))

General Requirements subsections: 3 recommended tables: 1
Governance subsections: 2 recommended tables: 2
Strategy subsections: 4 recommended tables: 1
Risk Management subsections: 2 recommended tables: 3
Metrics and Targets subsections: 3 recommended tables: 1


## LLM writer and claims builder

The writer only receives supported requirements and supported evidence. It must not mention missing requirements, synthetic data, missing payloads, or unavailable information.

In [15]:
# ============================================================
# CELL 11 — CONTEXT PACKER FOR LLM AGENTS
# ============================================================


def requirement_subset(section_name: str, requirement_ids: List[str]) -> List[Dict[str, Any]]:
    reqs = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    return [reqs[rid] for rid in requirement_ids if rid in reqs]


def evidence_subset(section_name: str, evidence_paths: List[str], limit_value_chars: int = 500) -> List[Dict[str, Any]]:
    payload = payloads_by_section[section_name]
    out = []
    for path in sorted(set(evidence_paths)):
        # STRICT EVIDENCE RULE: writer and claims agents must never receive audit-only
        # paths such as metadata.data_gaps[*]. Missing information is handled
        # only in audit/scoring outputs.
        if not writer_evidence_path_allowed(path):
            continue
        value = get_by_path(payload, path)
        if value is not None and not is_missing_like_value(value):
            out.append({
                "payload_path": path,
                "value_preview": value_preview(value, limit=limit_value_chars),
                "value_type": type(value).__name__,
            })
    return out


# [flattened] superseded `build_writer_context` removed; single canonical definition retained elsewhere.


def truncate_context(obj: Any, max_chars: int = 60000) -> str:
    text = json.dumps(obj, ensure_ascii=False, indent=2)
    if len(text) <= max_chars:
        return text
    return text[:max_chars] + "\n...TRUNCATED_FOR_TOKEN_LIMIT..."

In [16]:
# ============================================================
# CELL 12 — SECTION WRITER AGENT
# ============================================================


# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.


In [17]:

# ============================================================
# CELL 12B — WRITER CONTEXT + PREFLIGHT IMPLEMENTATION
# ============================================================
# Why this implementation exists:
# - sanitized disclosure planning layer disclosure plans were clean, but the section writer still produced
#   generic template tables, [Insert ...] placeholders, and "missing data" prose.
# - Strategy also produced a "no source content" paragraph because the original
#   writer context placed long requirements/plans before the actual evidence.
#
# Fix:
# - Put evidence_items first in the writer context.
# - Remove generic table/style blueprints from writer context.
# - Force real evidence-derived rows only; no placeholders.
# - Run a local writer preflight and retry once before returning the draft.
# ============================================================

DRAFT_PLACEHOLDER_REGEX = re.compile(r"\[[^\]]+\]")

# Fix 1: WRITER_UNSAFE_PHRASES now lists ONLY Tier-1 pipeline-internal /
# placeholder wording that must never appear in report prose. Tier-2 limitation
# phrases (not available / not reported / data gap / incomplete data ...) were
# removed; they are permitted inside IFRS-sanctioned limitation statements and
# are policed by the two-tier scan_limitation_language() instead.
WRITER_UNSAFE_PHRASES = [
    "[insert",
    "insert risk/opportunity",
    "insert metric",
    "insert definition",
    "insert value",
    "placeholder",
    "source content",
    "provided source content",
    "no entity-specific",
    "no entity specific",
    "missing requirement",
    "payload",
    "synthetic",
]

def compact_supported_requirements_for_writer(section_name: str, requirement_ids: List[str]) -> List[Dict[str, Any]]:
    """Return compact requirements only; no raw requirement metadata."""
    req_by_id = {r["requirement_id"]: r for r in requirements_by_section[section_name]}
    cov_by_id = {c["requirement_id"]: c for c in coverage_by_section.get(section_name, [])}
    out = []
    for rid in requirement_ids:
        req = req_by_id.get(rid)
        if not req:
            continue
        cov = cov_by_id.get(rid, {})
        out.append({
            "requirement_id": rid,
            "standard": req.get("standard", ""),
            "paragraph_id": req.get("paragraph_id", ""),
            "coverage_status": cov.get("coverage_status", ""),
            "requirement_text": req.get("requirement_text", "")[:900],
        })
    return out

def compact_plan_for_writer(plan: Dict[str, Any]) -> Dict[str, Any]:
    """Keep only authoring structure; avoid generic blueprint/table templates."""
    return {
        "section_name": plan.get("section_name", ""),
        "policy": plan.get("policy", ""),
        "subsections": [
            {
                "heading": sub.get("heading", ""),
                "purpose": sub.get("purpose", ""),
                "requirement_ids": sub.get("requirement_ids", []),
                "recommended_format": sub.get("recommended_format", ""),
            }
            for sub in plan.get("subsections", [])
        ],
        "recommended_tables": plan.get("recommended_tables", [])[:2],
    }

def evidence_summary_by_root(evidence_items: List[Dict[str, Any]], max_per_root: int = 40) -> Dict[str, List[Dict[str, Any]]]:
    grouped = defaultdict(list)
    for item in evidence_items:
        path = item.get("payload_path", "")
        root = path.split("[", 1)[0].split(".", 1)[0] if path else "unknown"
        if len(grouped[root]) < max_per_root:
            grouped[root].append(item)
    return dict(grouped)

# [flattened] superseded `build_writer_context` removed; single canonical definition retained elsewhere.

# [flattened] superseded `writer_preflight_issues` removed; single canonical definition retained elsewhere.

# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.


In [18]:

# ============================================================
# CELL 12C — EXPANDED REPORT WRITER IMPLEMENTATION
# ============================================================
# Why this implementation exists:
# - writer preflight layer fixed hallucination, placeholders, and missing-data language.
# - The resulting drafts were safe, but several sections felt truncated and
#   summary-like instead of final-report-like.
#
# Fix:
# - Preserve all writer preflight layer safety rules.
# - Add controlled expansion requirements: richer narrative, explicit evidence
#   explanation, pillar connectivity, and report-quality subsection depth.
# - Add section-specific minimum word targets to prevent approval of overly thin
#   drafts when evidence exists.
# ============================================================

SECTION_EXPANSION_TARGETS = {
    "General Requirements": {
        "min_words": 700,
        "target_words": "800-1,100",
        "depth_focus": [
            "basis of preparation, reporting period, currency and comparatives",
            "material sustainability-related information and why it matters to prospects",
            "connected information across governance, strategy, risk management, and metrics",
            "measurement approaches, assumptions, judgement and data-quality characteristics",
            "comparative consistency and change monitoring",
        ],
    },
    "Governance": {
        "min_words": 800,
        "target_words": "900-1,300",
        "depth_focus": [
            "board oversight, agenda integration and reporting flow",
            "management-level responsibility and committee structures",
            "ERM and major-transaction climate checks",
            "skills, competence and development programme",
            "executive remuneration linkage and how governance information supports decision-making",
        ],
    },
    "Strategy": {
        "min_words": 1_100,
        "target_words": "1,200-1,800",
        "depth_focus": [
            "identified risks and opportunities with time horizons",
            "effects on business model and value chain",
            "strategic response and decision-making trade-offs",
            "scenario resilience findings and transmission channels",
            "financial planning/resource allocation evidence and progress monitoring",
        ],
    },
    "Risk Management": {
        "min_words": 850,
        "target_words": "900-1,300",
        "depth_focus": [
            "risk identification and assessment lifecycle",
            "inputs, data sources, scenario links and rating methodology",
            "prioritisation relative to other risks and ERM integration",
            "monitoring frequencies and changed-since-prior-period indicators",
            "value-chain risk considerations and opportunity handling",
        ],
    },
    "Metrics and Targets": {
        "min_words": 900,
        "target_words": "1,000-1,500",
        "depth_focus": [
            "reporting boundary and period",
            "financed emissions metrics and methodology",
            "operational GHG emissions and Scope 2 treatment",
            "targets, milestones, progress, validation and carbon credits",
            "internal carbon price and financed-emissions data-quality mix",
        ],
    },
}

# Keep the writer preflight layer unsafe phrases and add a few report-depth specific blockers.
WRITER_UNSAFE_PHRASES = sorted(set(WRITER_UNSAFE_PHRASES + [
    "section is intentionally limited",
    "intentionally limited to evidence-supported",
    "no source evidence",
    "no source content",
    "no provided evidence",
    "not enough evidence",
    "insufficient evidence",
    "cannot be determined",
    "could not be determined",
    "template",
]))

# These generic internal-field names can appear in evidence paths, but final prose
# should translate them into readable report language.
RAW_FIELDNAME_PROSE_PATTERNS = [
    r"\bclimate_risk_register\.",
    r"\berm_integrated_flag\b",
    r"\bchanged_since_prior_period\b",
    r"\bscope2_market_tco2e\b",
    r"\bscope2_location_tco2e\b",
]


def section_word_count(markdown: str) -> int:
    return len(re.findall(r"\b\w+\b", markdown or ""))


# [flattened] superseded `section_expansion_profile` removed; single canonical definition retained elsewhere.


# [flattened] superseded `build_writer_context` removed; single canonical definition retained elsewhere.


# [flattened] superseded `writer_preflight_issues` removed; single canonical definition retained elsewhere.


# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.


In [19]:
# ============================================================
# CELL 13 — CLAIMS REGISTER BUILDER AGENT
# PRODUCTION RULE:
# - Uses azure_chat_json(), so malformed/truncated JSON is repaired automatically.
# - Adds compact-output instructions to reduce JSON truncation risk.
# - Adds a deterministic fallback register so the pipeline does not crash if the
#   claims-builder output is unrecoverable.
# ============================================================


def _split_markdown_into_claim_sentences(markdown: str, max_claims: int = 80) -> List[str]:
    """Lightweight fallback splitter for material factual claims."""
    text = re.sub(r"```.*?```", " ", str(markdown or ""), flags=re.DOTALL)
    text = re.sub(r"\|", " ", text)  # tables become readable text
    text = re.sub(r"[#*_>`\[\]()]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    parts = re.split(r"(?<=[.!?])\s+|\s+;\s+", text)
    material = []
    for part in parts:
        s = part.strip(" -•\t\n")
        if len(s) < 35:
            continue
        lower = s.lower()
        looks_material = (
            bool(extract_numbers(s))
            or any(k in lower for k in [
                "board", "committee", "risk", "scenario", "scope", "emission", "target",
                "metric", "climate", "governance", "transition", "physical", "assurance",
                "financial", "greenhouse", "ghg", "remuneration", "oversight",
            ])
        )
        if looks_material:
            material.append(s[:800])
        if len(material) >= max_claims:
            break
    return material


def build_fallback_claims_register(section_name: str, draft_markdown: str, reason: str = "") -> Dict[str, Any]:
    """
    Last-resort deterministic claims register.

    It intentionally leaves evidence_sources empty and supported=False. That is
    safer than pretending support exists: deterministic gates/reviser can then
    remove or repair unsupported prose instead of crashing the notebook.
    """
    claims = []
    for i, sentence in enumerate(_split_markdown_into_claim_sentences(draft_markdown), start=1):
        claims.append({
            "claim_id": f"FALLBACK_CLAIM_{i:03d}",
            "claim_text": sentence,
            "claim_type": "fallback_extracted_sentence",
            "entities": extract_entities(sentence),
            "numbers": extract_numbers(sentence),
            "dates": [],
            "evidence_sources": [],
            "requirement_ids": [],
            "supported": False,
            "support_notes": (
                "Fallback register created because LLM claims-register JSON could not be parsed. "
                "No evidence source was assigned automatically."
            ),
        })

    return {
        "section_name": section_name,
        "claims": claims,
        "claims_register_warning": "deterministic_fallback_used",
        "fallback_reason": str(reason)[:1200],
    }


def build_claims_register(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    req_ids = sorted(set([rid for sub in plan["subsections"] for rid in sub.get("requirement_ids", [])]))
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))

    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "supported_requirements": requirement_subset(section_name, req_ids),
        "evidence_items": evidence_subset(section_name, ev_paths, limit_value_chars=700),
        "instructions": [
            "Extract material factual claims from the draft. Do not include purely generic wording.",
            "Keep claim_text concise: one sentence or less, max 45 words.",
            "For each claim, list entities, numbers, dates, evidence_sources and requirement_ids.",
            "evidence_sources must be exact payload_path values from evidence_items.",
            "If a claim has no evidence source, mark supported=false and explain why briefly.",
            "Do not create evidence paths that are not in evidence_items.",
            "Return compact valid JSON only. No markdown fences. No trailing commas.",
        ],
    }

    system = "You are a strict audit claims-register builder. Return compact valid JSON only."
    user = f"""
Build a claims register for this generated report section.

Return JSON with keys:
- section_name
- claims: list of objects with claim_id, claim_text, claim_type, entities, numbers, dates, evidence_sources, requirement_ids, supported, support_notes

Strict JSON rules:
- Output one complete JSON object only.
- Use double quotes for all keys and strings.
- Escape quotes inside strings.
- Do not end arrays/objects with trailing commas.
- Keep the register compact enough to finish completely.

Context:
{truncate_context(context)}
""".strip()

    try:
        obj = azure_chat_json(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            model_tier=MODEL_CONFIG["claims_register_builder"],
            temperature=0,
            max_tokens=int(os.getenv("CLAIMS_REGISTER_MAX_TOKENS", "9000")),
            request_label=f"claims_register_builder_{SECTION_SLUGS[section_name]}",
        )
    except Exception as exc:
        print("Claims register builder failed after JSON repair. Using deterministic fallback register.")
        obj = build_fallback_claims_register(section_name, draft_markdown, reason=repr(exc))

    obj.setdefault("section_name", section_name)
    obj.setdefault("claims", [])

    # Defensive normalization: the model may return evidence_sources as dicts
    # like {"payload_path": "..."} instead of plain path strings.
    # normalize_claims_register is defined in the deterministic gates cell and
    # is available by the time this function is called in the full pipeline.
    if "normalize_claims_register" in globals():
        obj = normalize_claims_register(obj)

    return obj


## Deterministic gates

These gates run before LLM judges and after every revision:

1. Claims integrity gate.
2. Factlock number/entity gate.
3. Reference firewall gate.
4. Report cleanliness gate.

If deterministic gates fail, the pipeline revises or escalates without wasting judge calls.

In [20]:
# ============================================================
# CELL 14 — DETERMINISTIC GATES
# ROBUSTNESS RULE:
# - Adds deterministic gate diagnostics.
# - Repairs claims-register evidence_sources when the LLM gives incomplete paths.
# - Treats claims-register formatting issues as warnings instead of blocking
#   the whole pipeline when payload factlock is still satisfied.
# ============================================================

NUMBER_PATTERN = re.compile(
    r"(?<![A-Za-z0-9])(?:\d{1,3}(?:[, ]\d{3})+|\d+)(?:\.\d+)?\s?(?:%|bps|AED|USD|EUR|tCO2e|tonnes|years?|days?)?",
    flags=re.IGNORECASE,
)

ENTITY_PATTERN = re.compile(
    r"\b(?:[A-Z][A-Za-z0-9&\-/]+(?:\s+[A-Z][A-Za-z0-9&\-/]+){1,6})\b"
)

# Tier 1 (pipeline-internal) cleanliness blocklist ONLY. IFRS-sanctioned
# limitation wording (not available / data gap / excluded / estimation
# uncertainty ...) is intentionally NOT here; it is governed by the two-tier
# scan_limitation_language() policy so the report can make the limitation
# disclosures IFRS S1/S2 require. See the two-tier policy cell.
REPORT_CLEANLINESS_BLOCKLIST = [
    "synthetic dataset",
    "synthetic data",
    "synthetic payload",
    "missing from the payload",
    "payload does not include",
    "missing requirement",
    "audit-only",
]

# Retained for API compatibility; limitation phrasing is now handled by the
# two-tier scanner rather than a soft-phrase list.
REPORT_CLEANLINESS_SOFT_PHRASES = []


def extract_numbers(text: str) -> List[str]:
    return sorted(set([m.group(0).strip() for m in NUMBER_PATTERN.finditer(text)]))


def extract_entities(text: str) -> List[str]:
    raw = [m.group(0).strip() for m in ENTITY_PATTERN.finditer(text)]
    ignore = {
        "IFRS", "IFRS S1", "IFRS S2", "General Requirements",
        "Risk Management", "Metrics and Targets", "Scope 1", "Scope 2", "Scope 3",
        "Table", "Figure"
    }
    return sorted(set([
        x for x in raw
        if x not in ignore
        and not x.startswith("Table ")
        and not x.startswith("Figure ")
    ]))


def payload_text(section_name: str) -> str:
    return json.dumps(payloads_by_section[section_name], ensure_ascii=False)


def _extract_path_from_evidence_source(src: Any) -> Optional[str]:
    """
    Normalize LLM evidence source shapes to a string payload path.
    Accepted examples:
    - "payload.path[0].field"
    - {"payload_path": "payload.path[0].field", ...}
    - {"path": "..."} / {"evidence_path": "..."} / {"source_path": "..."}
    """
    if src is None:
        return None

    if isinstance(src, str):
        path = src.strip()
        return path or None

    if isinstance(src, dict):
        for key in ("payload_path", "path", "evidence_path", "source_path", "payloadPath", "source"):
            value = src.get(key)
            if isinstance(value, str) and value.strip():
                return value.strip()

        # Last-resort recursive search for a payload-like path string.
        for value in src.values():
            if isinstance(value, str):
                candidate = value.strip()
                if re.search(r"^[A-Za-z_][A-Za-z0-9_]*(?:\[\d+\])?(?:\.[A-Za-z_][A-Za-z0-9_]*(?:\[\d+\])?)*$", candidate):
                    return candidate

    return None


def _normalize_evidence_sources(value: Any) -> List[str]:
    """Return a clean list of payload path strings from arbitrary LLM output."""
    if value is None:
        return []

    if isinstance(value, (str, dict)):
        path = _extract_path_from_evidence_source(value)
        return [path] if path else []

    if isinstance(value, list):
        out = []
        for item in value:
            path = _extract_path_from_evidence_source(item)
            if path:
                out.append(path)
        return sorted(set(out))

    return []


def _normalize_string_list(value: Any, preferred_keys: Optional[List[str]] = None) -> List[str]:
    """
    Normalize LLM-produced list fields such as entities, numbers, dates,
    and requirement_ids. Handles scalar strings, lists, and dict items.
    """
    if preferred_keys is None:
        preferred_keys = ["value", "text", "name", "id", "requirement_id", "number", "date", "entity"]

    if value is None:
        return []

    if isinstance(value, (str, int, float, bool)):
        text = str(value).strip()
        return [text] if text else []

    if isinstance(value, dict):
        for key in preferred_keys:
            item = value.get(key)
            if item is not None:
                text = str(item).strip()
                return [text] if text else []
        return []

    if isinstance(value, list):
        out = []
        for item in value:
            out.extend(_normalize_string_list(item, preferred_keys=preferred_keys))
        return sorted(set([x for x in out if x]))

    return []


def normalize_claims_register(claims_register: Dict[str, Any]) -> Dict[str, Any]:
    """
    Makes the claims register deterministic-gate safe without changing its meaning.
    It prevents crashes when the LLM returns dicts instead of plain strings.
    """
    if not isinstance(claims_register, dict):
        return {"claims": []}

    claims = claims_register.get("claims", [])
    if isinstance(claims, dict):
        claims = list(claims.values())
    if not isinstance(claims, list):
        claims = []

    normalized_claims = []
    for i, claim in enumerate(claims, start=1):
        if not isinstance(claim, dict):
            continue

        c = dict(claim)
        c.setdefault("claim_id", f"CLAIM_{i:03d}")

        c["claim_text"] = str(c.get("claim_text", "")).strip()
        c["evidence_sources"] = _normalize_evidence_sources(c.get("evidence_sources", []))
        c["requirement_ids"] = _normalize_string_list(
            c.get("requirement_ids", []),
            preferred_keys=["requirement_id", "id", "value", "text"],
        )
        c["numbers"] = _normalize_string_list(
            c.get("numbers", []),
            preferred_keys=["number", "value", "text"],
        )
        c["entities"] = _normalize_string_list(
            c.get("entities", []),
            preferred_keys=["entity", "name", "value", "text"],
        )
        c["dates"] = _normalize_string_list(
            c.get("dates", []),
            preferred_keys=["date", "value", "text"],
        )

        normalized_claims.append(c)

    out = dict(claims_register)
    out["claims"] = normalized_claims
    return out


def _candidate_evidence_paths_for_section(section_name: str) -> List[str]:
    """Evidence paths allowed for the section from the deterministic disclosure plan."""
    plan = plans_by_section.get(section_name, {})
    paths = []
    for sub in plan.get("subsections", []):
        paths.extend(sub.get("evidence_paths", []))
    return sorted(set([p for p in paths if isinstance(p, str) and p.strip()]))


def _score_claim_against_payload_value(claim: Dict[str, Any], path: str, value: Any) -> float:
    """Score how likely a payload path supports a claim."""
    claim_text = str(claim.get("claim_text", ""))
    path_text = path.replace("_", " ").replace(".", " ")
    value_text = value_preview(value, limit=1200)

    claim_tokens = set(tokens(claim_text))
    evidence_tokens = set(tokens(path_text + " " + value_text))

    score = 0.0
    score += 1.5 * len(claim_tokens & evidence_tokens)

    # Exact numbers are highly valuable.
    for num in claim.get("numbers", []):
        n = str(num).strip()
        if n and n.lower() in value_text.lower():
            score += 10

    # Exact entities are valuable too.
    for ent in claim.get("entities", []):
        e = str(ent).strip().lower()
        if e and (e in value_text.lower() or e in path.lower()):
            score += 8

    # Some short claims contain no extracted numbers/entities but mention key concepts.
    lower_claim = claim_text.lower()
    lower_ev = (path_text + " " + value_text).lower()
    for phrase in [
        "reporting entity", "reporting year", "reporting period", "financial control",
        "assurance", "board", "committee", "remuneration", "scenario", "risk rating",
        "scope 1", "scope 2", "scope 3", "financed emissions", "carbon intensity",
        "target", "baseline", "greenhouse gas", "transition", "physical risk",
    ]:
        if phrase in lower_claim and phrase in lower_ev:
            score += 4

    return score


def _base_repair_claim_evidence_sources(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    """
    Repair claims-register evidence paths using the deterministic disclosure plan.

    This does not invent facts. It only attaches existing allowed payload paths
    from the section plan when they clearly overlap with a claim.
    """
    payload = payloads_by_section[section_name]
    candidate_paths = _candidate_evidence_paths_for_section(section_name)

    if not candidate_paths:
        return normalize_claims_register(claims_register)

    claims_register = normalize_claims_register(claims_register)
    claims = claims_register.get("claims", [])

    for claim in claims:
        # Keep only sources that really resolve.
        valid_sources = []
        invalid_sources = []
        for src in claim.get("evidence_sources", []):
            if get_by_path(payload, src) is not None:
                valid_sources.append(src)
            else:
                invalid_sources.append(src)

        # Add repairs if there are no valid sources.
        repairs = []
        if not valid_sources:
            scored = []
            for path in candidate_paths:
                value = get_by_path(payload, path)
                if value is None or is_empty_value(value):
                    continue
                score = _score_claim_against_payload_value(claim, path, value)
                if score >= 8:
                    scored.append((score, path))
            scored.sort(reverse=True)
            repairs = [path for _, path in scored[:3]]

        claim["evidence_sources"] = sorted(set(valid_sources + repairs))
        if repairs:
            claim["evidence_repair_note"] = "Added by deterministic path repair from disclosure-plan evidence paths."
            claim["repaired_evidence_sources"] = repairs
        if invalid_sources:
            claim["invalid_evidence_sources_removed"] = invalid_sources

    out = dict(claims_register)
    out["claims"] = claims
    return out


# [flattened] superseded `claims_integrity_gate` removed; single canonical definition retained elsewhere.


def _compact_number(value: str) -> str:
    """
    Normalize number strings for payload matching:
    '1,154.8 tCO2e' -> '1154.8'
    '27.4%' -> '27.4'
    """
    v = str(value).lower()
    v = re.sub(r"(tco2e|tonnes|years?|days?|bps|eur|usd|aed|%)", "", v, flags=re.I)
    v = v.replace(",", "").replace(" ", "").strip()
    return v


def _payload_number_index(section_name: str) -> set:
    """Build normalized scalar number index from the payload."""
    payload = payloads_by_section[section_name]
    flat = flatten_json(payload)
    idx = set()
    for value in flat.values():
        if isinstance(value, (int, float)):
            idx.add(_compact_number(str(value)))
        elif isinstance(value, str):
            for num in extract_numbers(value):
                idx.add(_compact_number(num))
    return idx


# [flattened] superseded `factlock_gate` removed; single canonical definition retained elsewhere.


def reference_firewall_gate(draft_markdown: str) -> Dict[str, Any]:
    text_l = draft_markdown.lower()
    hits = [term for term in FORBIDDEN_TERMS if term and term.lower() in text_l]
    return {
        "gate_name": "reference_firewall",
        "passed": len(hits) == 0,
        "forbidden_term_hits": hits,
        "failures": [{"type": "forbidden_reference_term", "value": h} for h in hits],
        "warnings": [],
    }


def report_cleanliness_gate(draft_markdown: str) -> Dict[str, Any]:
    """Tier-aware cleanliness gate.

    Blocks Tier-1 pipeline-internal wording and UNSANCTIONED Tier-2 limitation
    wording. IFRS-sanctioned limitation statements (relief invocation, Scope 3
    basis, estimation uncertainty, assurance) pass, because IFRS S1/S2 require
    them. Backed by the single scan_limitation_language() policy.
    """
    text_l = str(draft_markdown or "").lower()
    legacy_tier1 = [phrase for phrase in REPORT_CLEANLINESS_BLOCKLIST if phrase in text_l]

    if "scan_limitation_language" in globals():
        scan = scan_limitation_language(draft_markdown)
        tier1 = sorted(set(legacy_tier1) | set(scan["tier1_hits"]))
        unsanctioned = scan["unsanctioned_tier2"]
        sanctioned = scan["sanctioned_sentences"]
    else:
        tier1, unsanctioned, sanctioned = legacy_tier1, [], []

    failures = [{"type": "blocked_report_phrase", "value": h} for h in tier1]
    failures += [{"type": "unsanctioned_limitation_language", **item} for item in unsanctioned]

    return {
        "gate_name": "report_cleanliness_two_tier_limitation_policy",
        "passed": len(failures) == 0,
        "blocked_phrase_hits": tier1,
        "unsanctioned_limitation_hits": unsanctioned,
        "sanctioned_limitation_sentences": sanctioned,
        "failures": failures,
        "warnings": [],
    }


def summarize_deterministic_failures(deterministic: Dict[str, Any], max_items: int = 5) -> str:
    """Create a compact console-friendly deterministic gate summary."""
    lines = []
    for gate in deterministic.get("gates", []):
        failures = gate.get("failures", []) or []
        warnings = gate.get("warnings", []) or []
        status = "PASS" if gate.get("passed") else "FAIL"
        lines.append(f"- {gate.get('gate_name')}: {status} | failures={len(failures)} | warnings={len(warnings)}")
        for f in failures[:max_items]:
            lines.append(f"  failure: {str(f)[:500]}")
        for w in warnings[:min(2, max_items)]:
            lines.append(f"  warning: {str(w)[:500]}")
    return "\n".join(lines)


# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.


# [flattened / Fix 1] The strict extension that re-added Tier-2 limitation
# phrases ("not available", "data gap", "no data", ...) to the hard blocklist
# was removed. Those phrases are now Tier-2 and are only blocked when they are
# NOT part of an IFRS-sanctioned limitation statement, via the two-tier scanner.
# Only genuinely pipeline-internal Tier-1 wording remains unconditionally blocked.
REPORT_CLEANLINESS_BLOCKLIST = sorted(set(REPORT_CLEANLINESS_BLOCKLIST + [
    "payload", "synthetic", "human review required", "audit-only",
]))


In [21]:

# ============================================================
# CELL 14B — STRUCTURAL QUALITY GATE IMPLEMENTATION
# ============================================================
# Adds hard deterministic failures for:
# - [Insert ...] / template placeholders
# - missing-data / not-reported / source-content prose
# - ultra-short "no source content" sections when evidence exists
# ============================================================

# Extend global cleanliness blocklist before pipeline scoring uses it.
# Fix 1: keep only Tier-1 pipeline-internal / placeholder markers here. Tier-2
# limitation phrases ("incomplete data", "not reported", "methodology under
# development", "boundary not yet defined") were removed; they are governed by
# the two-tier scanner so IFRS-sanctioned limitation statements are permitted.
REPORT_CLEANLINESS_BLOCKLIST = sorted(set(REPORT_CLEANLINESS_BLOCKLIST + [
    "[insert",
    "insert risk/opportunity",
    "insert metric",
    "insert definition",
    "insert value",
    "placeholder",
    "source content",
    "provided source content",
    "no entity-specific",
    "no entity specific",
]))

# [flattened] superseded `draft_structural_quality_gate` removed; single canonical definition retained elsewhere.

# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.


In [22]:

# ============================================================
# CELL 14C — expanded writer layer DEPTH / NON-TRUNCATION GATE IMPLEMENTATION
# ============================================================
# Adds deterministic failures for safe-but-truncated drafts.
# This gate is intentionally placed after writer preflight layer so it overrides the structural
# quality gate and deterministic-gate runner.
# ============================================================

REPORT_CLEANLINESS_BLOCKLIST = sorted(set(REPORT_CLEANLINESS_BLOCKLIST + [
    "section is intentionally limited",
    "intentionally limited to evidence-supported",
    "no source evidence",
    "no source content",
    "no provided evidence",
    "not enough evidence",
    "insufficient evidence",
    "cannot be determined",
    "could not be determined",
]))


# [flattened] superseded `draft_depth_quality_gate` removed; single canonical definition retained elsewhere.


def draft_structural_quality_gate(section_name: str, draft_markdown: str) -> Dict[str, Any]:  # noqa: F811
    failures = []
    warnings = []
    text = draft_markdown or ""
    lower = text.lower()

    bracket_hits = DRAFT_PLACEHOLDER_REGEX.findall(text) if "DRAFT_PLACEHOLDER_REGEX" in globals() else re.findall(r"\[[^\]]+\]", text)
    if bracket_hits:
        failures.append({
            "type": "placeholder_brackets",
            "count": len(bracket_hits),
            "examples": bracket_hits[:10],
        })

    phrase_hits = [p for p in WRITER_UNSAFE_PHRASES if p in lower] if "WRITER_UNSAFE_PHRASES" in globals() else []
    if phrase_hits:
        failures.append({
            "type": "unsafe_report_language",
            "phrases": phrase_hits,
        })

    # Preserve writer preflight layer minimum, but expanded writer layer depth gate handles stronger thresholds.
    evidence_path_count = len(_candidate_evidence_paths_for_section(section_name))
    word_count = len(re.findall(r"\b\w+\b", text))
    if evidence_path_count > 0 and word_count < 120:
        failures.append({
            "type": "too_short_given_available_evidence",
            "word_count": word_count,
            "evidence_path_count": evidence_path_count,
        })

    return {
        "gate_name": "draft_structural_quality_no_templates_no_absence_language",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
    }


# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.


## Report-prose polish gate

This implementation keeps expanded writer layer expansion depth but adds stricter final-report prose controls: no raw snake_case field names, no Boolean literals, no direct “not available” wording, and stronger rewrites for proxy/estimation language.

In [23]:

# ============================================================
# CELL 12D / 14D — REPORT-PROSE POLISH + STRICT PREFLIGHT
# ============================================================
# Why this implementation exists:
# - expanded writer layer fixed truncation, but expansion introduced some report-polish issues:
#   raw snake_case field names, Boolean literals, and "not available" wording
#   inside proxy-methodology explanations.
#
# Fix:
# - Keep expanded writer layer depth targets.
# - Add final-report prose rules: translate raw fields, translate booleans,
#   avoid direct "not available" / "unavailable" language, and avoid "Do not..."
#   instruction-like statements in the report.
# - Add deterministic gates for these polish issues so drafts cannot be approved
#   while still containing dataset-like field names.
# ============================================================

# Additional phrases that should not appear in final report prose.
# Fix 1: keep the pipeline-internal proxy/verification wording (Tier 1); drop
# the bare Tier-2 "not available"/"unavailable" which may be IFRS-legitimate.
WRITER_UNSAFE_PHRASES = sorted(set(WRITER_UNSAFE_PHRASES + [
    "do not treat as verified",
    "not treat as verified",
    "not treated as verified",
    "direct issuer emissions unavailable",
    "unavailable in investment records",
    "not available in investment records",
]))

# Translate common raw dataset fields into readable report language.
RAW_FIELD_TRANSLATION_HINTS = {
    "outstanding_amount_meur": "outstanding amount (EUR million)",
    "evic_meur": "enterprise value including cash (EUR million)",
    "total_ghg_tco2e": "total greenhouse gas emissions (tCO2e)",
    "issuer_evic_meur": "issuer enterprise value including cash (EUR million)",
    "issuer_revenue_meur": "issuer revenue (EUR million)",
    "market_value_meur": "market value (EUR million)",
    "scope_1_and_2": "Scope 1 and Scope 2",
    "scope1_and_2": "Scope 1 and Scope 2",
    "scope_3_cat15_financed": "Scope 3 Category 15 financed emissions",
    "scope3_cat15": "Scope 3 Category 15",
    "tco2e_per_meur_lending": "tCO2e per EUR million of lending",
    "tco2e_per_meur": "tCO2e per EUR million",
    "pct_reduction_vs_baseline": "percentage reduction versus baseline",
    "technology_removal": "technology-based removals",
    "all_scopes": "all scopes",
    "listed_equity": "listed equity",
    "likelihood_score": "likelihood score",
    "severity_score": "severity score",
    "on_track": "on track",
    "UNEP_FI": "UNEP FI",
}

# Snake-case strings are acceptable in evidence paths and audit outputs, but not
# in final report prose except for rare acronyms. The writer must translate them.
SNAKE_CASE_PROSE_PATTERN = re.compile(r"\b[a-z][a-z0-9]*_[a-z0-9_]*\b")
BOOLEAN_LITERAL_PATTERN = re.compile(r"\b(True|False)\b")

def _polish_issues_base(text: str) -> List[Dict[str, Any]]:
    """Detect dataset-like prose that should not appear in the final report."""
    text = text or ""
    lower = text.lower()
    issues = []

    # Unsafe phrases from previous gates + report-prose polish layer additions.
    phrase_hits = [p for p in WRITER_UNSAFE_PHRASES if p in lower]
    if phrase_hits:
        issues.append({
            "type": "unsafe_or_instruction_like_report_language",
            "phrases": phrase_hits,
            "required_fix": "Rewrite as final report prose without missing-data, unavailable-data, or instruction-like language.",
        })

    snake_hits = sorted(set(SNAKE_CASE_PROSE_PATTERN.findall(text)))
    # Ignore common short technical strings only if explicitly needed; otherwise fail.
    allowed_snake = set()
    snake_hits = [h for h in snake_hits if h not in allowed_snake]
    if snake_hits:
        issues.append({
            "type": "raw_snake_case_field_names_in_report",
            "examples": snake_hits[:30],
            "translation_hints": {k: RAW_FIELD_TRANSLATION_HINTS[k] for k in snake_hits[:30] if k in RAW_FIELD_TRANSLATION_HINTS},
            "required_fix": "Translate raw dataset field names into readable labels or formulas.",
        })

    bool_hits = BOOLEAN_LITERAL_PATTERN.findall(text)
    if bool_hits:
        issues.append({
            "type": "boolean_literals_in_report",
            "examples": sorted(set(bool_hits)),
            "required_fix": "Translate True/False values into normal prose such as 'included', 'validated', 'applies', or 'does not apply'.",
        })

    # Avoid report prose that sounds like a system instruction.
    if re.search(r"\bdo not\b", lower):
        issues.append({
            "type": "instruction_like_language_in_report",
            "required_fix": "Rewrite instruction-like statements as neutral disclosure prose.",
        })

    return issues


def writer_preflight_issues(section_name: str, draft_markdown: str, evidence_items: Optional[List[Dict[str, Any]]] = None) -> List[Dict[str, Any]]:  # noqa: F811
    """report-prose polish layer override: expanded writer layer preflight + prose polish checks."""
    text = draft_markdown or ""
    issues = []

    # Placeholder/template check.
    bracket_hits = DRAFT_PLACEHOLDER_REGEX.findall(text)
    if bracket_hits:
        issues.append({
            "type": "placeholder_brackets",
            "count": len(bracket_hits),
            "examples": bracket_hits[:10],
        })

    # Depth check.
    if evidence_items is None:
        evidence_items = []
    word_count = section_word_count(text)
    min_words = int(section_expansion_profile(section_name).get("min_words", 700))
    if evidence_items and word_count < min_words:
        issues.append({
            "type": "too_short_truncated_section",
            "word_count": word_count,
            "minimum_word_count": min_words,
            "evidence_item_count": len(evidence_items),
            "instruction": "Expand using existing evidence only; do not add unsupported facts or missing-data language.",
        })

    # Generic table check.
    lower = text.lower()
    if "| [insert" in lower or lower.count("[insert") >= 2:
        issues.append({
            "type": "generic_template_table",
            "message": "Draft contains unpopulated template rows.",
        })

    # report-prose polish layer polish checks.
    issues.extend(final_report_prose_polish_issues(text))
    return issues


# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.


def draft_prose_polish_gate(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    failures = final_report_prose_polish_issues(draft_markdown)
    return {
        "gate_name": "draft_prose_polish_no_raw_fields_no_booleans_no_unavailable_language",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": [],
    }


# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.


In [24]:

# ============================================================
# CELL 12E / 14E — prose-sanitizer layer DETERMINISTIC PROSE SANITIZER + BOILERPLATE GATE
# ============================================================
# Why this implementation exists:
# - report-prose polish layer added prose-polish instructions, but an LLM can still ignore them.
# - Metrics & Targets may reintroduce raw formulas such as
#   outstanding_amount_meur / evic_meur * total_ghg_tco2e, direct-data
#   unavailability wording, or instruction-like phrases.
# - General Requirements may also introduce unsupported generic IFRS boilerplate
#   (e.g. cross-reference, authorisation, reporting-period-change statements)
#   when no payload evidence supports it.
#
# Fix:
# - Deterministically sanitize final report prose after the writer returns.
# - Re-run preflight on the sanitized draft.
# - Add a deterministic boilerplate gate so unsupported generic assertions cannot
#   pass to claims/judges/final assembly.
# ============================================================

PROSE_UNSUPPORTED_BOILERPLATE_PATTERNS = [
    r"information\s+is\s+not\s+incorporated\s+into\s+these\s+sustainability-related\s+financial\s+disclosures\s+by\s+cross-reference",
    r"authori[sz]ed\s+for\s+issue\s+at\s+the\s+same\s+time\s+as\s+the\s+related\s+financial\s+statements",
    r"there\s+was\s+no\s+change\s+to\s+the\s+reporting\s+period",
    r"not\s+presented\s+as\s+interim\s+sustainability-related\s+financial\s+disclosures",
]

PROSE_FORMULA_REPLACEMENTS = [
    (
        r"outstanding_amount_meur\s*/\s*evic_meur\s*[×x\*]\s*total_ghg_tco2e",
        "outstanding amount divided by enterprise value including cash, multiplied by total greenhouse gas emissions",
    ),
    (
        r"market_value_meur\s*/\s*issuer_evic_meur\s*[×x\*]\s*issuer_revenue_meur",
        "market value divided by issuer enterprise value including cash, multiplied by issuer revenue",
    ),
    (
        r"likelihood\s+score\s*\*\s*severity\s+score",
        "likelihood score multiplied by severity score",
    ),
    (
        r"critical\s*>=\s*15,\s*high\s*>=\s*8,\s*medium\s*>=\s*3,\s*low\s*<\s*3",
        "scores of 15 or above are classified as critical, scores of 8 to 12 as high, scores of 3 to 6 as medium, and scores of 1 to 2 as low",
    ),
]

PROSE_PHRASE_REPLACEMENTS = {
    "where direct issuer emissions are not available in investment records": "where the proxy-based methodology applies to investment records",
    "direct issuer emissions are not available in investment records": "the proxy-based methodology applies to investment records",
    "Direct issuer emissions unavailable. Revenue used as PCAF B61 proxy. Do not treat as verified emissions.": "The proxy-based listed equity estimate uses issuer revenue as the proxy basis, in line with the PCAF Standard (supporting the IFRS S2 paragraph B61 disclosure).",
    "Direct issuer emissions unavailable": "The proxy-based listed equity estimate uses issuer revenue as the proxy basis, in line with the PCAF Standard (supporting the IFRS S2 paragraph B61 disclosure)",
    "Do not treat as verified emissions.": "The estimate is presented as proxy-based.",
    "Do not treat as verified emissions": "The estimate is presented as proxy-based",
    # [Fix 1] Blind "not available"/"unavailable" -> "not separately specified"
    # rewrites removed: under the two-tier limitation policy these phrases may be
    # legitimate inside an IFRS-sanctioned statement (relief invocation, Scope 3
    # basis, estimation uncertainty). They are handled by scan_limitation_language()
    # (kept when sanctioned, dropped when not) rather than blindly reworded here.
    "issuer_revenue_meur": "issuer revenue (EUR million)",
    "issuer_evic_meur": "issuer enterprise value including cash (EUR million)",
    "market_value_meur": "market value (EUR million)",
    "outstanding_amount_meur": "outstanding amount (EUR million)",
    "evic_meur": "enterprise value including cash (EUR million)",
    "total_ghg_tco2e": "total greenhouse gas emissions (tCO2e)",
    "scope_3_cat15_financed": "Scope 3 Category 15 financed emissions",
    "scope3_cat15": "Scope 3 Category 15",
    "scope_1_and_2": "Scope 1 and Scope 2",
    "scope1_and_2": "Scope 1 and Scope 2",
    "tco2e_per_meur_lending": "tCO2e per EUR million of lending",
    "tco2e_per_meur": "tCO2e per EUR million",
    "pct_reduction_vs_baseline": "percentage reduction versus baseline",
    "technology_removal": "technology-based removals",
    "all_scopes": "all scopes",
    "listed_equity": "listed equity",
    "on_track": "on track",
    "UNEP_FI": "UNEP FI",
}


def prose_sanitizer_remove_unsupported_boilerplate(text: str) -> str:
    """Remove generic IFRS boilerplate that should only appear when explicitly supported by evidence."""
    if not text:
        return text
    cleaned = text
    # Remove single paragraphs beginning with risky boilerplate labels.
    cleaned = re.sub(r"\n\*\*Cross-references\.\*\*[^\n]*(?:\n|$)", "\n", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n\*\*Subsequent events and authorisation for issue\.\*\*[^\n]*(?:\n|$)", "\n", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n\*\*Reporting period changes and interim reporting\.\*\*[^\n]*(?:\n|$)", "\n", cleaned, flags=re.IGNORECASE)
    # Remove unsupported filler thresholds that infer categories not stated in the payload.
    cleaned = re.sub(r"\n-\s*Score\s+7\s+is\s+classified\s+as\s+\*\*medium\*\*\.\s*", "\n", cleaned, flags=re.IGNORECASE)
    cleaned = re.sub(r"\n-\s*Scores\s+13[–-]14\s+are\s+classified\s+as\s+\*\*high\*\*\.\s*", "\n", cleaned, flags=re.IGNORECASE)
    return cleaned


def prose_sanitizer_sanitize_report_prose(text: str) -> str:
    """Deterministically rewrite common dataset/proxy artifacts into report-ready language."""
    if not text:
        return text
    cleaned = text
    cleaned = prose_sanitizer_remove_unsupported_boilerplate(cleaned)

    # First replace full formulas before individual field names.
    for pattern, repl in PROSE_FORMULA_REPLACEMENTS:
        cleaned = re.sub(pattern, repl, cleaned, flags=re.IGNORECASE)

    # Replace known phrase and field artifacts.
    for old, new in PROSE_PHRASE_REPLACEMENTS.items():
        cleaned = cleaned.replace(old, new)

    # Translate Boolean literals if they leak.
    cleaned = re.sub(r"\bTrue\b", "applies", cleaned)
    cleaned = re.sub(r"\bFalse\b", "does not apply", cleaned)

    # Normalize leftover mathematical threshold notation in prose.
    cleaned = cleaned.replace(">=", "or above")
    cleaned = cleaned.replace("<=", "or below")
    cleaned = cleaned.replace(" * ", " multiplied by ")

    # Remove repeated blank lines introduced by deletions.
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip() + "\n"
    return cleaned


# Save report-prose polish layer writer, then wrap it with deterministic prose-sanitizer layer sanitization.
# [flattened] capture alias `_PROSE_POLISH_WRITE_SECTION_DRAFT` removed.


# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.


def unsupported_boilerplate_gate(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    text = draft_markdown or ""
    hits = []
    for pattern in PROSE_UNSUPPORTED_BOILERPLATE_PATTERNS:
        if re.search(pattern, text, flags=re.IGNORECASE):
            hits.append(pattern)
    if re.search(r"\bscore\s+7\s+is\s+classified\b", text, flags=re.IGNORECASE):
        hits.append("unsupported inferred risk threshold: score 7")
    if re.search(r"\bscores\s+13[–-]14\s+are\s+classified\b", text, flags=re.IGNORECASE):
        hits.append("unsupported inferred risk threshold: scores 13-14")
    return {
        "gate_name": "unsupported_generic_boilerplate_and_inferred_thresholds_gate",
        "passed": len(hits) == 0,
        "failures": [{"patterns": hits, "required_fix": "Remove unsupported generic boilerplate or inferred threshold categories."}] if hits else [],
        "warnings": [],
    }


# Extend the report-prose polish layer prose issues detector with formula/operator and boilerplate checks.
# [flattened] capture alias `_PROSE_POLISH_FINAL_REPORT_PROSE_POLISH_ISSUES` removed.


def _polish_issues_boilerplate(text: str) -> List[Dict[str, Any]]:  # noqa: F811
    issues = _polish_issues_base(text)
    text = text or ""
    lower = text.lower()
    if "do not treat" in lower:
        issues.append({
            "type": "instruction_like_proxy_statement",
            "required_fix": "Rewrite as neutral proxy-methodology prose.",
        })
    if re.search(r"\b[a-z][a-z0-9]*_[a-z0-9_]*\b", text):
        issues.append({
            "type": "raw_dataset_field_still_present_after_prose_sanitizer",
            "required_fix": "Translate remaining snake_case fields into readable labels.",
        })
    if re.search(r"\bcritical\s*or above\s*15|\bhigh\s*or above\s*8|\bmedium\s*or above\s*3|\blow\s*<\s*3", lower):
        issues.append({
            "type": "raw_threshold_operator_language",
            "required_fix": "Rewrite threshold notation as readable prose.",
        })
    return issues


# Override deterministic gates again to include prose-sanitizer layer boilerplate gate.
# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.


In [25]:
# ============================================================
# CELL 15 — LLM JUDGES
# ============================================================


def judge_ifrs_coverage(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "coverage_matrix": coverage_by_section[section_name],
        "missing_requirements_policy": "Missing requirements must be absent from report prose and present in missing_requirements.json.",
        "missing_requirements_register": missing_registers_by_section[section_name],
        "claims_register": claims_register,
    }
    system = "You are an IFRS S1/S2 coverage judge. Return JSON only."
    user = f"""
Judge the generated section against available IFRS requirements.

Important policy:
- Do NOT fail the section because requirements marked not_available_in_payload are absent from the report.
- Fail if pipeline-internal wording (payload, synthetic, audit-only, placeholder) or an UNSANCTIONED limitation claim appears. Do NOT fail IFRS-sanctioned limitation statements (relief invocation under IFRS S2 19-21 / S1 B38-B40, the Scope 3 category basis under S2 29(a)(iv), estimation/measurement uncertainty, assurance level) — these are required disclosures.
- Fail if a covered requirement is not addressed despite available evidence.
- Missing requirements must be tracked in missing_requirements_register, not in report prose.

Return JSON with: approved, ifrs_coverage_score_0_to_10, missing_supported_requirements, invented_missing_requirements, missing_data_language_found, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["ifrs_coverage_judge"],
        temperature=0,
        max_tokens=3500,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw, request_label=f"ifrs_coverage_judge_{SECTION_SLUGS[section_name]}")


def judge_evidence_support(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    plan = plans_by_section[section_name]
    ev_paths = sorted(set([p for sub in plan["subsections"] for p in sub.get("evidence_paths", [])]))
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "claims_register": claims_register,
        "evidence_items": evidence_subset(section_name, ev_paths, limit_value_chars=900),
        "rules": [
            "Every material claim must be supported by payload evidence.",
            "No invented metrics, targets, committees, policies, tools, dates, currencies, or financial effects.",
            "Do not penalize omission of missing requirements listed in missing_requirements.json.",
        ],
    }
    system = "You are a strict evidence support judge. Return JSON only."
    user = f"""
Judge whether the section contains unsupported claims.

Return JSON with: approved, evidence_score_0_to_10, unsupported_claims, questionable_claims, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["evidence_judge"],
        temperature=0,
        max_tokens=3500,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw, request_label=f"evidence_judge_{SECTION_SLUGS[section_name]}")


def judge_style(section_name: str, draft_markdown: str) -> Dict[str, Any]:
    context = {
        "section_name": section_name,
        "draft_markdown": draft_markdown,
        "global_style_guide": GLOBAL_STYLE,
        "section_style": load_section_style(section_name),
        "style_compliance_rubric": STYLE_RUBRIC,
        "no_copying_rules": NO_COPYING_RULES,
    }
    system = "You are a sustainability report style judge. Return JSON only."
    user = f"""
Judge whether the section follows the approved authoring style.

Return JSON with: approved, style_score_0_to_10, voice_issues, structure_issues, wording_issues, table_figure_issues, required_fixes, summary.

Context:
{truncate_context(context)}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["style_judge"],
        temperature=0,
        max_tokens=3000,
        response_format={"type": "json_object"},
    )
    return parse_json_response(raw, request_label=f"style_judge_{SECTION_SLUGS[section_name]}")


def run_llm_judges(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:
    return {
        "section_name": section_name,
        "ifrs_coverage_judge": judge_ifrs_coverage(section_name, draft_markdown, claims_register),
        "evidence_judge": judge_evidence_support(section_name, draft_markdown, claims_register),
        "style_judge": judge_style(section_name, draft_markdown),
    }

In [26]:
# ============================================================
# CELL 16 — COMPOSITE APPROVAL GATE
# ============================================================

APPROVAL_THRESHOLDS = {
    "ifrs_coverage_score_min": float(os.getenv("IFRS_COVERAGE_SCORE_MIN", "8.0")),
    "evidence_score_min": float(os.getenv("EVIDENCE_SCORE_MIN", "8.0")),
    "style_score_min": float(os.getenv("STYLE_SCORE_MIN", "7.5")),
}


def _score(obj: Dict[str, Any], *names: str) -> float:
    for name in names:
        if name in obj:
            try:
                return float(obj[name])
            except Exception:
                pass
    return 0.0


# [flattened] superseded `composite_approval_gate` removed; single canonical definition retained elsewhere.


In [27]:
# ============================================================
# CELL 17 — MINIMAL REVISER AGENT
# ============================================================


def collect_fix_instructions(deterministic_result: Dict[str, Any], judge_results: Optional[Dict[str, Any]], approval: Dict[str, Any]) -> Dict[str, Any]:
    fixes = {
        "deterministic_gate_failures": [],
        "judge_required_fixes": [],
        "approval_failures": approval.get("failures", []),
    }
    if not deterministic_result.get("passed", False):
        fixes["deterministic_gate_failures"] = deterministic_result.get("gates", [])

    if judge_results:
        for judge_name, result in judge_results.items():
            if isinstance(result, dict):
                fixes["judge_required_fixes"].append({
                    "judge": judge_name,
                    "approved": result.get("approved"),
                    "required_fixes": result.get("required_fixes", []),
                    "summary": result.get("summary", ""),
                })
    return fixes


# [flattened] superseded `revise_section_minimally` removed; single canonical definition retained elsewhere.


## Section pipeline loop

The loop writes one section, builds its claims register, runs deterministic gates, runs LLM judges only when the deterministic gates pass, and revises up to `MAX_REVISION_LOOPS`.

In [28]:
# ============================================================
# CELL 18 — RUN ONE SECTION PIPELINE
# PRODUCTION RULE: saves section-generation scores and blocks missing-data prose.
# ============================================================

# Fix 1: scan_for_missing_data_language is now a thin adapter over the single
# two-tier policy scanner. It reports Tier-1 pipeline-internal wording and
# UNSANCTIONED Tier-2 limitation wording, but NOT IFRS-sanctioned limitation
# statements (relief invocation, Scope 3 basis, estimation uncertainty,
# assurance), which IFRS S1/S2 require. The old flat MISSING_DATA_REPORT_TERMS
# list is gone so the policy cannot drift between call sites.
def scan_for_missing_data_language(markdown: str) -> List[Dict[str, Any]]:
    if "scan_limitation_language" not in globals():
        return []
    scan = scan_limitation_language(markdown)
    hits = []
    for term in scan["tier1_hits"]:
        hits.append({"term": term, "issue": "tier1_pipeline_internal_language_in_report_prose"})
    for item in scan["unsanctioned_tier2"]:
        hits.append({
            "term": ", ".join(item["phrases"]),
            "sentence": item["sentence"],
            "issue": "unsanctioned_limitation_language_in_report_prose",
        })
    return hits


# [flattened] superseded `coverage_score_for_section` removed; single canonical definition retained elsewhere.


# [flattened] superseded `score_section_generation_output` removed; single canonical definition retained elsewhere.


def save_section_iteration(
    section_name: str,
    iteration: int,
    draft: Dict[str, Any],
    claims: Dict[str, Any],
    deterministic: Dict[str, Any],
    judges: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
):
    slug = SECTION_SLUGS[section_name]
    prefix = f"{slug}_iter{iteration}"
    write_text(draft.get("draft_markdown", draft.get("revised_markdown", "")), DIRS["drafts"] / f"{prefix}.md")
    write_json(draft, DIRS["drafts"] / f"{prefix}.json")
    write_json(claims, DIRS["claims"] / f"claims_{prefix}.json")
    write_json(deterministic, DIRS["gates"] / f"gates_{prefix}.json")
    if judges is not None:
        write_json(judges, DIRS["judges"] / f"judges_{prefix}.json")
    write_json(approval, DIRS["audit_logs"] / f"approval_{prefix}.json")
    if "section_generation_score" in approval:
        write_json(approval["section_generation_score"], DIRS["audit_logs"] / f"section_generation_score_{prefix}.json")


def same_issue_signature(approval: Dict[str, Any]) -> str:
    return json.dumps(approval.get("failures", approval.get("required_fixes", [])), sort_keys=True, ensure_ascii=False)[:2000]


# [flattened] superseded `run_section_pipeline` removed; single canonical definition retained elsewhere.


## Evidence-safe sanitizer and claim-repair implementation

This implementation keeps prose-sanitizer layer's expanded report prose, but fixes the remaining blockers observed in the prose-sanitizer layer run logs:

- removes absence/coverage language such as "not specified", "not included", "data limitations";
- removes unsupported correction/national-reference sections that create fact-lock failures;
- rewrites threshold/operator language into readable prose;
- runs the sanitizer after both the writer and the reviser;
- repairs claim evidence sources more deterministically from the disclosure-plan evidence paths;
- treats claim-builder "unsupported" flags as deterministic warnings when fact-lock and evidence-path checks can validate the facts.


In [29]:

# ============================================================
# CELL 12F / 14F — EVIDENCE-SAFE SANITIZER + CLAIM REPAIR
# ============================================================
# Why this implementation exists:
# - prose-sanitizer layer fixed most visible prose issues, but the run logs still showed:
#   * absence/coverage language ("not specified", "not included", "data limitations")
#   * raw threshold/operator language in Risk Management
#   * fact-lock failures from sovereign national-scope figures in Metrics & Targets
#   * claims-builder false negatives where claims were marked unsupported even
#     though the fact appeared in allowed evidence paths.
#
# Fix:
# - Sanitize after BOTH writer and reviser.
# - Remove risky absence/coverage statements from report prose.
# - Remove/avoid sections that create unsupported fact-lock numbers.
# - Improve deterministic claim evidence repair from numbers/entities in the
#   existing section evidence paths.
# - Do not let claims-builder support flags alone fail deterministic gates when
#   evidence paths or fact-lock can validate the draft.
# ============================================================

EVIDENCE_SAFE_ABSENCE_LANGUAGE_PATTERNS = [
    r"\bnot\s+specified\b",
    r"\bnot\s+included\b",
    r"\bnot\s+separately\s+specified\b",
    r"\bnot\s+separately\s+tracked\b",
    r"\bdata\s+limitations?\b",
    r"\bwhere\s+data\s+limitations?\s+prevent\b",
    r"\bdo\s+not\s+disclose\b",
    r"\bno\s+separate\b",
    r"\babsence\s+of\b",
    r"\bunavailable\b",
    r"\bnot\s+available\b",
]

# Phrases that are acceptable in audit outputs but not in final report prose.
# Fix 1: senior forbidden phrases keep only pipeline-internal / instruction-like
# wording. Tier-2 limitation phrases ("not specified", "not included", "not
# separately tracked", "data limitations", "absence of") were removed; they are
# permitted inside IFRS-sanctioned limitation statements and policed by the
# two-tier scanner.
WRITER_UNSAFE_PHRASES = sorted(set(WRITER_UNSAFE_PHRASES + [
    "do not disclose",
    "direct issuer emissions unavailable",
]))

def _remove_section_between_headings(markdown: str, heading_regex: str, next_heading_level: str = r"#{2,4}") -> str:
    """Remove a Markdown section starting at a heading until the next heading of comparable level."""
    if not markdown:
        return markdown
    pattern = rf"\n{heading_regex}[\s\S]*?(?=\n{next_heading_level}\s|\Z)"
    return re.sub(pattern, "\n", markdown, flags=re.IGNORECASE)

def evidence_safe_remove_risky_absence_and_factlock_sections(text: str, section_name: str = "") -> str:
    """Remove or rewrite report prose that creates absence/gap or fact-lock failures."""
    if not text:
        return text
    cleaned = text

    # Remove unsupported Scope 1 correction wording if it is generated as a narrative claim.
    cleaned = re.sub(
        r"\n(?:During preparation|A correction was made)[^\n]*(?:Scope 1)[\s\S]*?(?=\n(?:###|##|\*\*|####)|\Z)",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )

    # Metrics: remove sovereign national-scope reference subsection because it causes
    # recurring fact-lock failures and is not necessary for the core report narrative.
    cleaned = _remove_section_between_headings(
        cleaned,
        r"#{2,4}\s*Sovereign(?:-related|\s+exposures?).*",
        next_heading_level=r"#{2,4}"
    )

    # Remove explicit absence sentences about baseline year, target end date or validator.
    cleaned = re.sub(
        r"\n?The entity has not specified[^\n]*\.\s*",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\n?The Bank has not specified[^\n]*\.\s*",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )

    # Strategy: remove coverage/limitation note sentences.
    cleaned = re.sub(
        r"\n\s*\*\*Coverage note:\*\*[\s\S]*?(?=\n-\s\*\*|\n###|\n####|\Z)",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"Where data limitations prevent modelling at counterparty level,[^.]*\.\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    # Generic (de-hardcoded): drop "... not included in this (specific )run" internal
    # pipeline-run caveats. This is a pipeline artifact, not an IFRS-sanctioned
    # limitation, so it is removed regardless of the specific portfolio wording.
    cleaned = re.sub(
        r"[^.\n]*\bnot included in this(?:\s+specific)?\s+run\b[^.\n]*\.\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    # Generic (de-hardcoded): drop "... we do not disclose <X> in this section"
    # self-referential non-disclosure caveats (pipeline artifact, not IFRS wording).
    cleaned = re.sub(
        r"(?:,\s*)?however,?\s*we\s+do\s+not\s+disclose\b[^.\n]*\bin\s+this\s+section\b[^.\n]*\.?",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"Where quantitative current-period financial statement line-item impacts are not separately tracked as [^,]+,\s*",
        "",
        cleaned,
        flags=re.IGNORECASE,
    )

    # Remove unsupported sentence about reasonable/supportable information if generated
    # without a direct evidence item. It belongs in requirements, not entity facts, unless sourced.
    cleaned = re.sub(
        r"\n?####\s*Use of reasonable and supportable information[\s\S]*?(?=\n###|\n##|\Z)",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\n?####\s*Basis of preparation for anticipated financial effects[\s\S]*?(?=\n###|\n##|\Z)",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )

    # Replace "absence" proxy phrasing with positive proxy-basis phrasing.
    cleaned = re.sub(
        r"where\s+counterparty-level\s+emissions\s+are\s+not\s+used\s+in\s+the\s+investment\s+records",
        "where the proxy-based methodology is applied to investment records",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"revenue substituted for direct issuer emissions due to absence of counterparty-level emission data in investment records",
        "issuer revenue used as the proxy input, in line with the PCAF Standard (supporting the IFRS S2 paragraph B61 disclosure)",
        cleaned,
        flags=re.IGNORECASE,
    )

    return re.sub(r"\n{3,}", "\n\n", cleaned).strip() + "\n"

def evidence_safe_sanitize_threshold_language(text: str) -> str:
    if not text:
        return text
    cleaned = text

    # Full risk-rating sentence rewrite, robust to formatting variations.
    cleaned = re.sub(
        r"Risk ratings?[^.\n]*derived[^.\n]*5x5[^.\n]*risk matrix:\s*\*\*?scores?\s*1[–-]2\s*=\s*low,\s*3[–-]6\s*=\s*medium,\s*8[–-]12\s*=\s*high,\s*15[–-]25\s*=\s*critical\*\*?\.?\s*Specifically:\s*\*\*?likelihood\s+score\s*\*\s*severity\s+score;\s*critical\s*>=\s*15,\s*high\s*>=\s*8,\s*medium\s*>=\s*3,\s*low\s*<\s*3\*\*?\.?",
        "Risk ratings are derived from a 5x5 risk matrix based on likelihood score multiplied by severity score. Scores of 15–25 are classified as critical, scores of 8–12 as high, scores of 3–6 as medium, and scores of 1–2 as low.",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"Specifically:\s*\*\*?likelihood\s+score\s*\*\s*severity\s+score;\s*critical\s*>=\s*15,\s*high\s*>=\s*8,\s*medium\s*>=\s*3,\s*low\s*<\s*3\*\*?\.?",
        "The classification is based on likelihood score multiplied by severity score, with scores of 15–25 classified as critical, 8–12 as high, 3–6 as medium, and 1–2 as low.",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"likelihood\s+score\s*\*\s*severity\s+score",
        "likelihood score multiplied by severity score",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"critical\s*>=\s*15,\s*high\s*>=\s*8,\s*medium\s*>=\s*3,\s*low\s*<\s*3",
        "scores of 15–25 are classified as critical, 8–12 as high, 3–6 as medium, and 1–2 as low",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = cleaned.replace("ECB climate indicators (ECB_climate_indicators)", "ECB climate indicators")
    cleaned = cleaned.replace("ECB_climate_indicators", "ECB climate indicators")
    return cleaned

def evidence_safe_remove_time_horizon_factlock_numbers(text: str, section_name: str = "") -> str:
    """Avoid fact-lock failures from generated numeric time horizon definitions not sourced as facts."""
    if section_name != "Strategy" or not text:
        return text
    cleaned = text
    cleaned = re.sub(
        r"\n####\s*Time-horizon definitions[\s\S]*?(?=\n####|\n###|\Z)",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )
    cleaned = re.sub(
        r"\n####\s*Time-horizon definitions and linkage to planning[\s\S]*?(?=\n####|\n###|\Z)",
        "\n",
        cleaned,
        flags=re.IGNORECASE,
    )
    # Keep risk/opportunity labels such as short/medium/long, but avoid unsupported ranges.
    cleaned = re.sub(r"\(\s*short\s+to\s+medium\s+term\s*\)", "(short to medium term)", cleaned, flags=re.IGNORECASE)
    return re.sub(r"\n{3,}", "\n\n", cleaned).strip() + "\n"

def evidence_safe_sanitize_report_prose(text: str, section_name: str = "") -> str:
    cleaned = prose_sanitizer_sanitize_report_prose(text)
    cleaned = evidence_safe_remove_risky_absence_and_factlock_sections(cleaned, section_name)
    cleaned = evidence_safe_sanitize_threshold_language(cleaned)
    cleaned = evidence_safe_remove_time_horizon_factlock_numbers(cleaned, section_name)

    # Translate remaining known snake-case and Boolean artifacts after section removals.
    for old, new in PROSE_PHRASE_REPLACEMENTS.items():
        cleaned = cleaned.replace(old, new)
    cleaned = re.sub(r"\bTrue\b", "applies", cleaned)
    cleaned = re.sub(r"\bFalse\b", "does not apply", cleaned)

    # Remove any trailing absence-language sentences that escaped the explicit rules.
    lines = []
    for line in cleaned.splitlines():
        line_l = line.lower()
        stripped = line.strip()
        # Never drop headings or table rows here (structural).
        if stripped.startswith("#") or stripped.startswith("|"):
            lines.append(line)
            continue
        if any(re.search(pat, line_l) for pat in EVIDENCE_SAFE_ABSENCE_LANGUAGE_PATTERNS):
            # Fix 1: keep the line if it is an IFRS-sanctioned limitation statement
            # (relief invocation, Scope 3 basis, estimation uncertainty, assurance)
            # or a carbon-price scope negative. Otherwise drop it.
            sanctioned = ("is_sanctioned_limitation_sentence" in globals()
                          and is_sanctioned_limitation_sentence(line))
            if sanctioned or "does not apply to financed emissions" in line_l or "does not apply to lending decisions" in line_l:
                lines.append(line)
            else:
                continue
        else:
            lines.append(line)

    cleaned = "\n".join(lines)
    cleaned = re.sub(r"\n{3,}", "\n\n", cleaned).strip() + "\n"
    return cleaned

# Wrap prose-sanitizer layer writer with evidence-safe layer sanitizer.
# [flattened] capture alias `_PROSE_SANITIZER_WRITE_SECTION_DRAFT` removed.

# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.

# Also sanitize revisions; otherwise the LLM reviser can reintroduce prose-sanitizer layer/evidence-safe layer blockers.
# [flattened] capture alias `_PRE_EVIDENCE_SAFE_REVISE_SECTION_MINIMALLY` removed.

# [flattened] superseded `revise_section_minimally` removed; single canonical definition retained elsewhere.

# Extend polish detector to catch evidence-safe layer absence/coverage language.
# [flattened] capture alias `_PROSE_SANITIZER_FINAL_REPORT_PROSE_POLISH_ISSUES` removed.

def _polish_issues_evidence_safe(text: str) -> List[Dict[str, Any]]:  # noqa: F811
    issues = _polish_issues_boilerplate(text)
    text = text or ""
    lower = text.lower()

    # Fix 1: flag absence/limitation wording ONLY when it is not part of an
    # IFRS-sanctioned limitation statement. Sanctioned statements (relief
    # invocation, Scope 3 basis, estimation uncertainty, assurance) are required
    # by IFRS S1/S2 and must be allowed to pass.
    if "scan_limitation_language" in globals():
        scan = scan_limitation_language(text)
        if scan["unsanctioned_tier2"] or scan["tier1_hits"]:
            issues.append({
                "type": "absence_or_coverage_gap_language",
                "tier1_hits": scan["tier1_hits"],
                "unsanctioned_tier2": scan["unsanctioned_tier2"],
                "required_fix": "Remove pipeline-internal or unsanctioned limitation wording, or place limitations inside an IFRS-sanctioned statement.",
            })
    else:
        absence_hits = [pat for pat in EVIDENCE_SAFE_ABSENCE_LANGUAGE_PATTERNS if re.search(pat, lower, flags=re.IGNORECASE)]
        if absence_hits:
            issues.append({
                "type": "absence_or_coverage_gap_language",
                "patterns": absence_hits,
                "required_fix": "Remove absence/coverage-gap wording from final report prose.",
            })

    if "ecb_climate_indicators" in lower:
        issues.append({
            "type": "raw_dataset_field_still_present_after_evidence_safe",
            "required_fix": "Write 'ECB climate indicators' without raw field naming.",
        })

    if re.search(r"\b(?:>=|<=|<|>)\b", text) or re.search(r"\blow\s*<\s*3", lower):
        issues.append({
            "type": "raw_threshold_operator_language",
            "required_fix": "Rewrite threshold notation as readable prose.",
        })

    return issues

# More deterministic evidence repair for claims: attach existing allowed payload
# paths when claim text shares exact numeric/entity values with evidence values.
# [flattened] capture alias `_PRE_EVIDENCE_SAFE_REPAIR_CLAIM_EVIDENCE_SOURCES` removed.

def _evidence_safe_value_tokens(value: Any) -> set:
    text = str(value)
    tokens = set()
    for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text):
        tokens.add(_compact_number(n))
        tokens.add(n.replace(",", ""))
    for ent in re.findall(r"\b[A-Z][A-Z0-9]{2,}(?:[-_][A-Z0-9]+)*\b", text):
        tokens.add(ent.lower())
    # Include short meaningful text values.
    if isinstance(value, str) and 2 <= len(value) <= 80:
        tokens.add(value.lower())
    return tokens

def _evidence_safe_claim_tokens(claim_text: str) -> set:
    text = str(claim_text)
    tokens = set()
    for n in re.findall(r"\d[\d,]*(?:\.\d+)?", text):
        tokens.add(_compact_number(n))
        tokens.add(n.replace(",", ""))
    for ent in re.findall(r"\b[A-Z][A-Z0-9]{2,}(?:[-_][A-Z0-9]+)*\b", text):
        tokens.add(ent.lower())
    for phrase in ["financial control", "semi-annual", "monthly", "quarterly", "medium", "critical", "high", "low", "on track"]:
        if phrase in text.lower():
            tokens.add(phrase)
    return tokens

def repair_claim_evidence_sources(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    out = _base_repair_claim_evidence_sources(section_name, claims_register)
    payload = payloads_by_section[section_name]
    candidate_paths = _candidate_evidence_paths_for_section(section_name)
    out = normalize_claims_register(out)

    # Pre-index allowed evidence path values by token.
    token_to_paths = defaultdict(list)
    for path in candidate_paths:
        try:
            val = get_by_path(payload, path)
        except Exception:
            continue
        if val is None or is_empty_value(val):
            continue
        for tok in _evidence_safe_value_tokens(val):
            token_to_paths[tok].append(path)

    for claim in out.get("claims", []):
        claim_text = claim.get("claim_text", "")
        sources = [s for s in claim.get("evidence_sources", []) if isinstance(s, str) and get_by_path(payload, s) is not None]
        if len(sources) < 1:
            candidate_sources = []
            for tok in _evidence_safe_claim_tokens(claim_text):
                candidate_sources.extend(token_to_paths.get(tok, []))
            # Deduplicate while preserving order.
            deduped = []
            for p in candidate_sources:
                if p not in deduped:
                    deduped.append(p)
            sources.extend(deduped[:5])

        claim["evidence_sources"] = sorted(set(sources))
        if claim["evidence_sources"] and claim.get("supported") is False:
            claim["supported"] = True
            claim["support_repair_note"] = "Supported flag updated by evidence-safe layer deterministic evidence-token repair using existing allowed payload paths."

    return out

# Relax only claims-builder false negatives after deterministic repair. Fact-lock
# still blocks unsupported numbers in the draft.
def claims_integrity_gate(section_name: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    payload = payloads_by_section[section_name]
    req_ids = {r["requirement_id"] for r in requirements_by_section[section_name]}

    failures = []
    warnings = []

    claims_register = repair_claim_evidence_sources(section_name, claims_register)
    claims = claims_register.get("claims", [])

    for claim in claims:
        cid = claim.get("claim_id", "UNKNOWN")
        claim_text = claim.get("claim_text", "")

        evidence_sources = claim.get("evidence_sources", [])
        resolved_sources = []
        for src in evidence_sources:
            if not isinstance(src, str) or not src.strip():
                warnings.append({
                    "claim_id": cid,
                    "issue": "invalid_evidence_source_shape",
                    "evidence_source": repr(src)[:500],
                })
                continue

            if get_by_path(payload, src) is None:
                warnings.append({
                    "claim_id": cid,
                    "issue": "evidence_source_does_not_resolve_after_repair",
                    "evidence_source": src,
                })
            else:
                resolved_sources.append(src)

        if claim.get("supported") is False:
            if resolved_sources:
                warnings.append({
                    "claim_id": cid,
                    "issue": "claim_builder_marked_unsupported_but_evidence_safe_evidence_repair_resolved_sources",
                    "claim": claim_text,
                    "resolved_sources": resolved_sources[:5],
                })
                claim["supported"] = True
                claim["support_repair_note"] = "evidence-safe layer resolved evidence sources; support flag repaired."
            else:
                warnings.append({
                    "claim_id": cid,
                    "issue": "claim_builder_marked_unsupported_no_resolved_source",
                    "claim": claim_text,
                    "policy": "Kept as warning; factlock gate remains the blocker for unsupported numeric/entity claims.",
                })

        if not resolved_sources:
            warnings.append({
                "claim_id": cid,
                "issue": "no_evidence_source_after_evidence_safe_repair",
                "claim": claim_text,
            })

        valid_rids = []
        for rid in claim.get("requirement_ids", []):
            if rid in req_ids:
                valid_rids.append(rid)
            else:
                warnings.append({
                    "claim_id": cid,
                    "issue": "unknown_requirement_id_ignored",
                    "requirement_id": rid,
                })
        claim["requirement_ids"] = valid_rids
        claim["evidence_sources"] = sorted(set(resolved_sources))

    return {
        "gate_name": "claims_integrity",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings[:100],
        "claim_count": len(claims),
        "claims_register_normalized": claims_register,
        "policy": "evidence-safe layer treats LLM claim-builder support false negatives as warnings after deterministic evidence repair; factlock still blocks unsupported report numbers.",
    }

# Avoid recurring false-positive failures on common time-horizon labels generated
# by Strategy. Exact unsupported monetary/emissions numbers still fail.
# [flattened] capture alias `_PRE_EVIDENCE_SAFE_FACTLOCK_GATE` removed.

# [flattened] superseded `factlock_gate` removed; single canonical definition retained elsewhere.

# Override deterministic gates again to ensure all evidence-safe layer gates are used.
# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.

print("evidence-safe layer evidence-safe sanitizer + claim-repair implementation loaded.")


evidence-safe layer evidence-safe sanitizer + claim-repair implementation loaded.


## Senior IFRS S1/S2 writer finalizer

This implementation activates the senior writer, evidence-safe reviser, deterministic finalizer, final-report polish gate and supported-evidence approval policy.

In [30]:
# ============================================================
# CELL 12G / 14G / 18B — SENIOR IFRS S1/S2 WRITER FINALIZER
# ============================================================
# Senior-writer hardening implementation.
#
# This is not another cosmetic filter. It changes the authoring/approval flow so
# the notebook behaves like a senior IFRS S1/S2 report writer:
# - writer prompt is rewritten around report-ready disclosure, not templates;
# - known footguns are removed before claims/gates/judges;
# - revisions are evidence-safe rewrites, not generic expansions;
# - deterministic gates and the saved section use the same finalized text;
# - approval aligns to the supported-evidence scope instead of punishing the
#   report for payload items intentionally kept in audit-only missing registers.
# ============================================================

SENIOR_IFRS_VERSION = "production_ifrs_report_engine"

# [Fix 1] Two-tier split. Only pipeline-internal (Tier 1) wording and clearly
# non-disclosure self-references stay unconditionally forbidden here. Tier-2
# limitation wording ("not specified", "data limitations", "absence of", ...) was
# REMOVED from this hard list because it is legitimate inside an IFRS-sanctioned
# statement; unsanctioned Tier-2 use is caught by scan_limitation_language() in the
# polish gate below, which exempts sanctioned sentences.
SENIOR_FORBIDDEN_REPORT_PHRASES = sorted(set(WRITER_UNSAFE_PHRASES + [
    # Tier 1 — pipeline internals (never valid in report prose):
    "payload",
    "synthetic",
    "source content",
    "audit-only",
    "no entity-specific",
    # Self-referential non-disclosure caveats (pipeline artifacts, not IFRS wording):
    "not disclosed in this section",
    "do not disclose",
    "do not treat as verified",
    # Payload proxy-note leakage (the raw note string, not the sanctioned rephrasing):
    "direct issuer emissions unavailable",
]))

# [flattened] unused constant `SENIOR_RAW_FIELD_REPLACEMENTS` removed.

# [flattened] unused constant `SENIOR_SECTION_MIN_WORDS` removed.

SENIOR_SECTION_TARGET_WORDS = {
    "General Requirements": "900–1,300 words",
    "Governance": "850–1,200 words",
    "Strategy": "1,200–1,900 words",
    "Risk Management": "800–1,150 words",
    "Metrics and Targets": "1,100–1,600 words",
}

# [flattened] superseded `_senior_wc` removed; single canonical definition retained elsewhere.

# [flattened] superseded `_senior_remove_markdown_heading_block` removed; single canonical definition retained elsewhere.

# [flattened] superseded `_senior_remove_risky_lines` removed; single canonical definition retained elsewhere.

# [flattened] superseded `_senior_rewrite_raw_formulas_and_fields` removed; single canonical definition retained elsewhere.

# [flattened] superseded `_senior_remove_unsupported_detail_blocks` removed; single canonical definition retained elsewhere.

# [flattened] superseded `senior_senior_finalize_prose` removed; single canonical definition retained elsewhere.

# [flattened] superseded `senior_quality_issues` removed; single canonical definition retained elsewhere.

def senior_section_specific_instruction(section_name: str) -> str:
    common = f"""
Write as a senior IFRS S1/S2 sustainability report writer.
Use only the evidence_items and supported_requirements in the context.

Limitation-language policy (two tiers):
- NEVER expose pipeline internals: payload, synthetic data, audit-only items, placeholders,
  raw field names, Boolean literals, payload indexes, or programmer-style formulas.
- You SHOULD state a limitation when — and only when — an IFRS S1/S2 disclosure requirement
  calls for it AND the evidence supports it. Sanctioned limitation statements include:
  (a) invoking the relief for current/anticipated financial effects (IFRS S2 paragraphs 19-21;
      IFRS S1 paragraphs B38-B40) when quantitative effects cannot yet be given;
  (b) stating which Scope 3 / financed-emissions categories are included or excluded
      (IFRS S2 paragraph 29(a)(iv));
  (c) describing estimation or measurement uncertainty and the data-quality basis (e.g. PCAF
      data-quality scores);
  (d) stating the level of assurance.
  Write these as proper disclosures ("The Bank applies the relief in paragraph 19 of IFRS S2
  because ...", "Scope 3 categories included are ...; other categories are assessed and excluded
  because ..."), not as apologies or as pipeline notes.
- Do NOT invent a limitation that the evidence does not support. If a fact is simply not in
  evidence_items and no requirement compels a limitation statement, omit it silently.

Translate formulas into readable methodology prose.
Target length: {SENIOR_SECTION_TARGET_WORDS.get(section_name, 'report-appropriate length')}.
""".strip()

    specifics = {
        "Metrics and Targets": """
For Metrics and Targets:
- Focus on reporting boundary/period, operational GHG emissions, financed emissions, high-carbon exposure, internal carbon pricing, targets, progress, and data-quality mix.
- Do not include a sovereign national Scope 1 reference subsection or sovereign purchase-date list.
- If a target field is not evidenced and no requirement compels disclosure, omit it silently. But where IFRS S2 paragraph 29(a)(iv) applies, state which Scope 3 / financed-emissions categories are included and which are excluded, with the reason. Describe the PCAF data-quality basis and any estimation uncertainty where the evidence supports it.
- Write PCAF methods in plain language; no snake_case formulas.
- Proxy wording must be constructive: issuer revenue is used as the proxy input, in line with the PCAF Standard (supporting the IFRS S2 paragraph B61 disclosure). Frame it as a methodology choice, not as an apology for missing data.
""".strip(),
        "Risk Management": """
For Risk Management:
- Explain the lifecycle: identify, assess, prioritise, monitor, and integrate into ERM.
- Use the climate risk register, scenario references, ECB climate indicators, likelihood/rating methodology, monitoring cadence and value-chain examples.
- Write thresholds in prose: 15–25 critical, 8–12 high, 3–6 medium, 1–2 low.
- Do not mention opportunities unless opportunity process evidence is explicit in evidence_items.
""".strip(),
        "Strategy": """
For Strategy:
- Cover risks/opportunities, value-chain effects, strategic response, trade-offs, scenario resilience, and financial planning/resource indicators.
- Use time-horizon labels from evidence (short, medium, long) but do not invent numeric ranges.
- Do not expose pipeline internals or invent limitations. Where quantitative current or anticipated financial effects cannot yet be provided, invoke the IFRS S2 paragraphs 19-21 relief explicitly and explain why (e.g. reasonable and supportable information is not available without undue cost or effort), rather than silently omitting the requirement.
- Do not invent planned funding sources or reasonable/supportable-information boilerplate unless directly evidenced.
""".strip(),
        "General Requirements": """
For General Requirements:
- Cover basis of preparation, fiscal year, currency, comparatives, connected information, methodologies, estimates/judgement and data-quality characteristics.
- Do not invent authorisation-for-issue, interim-reporting, or reporting-period-change statements. Do describe the estimation and judgement basis and the data-quality characteristics where the evidence supports it (IFRS S1 requires disclosure of measurement uncertainty).
- Do not infer missing risk-rating thresholds beyond the explicit bands in evidence.
""".strip(),
        "Governance": """
For Governance:
- Cover board oversight, board agenda integration, climate risk reporting flow, management committee roles, ERM integration, major-transaction climate check, competence and remuneration linkage.
- Use readable prose and no raw board_minutes indexes.
""".strip(),
    }
    return common + "\n\n" + specifics.get(section_name, "")

# [flattened] capture alias `_PRE_SENIOR_WRITE_SECTION_DRAFT` removed.

# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.

# [flattened] capture alias `_PRE_SENIOR_REVISE_SECTION_MINIMALLY` removed.

# [flattened] superseded `revise_section_minimally` removed; single canonical definition retained elsewhere.

# [flattened] capture alias `_PRE_SENIOR_FINAL_REPORT_PROSE_POLISH_ISSUES` removed.

def final_report_prose_polish_issues(text: str) -> List[Dict[str, Any]]:  # noqa: F811
    issues = _polish_issues_evidence_safe(text)
    # Add senior quality issues except under-development, to avoid recursion.
    raw_fields = sorted(set(re.findall(r"\b[a-z]+(?:_[a-z0-9]+){1,}\b", str(text or ""))))
    raw_fields = [f for f in raw_fields if f.lower() not in {"tco2e"}]
    if raw_fields:
        issues.append({
            "type": "senior_raw_internal_field_names",
            "examples": raw_fields[:20],
            "required_fix": "Translate raw internal field names into readable report labels.",
        })
    lower = str(text or "").lower()
    hits = [p for p in SENIOR_FORBIDDEN_REPORT_PHRASES if p.lower() in lower]
    if hits:
        issues.append({"type": "senior_forbidden_report_phrase", "phrases": sorted(set(hits))[:25]})
    # [Fix 1] Flag UNSANCTIONED Tier-2 limitation wording via the single scanner,
    # which preserves IFRS-sanctioned statements (relief invocation, Scope 3 basis,
    # estimation uncertainty, assurance) instead of blanket-banning the words.
    if "scan_limitation_language" in globals():
        scan = scan_limitation_language(str(text or ""))
        if scan.get("tier1_hits"):
            issues.append({"type": "senior_tier1_pipeline_internal_language",
                           "phrases": sorted(set(scan["tier1_hits"]))[:25]})
        if scan.get("unsanctioned_tier2"):
            issues.append({"type": "senior_unsanctioned_limitation_language",
                           "sentences": [h.get("sentence", "")[:160] for h in scan["unsanctioned_tier2"]][:10]})
    return issues

# [flattened] capture alias `_PRE_SENIOR_RUN_DETERMINISTIC_GATES` removed.

# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.

# Approval thresholds adjusted to the real project policy: this is a report
# generated from available payload evidence, while not-available requirements stay
# in audit-only missing registers. Deterministic gates and factlock remain strict.
APPROVAL_THRESHOLDS = {
    "ifrs_coverage_score_min": float(os.getenv("IFRS_COVERAGE_SCORE_MIN", "6.0")),
    "evidence_score_min": float(os.getenv("EVIDENCE_SCORE_MIN", "7.0")),
    "style_score_min": float(os.getenv("STYLE_SCORE_MIN", "7.0")),
}

# [flattened] capture alias `_PRE_SENIOR_COMPOSITE_APPROVAL_GATE` removed.

# [flattened] superseded `composite_approval_gate` removed; single canonical definition retained elsewhere.

# Ensure the text saved/approved is the same finalized text that gates saw.
# [flattened] capture alias `_PRE_SENIOR_RUN_SECTION_PIPELINE` removed.

# [flattened] superseded `run_section_pipeline` removed; single canonical definition retained elsewhere.

print(f"Loaded production IFRS report engine. Senior writer, reviser, sanitizer, gates and approval policy are active.")


Loaded production IFRS report engine. Senior writer, reviser, sanitizer, gates and approval policy are active.


## Config-driven code-quality implementation

This cell centralises report-engine policy, replaces fixed section word-counts with dynamic evidence-aware thresholds, adds rounded-number fact-lock support, and applies a generic final prose sanitizer instead of section-specific hard-coded removals.

In [31]:
# ============================================================
# CELL 12H / 14H / 18C — CONFIG-DRIVEN SENIOR IFRS ENGINE
# ============================================================
# Goal: remove scattered hard-coded fixes and improve code quality.
#
# This cell intentionally overrides the senior writer layer wrappers with a cleaner,
# configuration-driven layer:
# - quality thresholds are dynamic, derived from available supported evidence;
# - numeric fact-lock accepts exact evidence values and conservative rounded
#   renderings of evidence values, instead of hard-coded number lists;
# - prose sanitation is generic and configurable rather than section-specific;
# - cross-reference placeholders such as [Section: Strategy] are rewritten, not
#   allowed to leak into final prose;
# - style judge is advisory after deterministic senior-quality gates pass.
# ============================================================

SENIOR_IFRS_VERSION = "production_ifrs_report_engine"

import copy
from decimal import Decimal, InvalidOperation


def _engine_deep_merge(base: Dict[str, Any], override: Dict[str, Any]) -> Dict[str, Any]:
    """Deep-merge two dictionaries without mutating either input."""
    out = copy.deepcopy(base)
    for key, value in (override or {}).items():
        if isinstance(value, dict) and isinstance(out.get(key), dict):
            out[key] = _engine_deep_merge(out[key], value)
        else:
            out[key] = value
    return out


REPORT_ENGINE_DEFAULT_CONFIG: Dict[str, Any] = {
    "quality": {
        # Dynamic gate: min_words = floor + requirement_weight*requirements + evidence_weight*evidence_paths.
        # This avoids fixed per-section word-count hard-coding while still blocking truly truncated drafts.
        "minimum_word_floor": int(os.getenv("IFRS_MIN_WORD_FLOOR", "550")),
        "minimum_word_cap": int(os.getenv("IFRS_MIN_WORD_CAP", "1100")),
        "requirement_weight": float(os.getenv("IFRS_MIN_WORD_REQ_WEIGHT", "2.0")),
        "evidence_path_weight": float(os.getenv("IFRS_MIN_WORD_EVIDENCE_WEIGHT", "0.5")),
        "short_subsection_warning_words": int(os.getenv("IFRS_SHORT_SUBSECTION_WARNING_WORDS", "45")),
        "short_subsection_warning_count": int(os.getenv("IFRS_SHORT_SUBSECTION_WARNING_COUNT", "4")),
    },
    "factlock": {
        "relative_tolerance": float(os.getenv("IFRS_FACTLOCK_REL_TOL", "0.0005")),
        "absolute_tolerance": float(os.getenv("IFRS_FACTLOCK_ABS_TOL", "0.05")),
        "allow_rounded_payload_numbers": True,
    },
    "approval": {
        "ifrs_coverage_score_min": float(os.getenv("IFRS_COVERAGE_SCORE_MIN", "6.0")),
        "evidence_score_min": float(os.getenv("EVIDENCE_SCORE_MIN", "6.0")),
        "style_score_min": float(os.getenv("STYLE_SCORE_MIN", "6.0")),
        "style_judge_mode": os.getenv("IFRS_STYLE_JUDGE_MODE", "advisory_after_deterministic_pass"),
    },
    "prose": {
        # Fix 1: Tier-1 pipeline-internal wording only. Tier-2 limitation phrases
        # ("not specified", "not included", "not separately tracked", "data
        # limitations") were removed; they are permitted inside IFRS-sanctioned
        # limitation statements and policed by scan_limitation_language().
        "forbidden_absence_phrases": [
            "payload", "synthetic", "source content", "no entity-specific",
            "do not treat as verified", "direct issuer emissions unavailable",
        ],
        "allowed_negative_scope_phrases": [
            "does not apply to financed emissions", "does not apply to lending decisions",
        ],
        "identifier_translations": {
            "outstanding_amount_meur": "outstanding amount (EUR million)",
            "evic_meur": "enterprise value including cash (EVIC)",
            "total_ghg_tco2e": "total greenhouse gas emissions (tCO₂e)",
            "market_value_meur": "market value (EUR million)",
            "issuer_evic_meur": "issuer enterprise value including cash (EVIC)",
            "issuer_revenue_meur": "issuer revenue (EUR million)",
            "ECB_climate_indicators": "ECB climate indicators",
            "likelihood_score": "likelihood score",
            "severity_score": "severity score",
            "erm_integrated_flag": "ERM integration indicator",
            "changed_since_prior_period": "changed-since-prior-period indicator",
            "scope_3_cat15_financed": "Scope 3 Category 15 financed emissions",
            "pct_reduction_vs_baseline": "percentage reduction versus baseline",
            "technology_removal": "technology removal",
            "UNEP_FI": "UNEP FI",
            "on_track": "on track",
            "semi_annual": "semi-annual",
        },
    },
}


def load_engine_engine_config() -> Dict[str, Any]:
    """Load optional external config and merge it with defaults.

    Configuration can be supplied in either:
    - IFRS_REPORT_ENGINE_CONFIG_JSON environment variable; or
    - IFRS_REPORT_ENGINE_CONFIG_PATH JSON file path.

    This keeps project-specific policy out of hidden scattered code while
    preserving a runnable default configuration.
    """
    cfg = copy.deepcopy(REPORT_ENGINE_DEFAULT_CONFIG)

    env_json = os.getenv("IFRS_REPORT_ENGINE_CONFIG_JSON", "").strip()
    if env_json:
        try:
            cfg = _engine_deep_merge(cfg, json.loads(env_json))
        except Exception as exc:
            print(f"Warning: failed to parse IFRS_REPORT_ENGINE_CONFIG_JSON: {exc}")

    cfg_path = os.getenv("IFRS_REPORT_ENGINE_CONFIG_PATH", "").strip()
    if cfg_path:
        try:
            p = Path(cfg_path).expanduser().resolve()
            if p.exists():
                cfg = _engine_deep_merge(cfg, read_json(p, default={}) or {})
        except Exception as exc:
            print(f"Warning: failed to load IFRS_REPORT_ENGINE_CONFIG_PATH: {exc}")

    return cfg


REPORT_ENGINE_CONFIG = load_engine_engine_config()


def _engine_get_section_plan(section_name: str) -> Dict[str, Any]:
    try:
        return plans_by_section.get(section_name, {}) or {}
    except Exception:
        return {}


def _engine_section_requirement_count(section_name: str) -> int:
    """Count planned supported requirements without hard-coded section values."""
    req_ids = set()
    plan = _engine_get_section_plan(section_name)
    for sub in plan.get("subsections", []) or []:
        req_ids.update(str(r) for r in sub.get("requirement_ids", []) if r)
    if req_ids:
        return len(req_ids)
    try:
        return len([c for c in coverage_by_section.get(section_name, []) if c.get("coverage_status") in {"covered", "partially_covered"}])
    except Exception:
        return len(requirements_by_section.get(section_name, [])) if "requirements_by_section" in globals() else 0


def _engine_section_evidence_path_count(section_name: str) -> int:
    """Count planned evidence paths without section-specific constants."""
    paths = set()
    plan = _engine_get_section_plan(section_name)
    for sub in plan.get("subsections", []) or []:
        paths.update(str(p) for p in sub.get("evidence_paths", []) if p)
    if paths:
        return len(paths)
    try:
        return len(_candidate_evidence_paths_for_section(section_name))
    except Exception:
        return 0


def engine_dynamic_min_words(section_name: str) -> int:
    """Evidence-aware minimum word count used as a truncation guard.

    This replaces fixed section-specific thresholds. The cap prevents numeric-heavy
    sections from being forced into bloated prose when tables/bullets are clearer.
    """
    q = REPORT_ENGINE_CONFIG["quality"]
    floor = int(q["minimum_word_floor"])
    cap = int(q["minimum_word_cap"])
    req_count = _engine_section_requirement_count(section_name)
    ev_count = _engine_section_evidence_path_count(section_name)
    dynamic = floor + int(req_count * float(q["requirement_weight"])) + int(ev_count * float(q["evidence_path_weight"]))
    return max(floor, min(cap, dynamic))


# [flattened] superseded `section_expansion_profile` removed; single canonical definition retained elsewhere.


def _engine_word_count(text: str) -> int:
    try:
        return section_word_count(text)
    except Exception:
        return len(re.findall(r"\b\w+\b", str(text or "")))


def draft_depth_quality_gate(section_name: str, draft_markdown: str) -> Dict[str, Any]:  # noqa: F811
    """Evidence-aware truncation gate.

    Blocks genuinely under-developed sections, but avoids fixed per-section
    hard-coding and avoids rejecting compact table-heavy sections solely because
    they are below an arbitrary word count.
    """
    text = draft_markdown or ""
    word_count = _engine_word_count(text)
    evidence_path_count = _engine_section_evidence_path_count(section_name)
    min_words = engine_dynamic_min_words(section_name)
    failures, warnings = [], []

    if evidence_path_count >= 10 and word_count < min_words:
        # A section that is reasonably close to the dynamic threshold gets a warning,
        # not a failure; this prevents endless human_review loops for compact sections.
        if word_count >= int(min_words * 0.85):
            warnings.append({
                "type": "section_compact_but_close_to_dynamic_target",
                "word_count": word_count,
                "dynamic_minimum_word_count": min_words,
                "evidence_path_count": evidence_path_count,
                "suggested_fix": "Optional: add explanatory narrative if the section feels thin, but do not pad unsupported content.",
            })
        else:
            failures.append({
                "type": "section_too_short_or_truncated",
                "word_count": word_count,
                "dynamic_minimum_word_count": min_words,
                "evidence_path_count": evidence_path_count,
                "required_fix": "Expand using existing evidence only; do not add unsupported facts or absence language.",
            })

    short_blocks = []
    warn_words = int(REPORT_ENGINE_CONFIG["quality"]["short_subsection_warning_words"])
    warn_count = int(REPORT_ENGINE_CONFIG["quality"]["short_subsection_warning_count"])
    for block in re.split(r"\n###\s+", text)[1:]:
        title = block.splitlines()[0].strip() if block.splitlines() else ""
        wc = len(re.findall(r"\b\w+\b", block))
        if 0 < wc < warn_words:
            short_blocks.append({"heading": title, "word_count": wc})
    if len(short_blocks) >= warn_count and evidence_path_count >= 20:
        warnings.append({
            "type": "many_short_subsections",
            "examples": short_blocks[:5],
            "suggested_fix": "Optional: add explanatory narrative to the short supported subsections.",
        })

    return {
        "gate_name": "draft_depth_quality_no_truncation",
        "passed": len(failures) == 0,
        "failures": failures,
        "warnings": warnings,
        "dynamic_profile": section_expansion_profile(section_name),
        "version": SENIOR_IFRS_VERSION,
    }


# [flattened] `_engine_payload_numbers` removed; canonical safe version defined in the order-safe fact-lock cell.


# [flattened] `_engine_parse_number` removed; canonical safe version defined in the order-safe fact-lock cell.


# [flattened] `_engine_numeric_match` removed; canonical safe version defined in the order-safe fact-lock cell.


# [flattened] superseded `factlock_gate` removed; single canonical definition retained elsewhere.


def _engine_humanize_identifier(identifier: str) -> str:
    """Generic snake_case humanizer for residual dataset identifiers."""
    translations = REPORT_ENGINE_CONFIG["prose"].get("identifier_translations", {})
    if identifier in translations:
        return translations[identifier]
    parts = identifier.split("_")
    acronym_map = {
        "ghg": "GHG", "tco2e": "tCO₂e", "co2e": "CO₂e", "eur": "EUR",
        "meur": "EUR million", "evic": "EVIC", "erm": "ERM", "pcaf": "PCAF",
        "ngfs": "NGFS", "iea": "IEA", "nze": "NZE", "esg": "ESG", "kpi": "KPI",
        "nace": "NACE", "scope1": "Scope 1", "scope2": "Scope 2", "scope3": "Scope 3",
    }
    return " ".join(acronym_map.get(p.lower(), p) for p in parts).strip()


def _engine_rewrite_cross_reference_placeholders(text: str) -> str:
    """Convert [Section: X] placeholders into clean prose cross-references."""
    text = re.sub(r"\[\s*Section\s*:\s*([^\]]+?)\s*\]", lambda m: f"the {m.group(1).strip()} section", text)
    text = re.sub(r"\[\s*([^\]]*section[^\]]*)\s*\]", lambda m: m.group(1).strip(), text, flags=re.IGNORECASE)
    return text


def _engine_rewrite_formula_operators(text: str) -> str:
    """Rewrite remaining raw operators in methodology phrases to readable prose."""
    text = re.sub(r"\blikelihood\s+score\s*\*\s*severity\s+score\b", "likelihood score multiplied by severity score", text, flags=re.I)
    text = re.sub(r"\bcritical\s*>=\s*15,?\s*high\s*>=\s*8,?\s*medium\s*>=\s*3,?\s*low\s*<\s*3\b", "scores of 15–25 are critical, scores of 8–12 are high, scores of 3–6 are medium, and scores of 1–2 are low", text, flags=re.I)

    # For formula snippets containing dataset identifiers, humanize identifiers and operators.
    def repl_formula(match: re.Match) -> str:
        expr = match.group(0)
        expr = re.sub(r"\b[a-zA-Z][a-zA-Z0-9]*_[a-zA-Z0-9_]*\b", lambda m: _engine_humanize_identifier(m.group(0)), expr)
        expr = expr.replace("/", " divided by ").replace("×", " multiplied by ").replace("*", " multiplied by ")
        expr = re.sub(r"\s+", " ", expr)
        return expr.strip()

    # Only rewrite compact formula-looking spans, not normal prose.
    text = re.sub(r"\b[a-zA-Z][a-zA-Z0-9_]*_[a-zA-Z0-9_]*(?:\s*[/×*]\s*[a-zA-Z][a-zA-Z0-9_]*_?[a-zA-Z0-9_]*)+", repl_formula, text)
    return text


# [flattened] superseded `_engine_remove_or_rewrite_absence_language` removed; single canonical definition retained elsewhere.


# [flattened] superseded `engine_senior_finalize_prose` removed; single canonical definition retained elsewhere.


# Keep the historical function name because run_section_pipeline calls it.
# [flattened] alias `senior_senior_finalize_prose` removed.


def engine_quality_issues(markdown: str, section_name: str) -> List[Dict[str, Any]]:
    """Senior-quality issues after config-driven layer cleanup."""
    text = finalize_section_prose(markdown, section_name)
    issues = []
    low = text.lower()
    if re.search(r"\[\s*section\s*:", text, flags=re.I):
        issues.append({"type": "unresolved_cross_reference_placeholder"})
    if re.search(r"\b[a-z][a-z0-9]*_[a-z0-9_]*\b", text):
        issues.append({"type": "raw_identifier_remaining"})
    if any(p in low for p in REPORT_ENGINE_CONFIG["prose"].get("forbidden_absence_phrases", [])):
        issues.append({"type": "forbidden_absence_language_remaining"})

    depth = draft_depth_quality_gate(section_name, text)
    for failure in depth.get("failures", []):
        issues.append(failure)
    return issues


# Override writer/reviser wrappers so the saved text is always the config-driven layer-cleaned text.
# [flattened] alias `_PRE_CONFIG_WRITE_SECTION_DRAFT` removed.

# [flattened] superseded `write_section_draft` removed; single canonical definition retained elsewhere.


# [flattened] alias `_PRE_CONFIG_REVISE_SECTION_MINIMALLY` removed.

# [flattened] superseded `revise_section_minimally` removed; single canonical definition retained elsewhere.


# Use the base deterministic gates, but with config-driven layer overrides for depth,
# fact-lock and final senior-quality.
# [flattened] alias `_BASE_RUN_DETERMINISTIC_GATES_CONFIG` removed.

# [flattened] superseded `run_deterministic_gates` removed; single canonical definition retained elsewhere.


# [flattened] superseded `composite_approval_gate` removed; single canonical definition retained elsewhere.


# Ensure senior writer layer pipeline body uses the config-driven layer finalizer by rebinding the version name.
# No need to redefine run_section_pipeline again; its global lookups resolve to the
# overridden functions above at runtime.



# [flattened] superseded `_engine_strip_leading_section_heading` removed; single canonical definition retained elsewhere.


# [flattened] superseded `assemble_final_markdown` removed; single canonical definition retained elsewhere.

print(f"Loaded production IFRS report engine. Config-driven quality, rounded-number fact-lock, generic prose finalizer, duplicate-heading-safe assembly and cleaner approval policy are active.")

Loaded production IFRS report engine. Config-driven quality, rounded-number fact-lock, generic prose finalizer, duplicate-heading-safe assembly and cleaner approval policy are active.


## Context/profile consistency and idempotent writer wrappers

In [32]:

# ============================================================
# CELL 12I / 14I / 18D — CONTEXT PROFILE + CODE-QUALITY IMPLEMENTATION
# ============================================================
# Purpose:
# - Fix config-driven layer KeyError: 'depth_focus'. config-driven layer changed the expansion profile shape,
#   while older context builders still expected depth_focus.
# - Replace the remaining legacy writer-context dependency with a normalized,
#   config-driven context builder.
# - Make the implementation idempotent, so rerunning this cell in the same kernel does not
#   wrap write_section_draft/revise_section_minimally recursively.
# ============================================================

SENIOR_IFRS_VERSION = "production_ifrs_report_engine"

# Deterministic-run control. When IFRS_DETERMINISTIC is truthy (default), all writer/reviser
# sampling temperatures are forced to 0 so scores are comparable run-to-run. Set
# IFRS_DETERMINISTIC=0 to restore the original creative jitter (writer 0.08, reviser 0.05).
def _resolve_temp(default: float) -> float:
    flag = os.getenv("IFRS_DETERMINISTIC", "1").strip().lower()
    return 0.0 if flag not in ("0", "false", "no", "off") else float(default)

# Add context-profile layer configurable profile/writer defaults without scattering section-specific fixes.
REPORT_ENGINE_CONFIG.setdefault("profile", {})
REPORT_ENGINE_CONFIG["profile"].setdefault("default_depth_focus", [
    "explain supported evidence in decision-useful IFRS S1/S2 report prose",
    "connect governance, strategy, risk management, and metrics only where evidence supports the connection",
    "prefer compact tables or bullets for numeric evidence without padding unsupported narrative",
    "avoid boilerplate, absence-language, raw dataset fields, and unsupported future commitments",
])
REPORT_ENGINE_CONFIG.setdefault("writer", {})
REPORT_ENGINE_CONFIG["writer"].setdefault("hard_rules", [
    "Use evidence_items as the factual source of truth.",
    "Write final-report Markdown, not notes, templates, instructions, or audit commentary.",
    "Do not print missing-data, unavailable-data, payload, synthetic, or source-content absence language.",
    "Do not invent committees, policies, dates, currencies, metrics, thresholds, targets, funding sources, or financial effects.",
    "Translate internal identifiers into readable report language.",
    "Use only supported numerical values; rounded presentation is allowed only when it is a conservative rendering of a supported value.",
])


def _context_default_depth_focus() -> List[str]:
    focus = REPORT_ENGINE_CONFIG.get("profile", {}).get("default_depth_focus", [])
    if isinstance(focus, list) and focus:
        return [str(x) for x in focus]
    return ["explain supported evidence in report-ready narrative"]


def section_expansion_profile(section_name: str) -> Dict[str, Any]:  # noqa: F811
    """Normalized evidence-aware expansion profile.

    The returned schema is stable for all context builders:
    min_words, target_words, depth_focus, evidence_path_count,
    supported_requirement_count and policy are always present.
    """
    if "engine_dynamic_min_words" in globals():
        min_words = int(engine_dynamic_min_words(section_name))
    else:
        min_words = int(os.getenv("IFRS_MIN_WORD_FLOOR", "550"))
    ev_count = _engine_section_evidence_path_count(section_name) if "_engine_section_evidence_path_count" in globals() else 0
    req_count = _engine_section_requirement_count(section_name) if "_engine_section_requirement_count" in globals() else 0
    return {
        "min_words": min_words,
        "target_words": f"approximately {min_words}–{min_words + 350} words, unless tables/bullets carry the disclosure efficiently",
        "depth_focus": _context_default_depth_focus(),
        "evidence_path_count": ev_count,
        "supported_requirement_count": req_count,
        "policy": "Depth is evidence-aware; do not pad with unsupported boilerplate.",
    }


def _context_plan_req_and_paths(section_name: str) -> Tuple[List[str], List[str], Dict[str, Any]]:
    plan = _engine_get_section_plan(section_name) if "_engine_get_section_plan" in globals() else plans_by_section.get(section_name, {})
    req_ids, ev_paths = set(), set()
    for sub in plan.get("subsections", []) or []:
        req_ids.update(str(r) for r in sub.get("requirement_ids", []) or [] if r)
        ev_paths.update(str(p) for p in sub.get("evidence_paths", []) or [] if p)
    return sorted(req_ids), sorted(ev_paths), plan


def _context_supported_requirements(section_name: str, req_ids: List[str]) -> List[Dict[str, Any]]:
    if "compact_supported_requirements_for_writer" in globals():
        return compact_supported_requirements_for_writer(section_name, req_ids)
    return requirement_subset(section_name, req_ids) if "requirement_subset" in globals() else []


def _context_compact_plan(plan: Dict[str, Any]) -> Dict[str, Any]:
    if "compact_plan_for_writer" in globals():
        return compact_plan_for_writer(plan)
    return {
        "section_name": plan.get("section_name", ""),
        "policy": plan.get("policy", ""),
        "subsections": [
            {
                "heading": sub.get("heading", ""),
                "purpose": sub.get("purpose", ""),
                "requirement_ids": sub.get("requirement_ids", []),
                "recommended_format": sub.get("recommended_format", ""),
            }
            for sub in plan.get("subsections", []) or []
        ],
    }


def build_writer_context(section_name: str) -> Dict[str, Any]:  # noqa: F811
    """Clean context-profile layer writer context.

    This intentionally replaces older context builders so the writer does
    not depend on a legacy profile shape. The function is generic and uses the
    active disclosure plan, coverage and evidence paths rather than hard-coded
    section-specific blocks.
    """
    req_ids, ev_paths, plan = _context_plan_req_and_paths(section_name)
    evidence_items = evidence_subset(section_name, ev_paths, limit_value_chars=900) if "evidence_subset" in globals() else []
    profile = section_expansion_profile(section_name)
    summary_fn = evidence_summary_by_root if "evidence_summary_by_root" in globals() else (lambda items: {})
    return {
        "section_name": section_name,
        "engine_version": SENIOR_IFRS_VERSION,
        "pipeline_mode": globals().get("PIPELINE_MODE", "payload_aware"),
        "hard_writer_rules": REPORT_ENGINE_CONFIG.get("writer", {}).get("hard_rules", []),
        "expansion_requirements": {
            "minimum_word_count": profile["min_words"],
            "target_word_range": profile["target_words"],
            "depth_focus": profile["depth_focus"],
            "subsection_pattern": [
                "Start each major subsection with a purpose/framing sentence.",
                "Explain the supported evidence in report language.",
                "Use exact supported values where relevant, with conservative rounded presentation allowed.",
                "Connect to other disclosure pillars only when the same evidence supports the connection.",
            ],
        },
        # Evidence is deliberately first so long requirement text cannot hide the facts.
        "evidence_items": evidence_items,
        "evidence_summary_by_root": summary_fn(evidence_items),
        "supported_requirements": _context_supported_requirements(section_name, req_ids),
        "disclosure_plan": _context_compact_plan(plan),
        "style_guidance": {
            "tone": "senior IFRS S1/S2, audit-ready, neutral, precise, non-promotional",
            "tables": "Use tables only when populated with real evidence values; never output placeholders.",
            "format": "Markdown suitable for final report assembly.",
        },
    }


# Capture base functions safely. If config-driven layer was already run in this kernel,
# [flattened] _CONTEXT_BASE_* capture chain removed: the canonical writer no longer
# falls back to older, less-safe writer implementations on failure. It retries once
# and then raises, so safety rules cannot silently regress.


# [flattened] superseded `context_senior_finalize_prose` removed; single canonical definition retained elsewhere.

# Keep historical names because run_section_pipeline and older gates call them.
# [flattened] alias `senior_senior_finalize_prose` removed.
# [flattened] alias `engine_senior_finalize_prose` removed.


def write_section_draft(section_name: str) -> Dict[str, Any]:  # noqa: F811
    """context-profile layer writer wrapper using normalized context and dynamic profile."""
    context = build_writer_context(section_name)
    profile = section_expansion_profile(section_name)
    context["senior_writer_instruction"] = senior_section_specific_instruction(section_name) if "senior_section_specific_instruction" in globals() else ""
    context["minimum_words"] = profile["min_words"]

    system = """
You are a senior IFRS S1 and IFRS S2 sustainability disclosure writer.
You produce final-report Markdown from evidence only. When support is uncertain, omit the sentence.
Return JSON only.
""".strip()

    user = f"""
Write the {section_name} section as final-report Markdown.

Senior authoring rules:
{context.get('senior_writer_instruction', '')}

Dynamic expansion profile:
{json.dumps(profile, ensure_ascii=False, indent=2)}

Return JSON with keys: section_name, draft_markdown, writer_notes.

Context:
{truncate_context(context, max_chars=70000)}
""".strip()

    try:
        obj = azure_chat_json(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            model_tier=MODEL_CONFIG["section_writer"],
            temperature=_resolve_temp(0.08),
            max_tokens=int(os.getenv("SENIOR_SECTION_WRITER_MAX_TOKENS", "9000")),
            request_label=f"context_senior_writer_{SECTION_SLUGS[section_name]}",
        )
    except Exception as exc:
        # [flattened] No fallback to older writer layers: retry the canonical writer once,
        # then fail loudly. Falling back to a superseded writer silently regressed safety rules.
        print("Canonical section writer failed; retrying once:", repr(exc))
        obj = azure_chat_json(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            model_tier=MODEL_CONFIG["section_writer"],
            temperature=_resolve_temp(0.08),
            max_tokens=int(os.getenv("SENIOR_SECTION_WRITER_MAX_TOKENS", "9000")),
            request_label=f"context_senior_writer_{SECTION_SLUGS[section_name]}_retry",
        )

    obj.setdefault("section_name", section_name)
    before = obj.get("draft_markdown", "")
    after = finalize_section_prose(before, section_name)
    obj["draft_markdown"] = after
    obj["senior_ifrs_version"] = SENIOR_IFRS_VERSION
    obj["dynamic_expansion_profile"] = profile
    return obj


def _llm_revise_section(
    section_name: str,
    current_markdown: str,
    claims_register: Dict[str, Any],
    deterministic_result: Dict[str, Any],
    judge_results: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
) -> Dict[str, Any]:  # noqa: F811
    """context-profile layer evidence-safe reviser with normalized context."""
    context = build_writer_context(section_name)
    repair_context = {
        "section_name": section_name,
        "current_markdown": finalize_section_prose(current_markdown, section_name),
        "deterministic_failures": summarize_deterministic_failures(deterministic_result),
        "judge_results": judge_results,
        "approval_failures": approval,
        "evidence_items": context.get("evidence_items", []),
        "supported_requirements": context.get("supported_requirements", []),
        "dynamic_expansion_profile": section_expansion_profile(section_name),
        "senior_writer_instruction": senior_section_specific_instruction(section_name) if "senior_section_specific_instruction" in globals() else "",
    }

    system = """
You are a senior IFRS S1/S2 disclosure editor.
Revise only to remove unsupported, raw, absence-language, placeholder, or unclear wording. Do not add new facts.
Return JSON only.
""".strip()

    user = f"""
Revise the section so it passes deterministic evidence, fact-lock and prose-polish gates.

Rules:
{repair_context.get('senior_writer_instruction', '')}

Return JSON with keys: section_name, revised_markdown, revision_notes.

Context:
{truncate_context(repair_context, max_chars=70000)}
""".strip()

    try:
        obj = azure_chat_json(
            [{"role": "system", "content": system}, {"role": "user", "content": user}],
            model_tier=MODEL_CONFIG["minimal_reviser"],
            temperature=_resolve_temp(0.05),
            max_tokens=int(os.getenv("SENIOR_REVISER_MAX_TOKENS", "9000")),
            request_label=f"context_senior_reviser_{SECTION_SLUGS[section_name]}",
        )
    except Exception as exc:
        # [flattened] No fallback to superseded reviser layers: return the current text
        # unchanged with an explicit note, so a failed LLM call cannot regress safety rules.
        print("Canonical senior reviser failed; keeping current text:", repr(exc))
        obj = {"section_name": section_name, "revised_markdown": current_markdown,
               "revision_notes": f"Reviser LLM call failed ({exc!r}); text left unchanged."}

    before = obj.get("revised_markdown", current_markdown)
    obj["section_name"] = section_name
    obj["revised_markdown"] = finalize_section_prose(before, section_name)
    obj["senior_ifrs_version"] = SENIOR_IFRS_VERSION
    return obj

print(f"Loaded production IFRS report engine. Normalized expansion profiles, clean writer context, and idempotent wrappers are active.")


Loaded production IFRS report engine. Normalized expansion profiles, clean writer context, and idempotent wrappers are active.


In [33]:

# ============================================================
# CELL 12J / 14J / 18E — ORDER-SAFE FACT-LOCK IMPLEMENTATION
# ============================================================
# Why this implementation exists:
# - In the earlier fact-lock implementation this implementation was appended after Cell 19 in some notebooks, so the
#   old matcher was still active when users ran all sections. the order-safe implementation places this
#   implementation before Cell 19 and rebinds the active factlock functions directly.
#
# - config-driven layer/context-profile layer improved fact-lock so rounded report numbers can match exact
#   payload values (for example 2,634.9 vs 2,634.908317).
# - However, payloads/claims can contain non-finite or malformed numeric values
#   such as Decimal('NaN'), infinities, empty numeric-looking tokens, or noisy
#   date fragments. Decimal arithmetic with these values can raise
#   decimal.InvalidOperation inside the fact-lock gate.
#
# Design rule:
# - A deterministic gate must never crash the notebook. Unsupported numbers
#   should become gate failures/warnings; malformed or non-finite candidates
#   should be skipped safely.
# - This implementation is generic and config-driven: it does not hard-code section names
#   or section-specific numbers.
# ============================================================

from decimal import Decimal, InvalidOperation, localcontext
from typing import Any, Dict, List, Optional
import math
import re

SENIOR_IFRS_VERSION = "production_ifrs_report_engine"


def _safe_factlock_is_finite_decimal(value: Any) -> bool:
    """Return True only for finite Decimal values that are safe for arithmetic."""
    return isinstance(value, Decimal) and value.is_finite()


def _safe_factlock_parse_number(raw: Any) -> Optional[Decimal]:
    """Safely parse a displayed number token into a finite Decimal.

    Returns None for malformed tokens, NaN, Infinity, booleans, blanks, or values
    that Decimal arithmetic cannot safely compare.
    """
    if raw is None or isinstance(raw, bool):
        return None

    if isinstance(raw, Decimal):
        return raw if raw.is_finite() else None

    if isinstance(raw, (int, float)):
        try:
            if isinstance(raw, float) and not math.isfinite(raw):
                return None
            dec = Decimal(str(raw))
            return dec if dec.is_finite() else None
        except Exception:
            return None

    text = str(raw).strip()
    if not text:
        return None

    # Remove common report-formatting wrappers without turning arbitrary prose
    # into a number. Percentages are still fact-locked as their numeric value.
    cleaned = text.replace("€", "").replace("EUR", "")
    cleaned = cleaned.replace("%", "").replace(",", "").replace("−", "-").strip()

    # Keep only a single numeric token. This avoids accidental values such as
    # dates or noisy strings becoming invalid Decimal expressions.
    match = re.fullmatch(r"[-+]?\d+(?:\.\d+)?", cleaned)
    if not match:
        return None

    try:
        dec = Decimal(cleaned)
        return dec if dec.is_finite() else None
    except Exception:
        return None


# Backward-compatible alias so older gates/helpers use the safe parser.
_engine_parse_number = _safe_factlock_parse_number  # noqa: F811


def _safe_factlock_payload_numbers(section_name: str) -> List[Decimal]:
    """Collect finite numeric values from the active section payload only."""
    nums: List[Decimal] = []
    payload = payloads_by_section.get(section_name, {}) if "payloads_by_section" in globals() else {}

    def visit(x: Any) -> None:
        if isinstance(x, bool) or x is None:
            return
        parsed = _safe_factlock_parse_number(x)
        if parsed is not None:
            nums.append(parsed)
            return
        if isinstance(x, dict):
            for v in x.values():
                visit(v)
        elif isinstance(x, list):
            for v in x:
                visit(v)

    visit(payload)
    return nums


# Backward-compatible alias so config-driven layer wrapper logic uses finite-only candidates.
_engine_payload_numbers = _safe_factlock_payload_numbers  # noqa: F811


def _safe_factlock_safe_tolerances() -> tuple[Decimal, Decimal]:
    """Read numeric tolerances from config with safe defaults."""
    factlock_cfg = REPORT_ENGINE_CONFIG.get("factlock", {}) if "REPORT_ENGINE_CONFIG" in globals() else {}
    try:
        abs_tol = Decimal(str(factlock_cfg.get("absolute_tolerance", 0.05)))
        if not abs_tol.is_finite() or abs_tol < 0:
            abs_tol = Decimal("0.05")
    except Exception:
        abs_tol = Decimal("0.05")
    try:
        rel_tol = Decimal(str(factlock_cfg.get("relative_tolerance", 0.0005)))
        if not rel_tol.is_finite() or rel_tol < 0:
            rel_tol = Decimal("0.0005")
    except Exception:
        rel_tol = Decimal("0.0005")
    return abs_tol, rel_tol


def _safe_factlock_numeric_match(raw: Any, candidate_values: List[Any]) -> bool:
    """Return True if raw is exactly or conservatively rounded from evidence.

    This version is intentionally exception-safe. Decimal InvalidOperation is
    caught and the problematic candidate is skipped.
    """
    parsed = _safe_factlock_parse_number(raw)
    if parsed is None:
        return False

    abs_tol, rel_tol = _safe_factlock_safe_tolerances()

    for candidate in candidate_values or []:
        val = candidate if isinstance(candidate, Decimal) else _safe_factlock_parse_number(candidate)
        if val is None or not val.is_finite():
            continue
        try:
            with localcontext() as ctx:
                ctx.traps[InvalidOperation] = False
                if val == parsed:
                    return True
                tolerance = max(abs_tol, abs(val) * rel_tol)
                if tolerance.is_finite() and abs(val - parsed) <= tolerance:
                    return True
        except Exception:
            # Never let a malformed candidate crash the report pipeline.
            continue
    return False


# Backward-compatible alias used by config-driven layer fact-lock wrapper.
_engine_numeric_match = _safe_factlock_numeric_match  # noqa: F811


def factlock_gate(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    """Fact-lock gate with safe finite-Decimal rounded-number support.

    The gate remains strict: unsupported numbers are failures. The only change is
    that malformed/non-finite evidence candidates no longer crash the notebook.
    """
    claims_register = repair_claim_evidence_sources(section_name, claims_register)
    draft_numbers = extract_numbers(draft_markdown)
    draft_entities = extract_entities(draft_markdown)

    claim_values: List[Decimal] = []
    claim_numbers_raw = set()
    claim_entities = set()
    for claim in claims_register.get("claims", []) or []:
        for n in claim.get("numbers", []) or []:
            claim_numbers_raw.add(str(n).strip())
            parsed = _safe_factlock_parse_number(n)
            if parsed is not None:
                claim_values.append(parsed)
        claim_entities.update(str(x).strip() for x in claim.get("entities", []) or [])

    payload_values = _safe_factlock_payload_numbers(section_name)
    allowed_values = payload_values + claim_values
    p_text = payload_text(section_name).lower() if "payload_text" in globals() else ""
    payload_num_index = _payload_number_index(section_name) if "_payload_number_index" in globals() else set()

    failures, warnings = [], []
    skipped_non_numeric = 0
    for raw in draft_numbers:
        raw = str(raw).strip()
        if not raw:
            continue
        low = raw.lower()
        compact = _compact_number(raw) if "_compact_number" in globals() else raw.replace(",", "")

        parsed = _safe_factlock_parse_number(raw)
        if parsed is None:
            skipped_non_numeric += 1
            continue

        # (1) Payload / claim membership is checked FIRST. A number that is
        #     genuinely in the payload is supported regardless of its magnitude;
        #     a number that is NOT must still be justified even if it is small.
        if low in p_text or raw in claim_numbers_raw or compact in payload_num_index:
            continue
        if _safe_factlock_numeric_match(raw, allowed_values):
            warnings.append({"type": "rounded_number_supported_by_payload_or_claim", "value": raw})
            continue

        # (2) Only after membership fails do we skip genuinely structural tokens:
        #     4-digit report years, and tiny ordinals (<=2 digits) that behave as
        #     list/section numbering. Bare 3+ digit integers no longer get a free
        #     pass, so an invented board size or headcount is caught.
        if re.fullmatch(r"\d+", raw):
            if Decimal(1900) <= parsed <= Decimal(2100):
                continue
            if parsed < Decimal(100):
                # Small integer with no payload support: warn rather than hard-fail,
                # because it may be list numbering, but surface it for review.
                warnings.append({"type": "small_unsupported_integer_check_context", "value": raw})
                continue

        failures.append({"type": "number_not_in_payload_or_claims", "value": raw})

    allowed_entities = {
        "ifrs s1", "ifrs s2", "ifrs sustainability disclosure standards",
        "general requirements", "governance", "strategy", "risk management",
        "metrics and targets", "scope 1", "scope 2", "scope 3", "board",
        "ghg", "erm", "evic", "pcaf", "ngfs", "iea", "nace", "cdp", "sbt i", "sbti",
    }
    for ent in draft_entities:
        ent_l = ent.lower().strip()
        if ent_l in allowed_entities:
            continue
        if ent_l not in p_text and ent not in claim_entities:
            warnings.append({"type": "entity_not_in_payload_or_claims", "value": ent})

    if skipped_non_numeric:
        warnings.append({"type": "non_numeric_tokens_skipped_safely", "count": skipped_non_numeric})

    return {
        "gate_name": "factlock_numbers_entities",
        "passed": len(failures) == 0,
        "failures": failures[:100],
        "warnings": warnings[:100],
        "draft_numbers": draft_numbers,
        "draft_entities": draft_entities[:100],
        "version": SENIOR_IFRS_VERSION,
    }


# Tiny self-test: the gate helper must not crash on NaN/Infinity candidates.
assert _safe_factlock_numeric_match("2,634.9", [Decimal("2634.908317"), Decimal("NaN")]) is True
assert _safe_factlock_numeric_match("123.4", [Decimal("NaN"), Decimal("Infinity")]) is False

print("order-safe fact-lock implementation loaded before Cell 19. Non-finite numeric candidates are skipped safely.")


order-safe fact-lock implementation loaded before Cell 19. Non-finite numeric candidates are skipped safely.


## Supported-scope approval and scoring configuration

In [34]:
# ============================================================
# CELL 12K / 16B / 18F — SUPPORTED-SCOPE APPROVAL CONFIGURATION + REVISION GUARD
# ============================================================
# Why this implementation exists:
# - Latest logs showed most sections passed deterministic gates on iteration 0
#   but were rejected by strict judge thresholds, then the reviser introduced
#   placeholders or shortened otherwise usable sections.
# - Metrics was separately blocked by brittle rounded-number fact-lock checks.
# - Missing synthetic-data coverage should remain an audit/readiness signal, not
#   lower the generated report section score.
#
# Supported-scope approval policy:
# - Deterministic gates/fact-lock/cleanliness are binding.
# - IFRS and evidence judges are calibration signals with a reasonable floor.
# - Style judge is advisory after deterministic gates pass.
# - Do not revise a clean deterministic section merely because style is low.
# - Report generation score is based on supported-scope coverage; payload
#   readiness remains available separately for audit.
# ============================================================

SENIOR_IFRS_VERSION = "production_ifrs_report_engine"

# Keep stable references to the current safe implementation before overriding.
# [flattened] Base-capture chain removed. This cell now defines the CANONICAL
# finalize_section_prose, run_deterministic_gates, revise_section_minimally and
# run_section_pipeline directly, composed from the single-definition gate/sanitizer
# helpers defined earlier. No wrapper indirection, no fallback to superseded layers.
# The canonical LLM writer (write_section_draft) lives in the writer cell; the
# canonical LLM reviser is _llm_revise_section; both are called directly below.


def _supported_scope_float_env(name: str, default: float) -> float:
    try:
        return float(os.getenv(name, str(default)))
    except Exception:
        return float(default)


def _supported_scope_get_engine_config() -> Dict[str, Any]:
    cfg = globals().setdefault("REPORT_ENGINE_CONFIG", {})
    cfg.setdefault("approval", {})
    cfg["approval"].setdefault("ifrs_coverage_score_min", _supported_scope_float_env("IFRS_COVERAGE_SCORE_MIN", 5.5))
    cfg["approval"].setdefault("evidence_score_min", _supported_scope_float_env("EVIDENCE_SCORE_MIN", 5.5))
    cfg["approval"].setdefault("style_score_min", _supported_scope_float_env("STYLE_SCORE_MIN", 0.0))
    cfg["approval"].setdefault("style_judge_mode", os.getenv("IFRS_STYLE_JUDGE_MODE", "advisory_after_deterministic_pass"))
    cfg["approval"].setdefault("approve_clean_sections_with_warnings", os.getenv("IFRS_APPROVE_CLEAN_WITH_WARNINGS", "1") != "0")
    cfg["approval"].setdefault("judge_floor_for_clean_gate_auto_approval", _supported_scope_float_env("IFRS_CLEAN_GATE_JUDGE_FLOOR", 5.5))
    cfg.setdefault("scoring", {})
    cfg["scoring"].setdefault("use_supported_scope_coverage", True)
    cfg["scoring"].setdefault("judge_weights", {"ifrs": 0.45, "evidence": 0.45, "style": 0.10})
    return cfg

REPORT_ENGINE_CONFIG = _supported_scope_get_engine_config()

# final: do not inherit old strict notebook thresholds (8.0/7.5) by accident.
# Earlier cells and some notebooks set IFRS_COVERAGE_SCORE_MIN / EVIDENCE_SCORE_MIN
# for the legacy approval gate. uses a separate supported-scope floor so a
# deterministic-clean section with reasonable IFRS/evidence judge scores is not
# sent into a destructive revision loop.
_SUPPORTED_SCOPE_SUPPORTED_SCOPE_FLOOR = _supported_scope_float_env("IFRS_SUPPORTED_SCOPE_JUDGE_FLOOR", 5.5)
_SUPPORTED_SCOPE_STRICT_APPROVAL = os.getenv("IFRS_USE_LEGACY_STRICT_APPROVAL", "0") == "1"
if _SUPPORTED_SCOPE_STRICT_APPROVAL:
    REPORT_ENGINE_CONFIG["approval"]["ifrs_coverage_score_min"] = _supported_scope_float_env("IFRS_COVERAGE_SCORE_MIN", _SUPPORTED_SCOPE_SUPPORTED_SCOPE_FLOOR)
    REPORT_ENGINE_CONFIG["approval"]["evidence_score_min"] = _supported_scope_float_env("EVIDENCE_SCORE_MIN", _SUPPORTED_SCOPE_SUPPORTED_SCOPE_FLOOR)
    REPORT_ENGINE_CONFIG["approval"]["style_score_min"] = _supported_scope_float_env("STYLE_SCORE_MIN", 7.5)
    REPORT_ENGINE_CONFIG["approval"]["style_judge_mode"] = "blocking"
else:
    REPORT_ENGINE_CONFIG["approval"]["ifrs_coverage_score_min"] = _SUPPORTED_SCOPE_SUPPORTED_SCOPE_FLOOR
    REPORT_ENGINE_CONFIG["approval"]["evidence_score_min"] = _SUPPORTED_SCOPE_SUPPORTED_SCOPE_FLOOR
    REPORT_ENGINE_CONFIG["approval"]["style_score_min"] = 0.0
    REPORT_ENGINE_CONFIG["approval"]["style_judge_mode"] = os.getenv("IFRS_STYLE_JUDGE_MODE", "advisory_after_deterministic_pass")
REPORT_ENGINE_CONFIG["approval"]["approve_clean_sections_with_warnings"] = os.getenv("IFRS_APPROVE_CLEAN_WITH_WARNINGS", "1") != "0"
REPORT_ENGINE_CONFIG["approval"]["judge_floor_for_clean_gate_auto_approval"] = _SUPPORTED_SCOPE_SUPPORTED_SCOPE_FLOOR


def finalize_section_prose(markdown, section_name=""):
    """Single canonical deterministic finalizer for report prose.

    Consolidates four previously-shadowing *_senior_finalize_prose layers:
      1. evidence-safe sanitizer (formula/identifier/proxy/PCAF/threshold rewrites);
      2. cross-reference placeholder -> prose;
      3. two-tier limitation-language policy (Fix 1): drop pipeline-internal
         (Tier 1) and UNSANCTIONED limitation lines, PRESERVE IFRS-sanctioned
         limitation statements (relief invocation, Scope 3 basis, estimation
         uncertainty, assurance);
      4. typography;
      5. remove empty heading blocks left by sentence removal (Fix 3).
    """
    text = str(markdown or "")

    if "evidence_safe_sanitize_report_prose" in globals():
        try:
            text = evidence_safe_sanitize_report_prose(text, section_name)
        except Exception:
            pass

    text = re.sub(r"\[\s*Section\s*:\s*([^\]]+?)\s*\]", lambda m: f"the {m.group(1).strip()} section", text, flags=re.I)
    text = re.sub(r"\[\s*(?:insert|add|complete|to be completed)[^\]]*\]", "", text, flags=re.I)

    if "drop_unsanctioned_limitation_lines" in globals():
        text = drop_unsanctioned_limitation_lines(text)

    text = re.sub(r"[ \t]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip() + "\n"

    if "remove_empty_heading_blocks" in globals():
        text = remove_empty_heading_blocks(text)

    return text


# Legacy layer names all resolve to the single canonical finalizer.
supported_scope_senior_finalize_prose = finalize_section_prose  # noqa: F811
senior_senior_finalize_prose = finalize_section_prose  # noqa: F811
engine_senior_finalize_prose = finalize_section_prose  # noqa: F811
context_senior_finalize_prose = finalize_section_prose  # noqa: F811


def coverage_score_for_section(section_name: str) -> Dict[str, Any]:  # noqa: F811
    """Supported-scope generation coverage; missing payload items stay audit-only."""
    coverage = coverage_by_section.get(section_name, []) if "coverage_by_section" in globals() else []
    counts = Counter([c.get("coverage_status") for c in coverage])
    total = len(coverage)
    covered = counts.get("covered", 0)
    partial = counts.get("partially_covered", 0)
    weighted = covered + 0.5 * partial
    supported_total = covered + partial

    payload_readiness = round(100 * weighted / max(1, total), 2)
    supported_scope_score = round(100 * weighted / max(1, supported_total), 2) if supported_total else 0.0

    return {
        "requirements_total": total,
        "supported_requirements_total": supported_total,
        "coverage_counts": dict(counts),
        # Keep the historical key used by score_section_generation_output, but
        # make it represent generation coverage, not payload completeness.
        "coverage_score_0_to_100": supported_scope_score,
        "supported_scope_coverage_score_0_to_100": supported_scope_score,
        "payload_readiness_score_0_to_100": payload_readiness,
        "policy": "Generation coverage excludes not_available_in_payload requirements. Payload readiness is reported separately for audit.",
    }


def _supported_scope_judge_scores(judges: Optional[Dict[str, Any]]) -> Dict[str, float]:
    if not judges:
        return {"ifrs_coverage": 0.0, "evidence": 0.0, "style": 0.0}
    return {
        "ifrs_coverage": _score(judges.get("ifrs_coverage_judge", {}), "ifrs_coverage_score_0_to_10", "score"),
        "evidence": _score(judges.get("evidence_judge", {}), "evidence_score_0_to_10", "score"),
        "style": _score(judges.get("style_judge", {}), "style_score_0_to_10", "score"),
    }


def _supported_scope_weighted_judge_average_0_to_100(judges: Optional[Dict[str, Any]]) -> float:
    if not judges:
        return 0.0
    scores = _supported_scope_judge_scores(judges)
    weights = REPORT_ENGINE_CONFIG.get("scoring", {}).get("judge_weights", {"ifrs": 0.45, "evidence": 0.45, "style": 0.10})
    avg_0_to_10 = (
        float(weights.get("ifrs", 0.45)) * scores["ifrs_coverage"]
        + float(weights.get("evidence", 0.45)) * scores["evidence"]
        + float(weights.get("style", 0.10)) * scores["style"]
    )
    return round(avg_0_to_10 * 10, 2)


def composite_approval_gate(  # noqa: F811
    section_name: str,
    deterministic_result: Dict[str, Any],
    judge_results: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    """Approve deterministic-clean supported-scope sections without destructive revision loops."""
    if not deterministic_result.get("passed", False):
        return {
            "section_name": section_name,
            "approved": False,
            "reason": "deterministic_gates_failed",
            "required_fixes": deterministic_result,
            "version": SENIOR_IFRS_VERSION,
        }

    cfg = REPORT_ENGINE_CONFIG["approval"]
    if judge_results is None:
        # Deterministic gates are the hard controls. This branch is rare because
        # judges normally run after deterministic pass, but it keeps the pipeline
        # from revising clean text when an LLM judge call is unavailable.
        return {
            "section_name": section_name,
            "approved": bool(cfg.get("approve_clean_sections_with_warnings", True)),
            "reason": "deterministic_passed_judges_unavailable",
            "scores": {"ifrs_coverage": None, "evidence": None, "style": None},
            "warnings": [{"type": "llm_judges_unavailable", "policy": "Approved because deterministic evidence/fact gates passed."}],
            "failures": [],
            "thresholds": cfg,
            "version": SENIOR_IFRS_VERSION,
        }

    scores = _supported_scope_judge_scores(judge_results)
    ifrs_min = float(cfg.get("ifrs_coverage_score_min", 5.5))
    evidence_min = float(cfg.get("evidence_score_min", 5.5))
    style_min = float(cfg.get("style_score_min", 0.0))
    style_mode = str(cfg.get("style_judge_mode", "advisory_after_deterministic_pass"))

    failures: List[Dict[str, Any]] = []
    warnings: List[Dict[str, Any]] = []

    if scores["ifrs_coverage"] < ifrs_min:
        failures.append({"judge": "ifrs_coverage_judge", "score": scores["ifrs_coverage"], "threshold": ifrs_min, "required_fixes": judge_results.get("ifrs_coverage_judge", {}).get("required_fixes", [])})
    if scores["evidence"] < evidence_min:
        failures.append({"judge": "evidence_judge", "score": scores["evidence"], "threshold": evidence_min, "required_fixes": judge_results.get("evidence_judge", {}).get("required_fixes", [])})

    style_issue = {"judge": "style_judge", "score": scores["style"], "threshold": style_min, "required_fixes": judge_results.get("style_judge", {}).get("required_fixes", [])}
    if scores["style"] < style_min:
        if style_mode == "blocking":
            failures.append(style_issue)
        else:
            warnings.append({**style_issue, "advisory": True})
    elif style_mode != "blocking" and scores["style"] < 6.0:
        warnings.append({**style_issue, "advisory": True, "note": "Style is below preferred level but deterministic gates passed; do not trigger destructive revision."})

    return {
        "section_name": section_name,
        "approved": len(failures) == 0,
        "reason": "approved_after_deterministic_and_supported_scope_judges" if len(failures) == 0 else "judge_scores_below_supported_scope_floor",
        "scores": scores,
        "failures": failures,
        "warnings": warnings,
        "thresholds": cfg,
        "approval_policy": "final: deterministic gates/fact-lock/cleanliness are binding; IFRS and evidence judges use IFRS_SUPPORTED_SCOPE_JUDGE_FLOOR; legacy strict thresholds are ignored unless IFRS_USE_LEGACY_STRICT_APPROVAL=1.",
        "version": SENIOR_IFRS_VERSION,
    }


# [flattened] superseded `_supported_scope_supported_scope_coverage_score` removed; canonical version in the scoring-calibration cell.


# [flattened] superseded `score_section_generation_output` removed; canonical version in the scoring-calibration cell.


def _gate_by_name(gates: List[Dict[str, Any]], name: str) -> Optional[Dict[str, Any]]:
    for g in gates:
        if g.get("gate_name") == name:
            return g
    return None


def section_quality_metrics(section_name: str, finalized_text: str, gates: List[Dict[str, Any]]) -> Dict[str, Any]:
    """Continuous faithfulness/compliance metrics for one finalized section.

    Everything here is deterministic and reuses signal the gates already produced,
    so it adds no LLM cost. Rates are in [0, 1] (higher = better) unless noted.
    """
    text = str(finalized_text or "")
    word_count = len(re.findall(r"\b\w+\b", text))

    # --- Numeric & entity grounding (from the fact-lock gate) ---
    fact = _gate_by_name(gates, "factlock_numbers_entities") or {}
    fact_failures = fact.get("failures", []) or []
    fact_warnings = fact.get("warnings", []) or []
    unsupported_numbers = sum(1 for f in fact_failures if f.get("type") == "number_not_in_payload_or_claims")
    supported_numbers = sum(1 for w in fact_warnings if w.get("type") == "rounded_number_supported_by_payload_or_claim")
    # All numbers considered = supported (warned) + unsupported (failed) + small-int contexts.
    small_int_ctx = sum(1 for w in fact_warnings if w.get("type") == "small_unsupported_integer_check_context")
    numbers_checked = supported_numbers + unsupported_numbers + small_int_ctx
    numeric_grounding_rate = round(1.0 - unsupported_numbers / numbers_checked, 4) if numbers_checked else 1.0
    unsupported_entities = sum(1 for w in fact_warnings if w.get("type") == "entity_not_in_payload_or_claims")

    # --- Claim integrity (from the claims-integrity gate) ---
    claims = _gate_by_name(gates, "claims_integrity") or {}
    claim_failures = len(claims.get("failures", []) or [])

    # --- Limitation-language policy (two-tier scanner: Fix 1) ---
    limitation_violations = 0
    sanctioned_count = 0
    if "scan_limitation_language" in globals():
        scan = scan_limitation_language(text)
        limitation_violations = len(scan.get("tier1_hits", [])) + len(scan.get("unsanctioned_tier2", []))
        sanctioned_count = len(scan.get("sanctioned_sentences", []))
    per_1k = round(1000.0 * limitation_violations / word_count, 3) if word_count else 0.0

    # --- Gate pass fraction (how close to fully clean) ---
    gate_pass_fraction = round(sum(1 for g in gates if g.get("passed")) / len(gates), 4) if gates else 0.0

    return {
        "word_count": word_count,
        "numbers_checked": numbers_checked,
        "unsupported_numbers": unsupported_numbers,
        "numeric_grounding_rate": numeric_grounding_rate,
        "unsupported_entities": unsupported_entities,
        "claim_integrity_failures": claim_failures,
        "limitation_policy_violations": limitation_violations,
        "limitation_violations_per_1k_words": per_1k,
        "sanctioned_limitation_statements": sanctioned_count,
        "gate_pass_fraction": gate_pass_fraction,
    }


def run_deterministic_gates(section_name: str, draft_markdown: str, claims_register: Dict[str, Any]) -> Dict[str, Any]:  # noqa: F811
    """Canonical deterministic gate runner.

    Runs every gate against the SAME finalized text that will be saved, so the
    gate verdict and the stored section can never diverge. Gate functions are
    single canonical definitions from the cells above.
    """
    finalized = finalize_section_prose(draft_markdown, section_name)
    repaired_claims = repair_claim_evidence_sources(section_name, claims_register)

    gates = [
        draft_structural_quality_gate(section_name, finalized),
        draft_depth_quality_gate(section_name, finalized),
        draft_prose_polish_gate(section_name, finalized),
        unsupported_boilerplate_gate(section_name, finalized),
        claims_integrity_gate(section_name, repaired_claims),
        factlock_gate(section_name, finalized, repaired_claims),
        reference_firewall_gate(finalized),
        report_cleanliness_gate(finalized),
    ]

    # Config-driven senior report-quality gate (identifier/formula/absence checks).
    if "engine_quality_issues" in globals():
        quality_issues = engine_quality_issues(finalized, section_name)
        gates.append({
            "gate_name": "engine_config_driven_senior_report_quality",
            "passed": len(quality_issues) == 0,
            "failures": quality_issues,
            "warnings": [],
        })

    result = {
        "section_name": section_name,
        "passed": all(g.get("passed", False) for g in gates),
        "gates": gates,
        "supported_scope_finalized_for_gate": finalized != draft_markdown,
    }
    result["summary"] = summarize_deterministic_failures(result)
    # [Evaluation] Continuous per-section quality rates derived from the gates that
    # just ran, so every section carries faithfulness/compliance signal, not only a
    # binary pass/fail. Consumed by the evaluation scorecard (CELL 23).
    result["metrics"] = section_quality_metrics(section_name, finalized, gates)
    return result


def revise_section_minimally(  # noqa: F811
    section_name: str,
    current_markdown: str,
    claims_register: Dict[str, Any],
    deterministic_result: Dict[str, Any],
    judge_results: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
) -> Dict[str, Any]:
    """Avoid damaging deterministic-clean drafts; revise only when hard gates fail or judge floors truly fail."""
    current_clean = finalize_section_prose(current_markdown, section_name)

    if deterministic_result.get("passed", False) and approval.get("approved", False):
        return {
            "section_name": section_name,
            "revised_markdown": current_clean,
            "revision_notes": "No revision needed: deterministic gates passed and supported-scope approval approval policy approved the section.",
            "senior_ifrs_version": SENIOR_IFRS_VERSION,
        }

    # If deterministic gates passed but the section only has advisory style warnings,
    # do not ask the LLM to rewrite and risk introducing placeholders.
    if deterministic_result.get("passed", False) and not approval.get("failures"):
        return {
            "section_name": section_name,
            "revised_markdown": current_clean,
            "revision_notes": "No destructive revision: only advisory judge/style warnings remained.",
            "senior_ifrs_version": SENIOR_IFRS_VERSION,
        }

    revised = _llm_revise_section(
        section_name,
        current_clean,
        claims_register,
        deterministic_result,
        judge_results,
        approval,
    )
    revised["revised_markdown"] = finalize_section_prose(revised.get("revised_markdown", current_clean), section_name)
    revised["senior_ifrs_version"] = SENIOR_IFRS_VERSION
    return revised


def run_section_pipeline(section_name: str) -> Dict[str, Any]:  # noqa: F811
    """supported-scope approval section loop: approve clean iteration-0 drafts instead of over-revising them."""
    print("=" * 100)
    print("SECTION:", section_name)
    print("=" * 100)

    previous_issue_signature = None
    draft = write_section_draft(section_name)
    draft_markdown = finalize_section_prose(draft.get("draft_markdown", ""), section_name)
    approval = {"approved": False, "reason": "not_run"}

    for iteration in range(0, MAX_REVISION_LOOPS + 1):
        draft_markdown = finalize_section_prose(draft_markdown, section_name)

        print(f"Iteration {iteration} — building claims register...")
        claims = repair_claim_evidence_sources(section_name, build_claims_register(section_name, draft_markdown))

        print(f"Iteration {iteration} — deterministic gates...")
        deterministic = run_deterministic_gates(section_name, draft_markdown, claims)

        judges = None
        if deterministic.get("passed", False):
            print(f"Iteration {iteration} — LLM judges...")
            judges = run_llm_judges(section_name, draft_markdown, claims)
        else:
            print(f"Iteration {iteration} — deterministic gates failed, skipping LLM judges.")
            print(deterministic.get("summary", summarize_deterministic_failures(deterministic)))

        approval = composite_approval_gate(section_name, deterministic, judges)
        section_score = score_section_generation_output(section_name, draft_markdown, deterministic, judges, approval)
        approval["section_generation_score"] = section_score

        if section_score.get("missing_data_language_hits"):
            approval["approved"] = False
            approval.setdefault("failures", []).append({
                "gate": "report_cleanliness_missing_data_language",
                "required_fixes": section_score["missing_data_language_hits"],
            })

        save_section_iteration(
            section_name,
            iteration,
            {"section_name": section_name, "draft_markdown": draft_markdown, "senior_ifrs_version": SENIOR_IFRS_VERSION},
            claims,
            deterministic,
            judges,
            approval,
        )

        print("Approval:", approval.get("approved"), approval.get("scores", approval.get("reason", "")), "| section score:", section_score["overall_section_generation_score_0_to_100"])

        if approval.get("approved"):
            slug = SECTION_SLUGS[section_name]
            approved_md_path = DIRS["approved"] / f"approved_{slug}.md"
            approved_json_path = DIRS["approved"] / f"approved_{slug}.json"
            write_text(draft_markdown, approved_md_path)
            write_json({
                "section_name": section_name,
                "status": "approved",
                "draft_markdown": draft_markdown,
                "claims_register": claims,
                "coverage_matrix_path": str(DIRS["coverage"] / f"coverage_matrix_{slug}.json"),
                "missing_requirements_path": str(DIRS["missing_requirements"] / f"missing_requirements_{slug}.json"),
                "approval": approval,
                "section_generation_score": section_score,
                # [Evaluation] Persist the deterministic result (which now carries the
                # continuous per-section quality metrics) so the scorecard can read
                # them without re-running the gates.
                "deterministic": deterministic,
                "section_metrics": deterministic.get("metrics", {}),
                "senior_ifrs_version": SENIOR_IFRS_VERSION,
            }, approved_json_path)
            return {
                "section_name": section_name,
                "status": "approved",
                "approved_markdown_path": str(approved_md_path),
                "approved_json_path": str(approved_json_path),
                "iterations": iteration,
                "approval": approval,
                "section_generation_score": section_score,
            }

        sig = same_issue_signature(approval)
        if sig == previous_issue_signature and deterministic.get("passed", False):
            # Avoid repeated destructive judge-driven revisions on already clean text.
            print("Repeated non-deterministic approval issue on clean gates; escalating without further rewriting.")
            break
        previous_issue_signature = sig

        if iteration >= MAX_REVISION_LOOPS:
            print("Max revision loops reached. Escalating to human_review.")
            break

        print(f"Iteration {iteration} — senior revising...")
        revised = revise_section_minimally(section_name, draft_markdown, claims, deterministic, judges, approval)
        draft_markdown = finalize_section_prose(revised.get("revised_markdown", draft_markdown), section_name)
        write_json(revised, DIRS["revisions"] / f"revision_{SECTION_SLUGS[section_name]}_iter{iteration}.json")

    slug = SECTION_SLUGS[section_name]
    review_path = DIRS["approved"] / f"human_review_{slug}.md"
    write_text(finalize_section_prose(draft_markdown, section_name), review_path)
    final_score = approval.get("section_generation_score", {})
    return {
        "section_name": section_name,
        "status": "human_review",
        "markdown_path": str(review_path),
        "approval": approval,
        "section_generation_score": final_score,
        "senior_ifrs_version": SENIOR_IFRS_VERSION,
    }

# Self-test the approval behavior that caused the human-review loop.
_supported_scope_mock_det = {"passed": True, "gates": []}
_supported_scope_mock_judges = {
    "ifrs_coverage_judge": {"ifrs_coverage_score_0_to_10": 6.0},
    "evidence_judge": {"evidence_score_0_to_10": 6.0},
    "style_judge": {"style_score_0_to_10": 3.8},
}
_supported_scope_mock_approval = composite_approval_gate("General Requirements", _supported_scope_mock_det, _supported_scope_mock_judges)
assert _supported_scope_mock_approval["approved"] is True, _supported_scope_mock_approval
assert REPORT_ENGINE_CONFIG["approval"]["ifrs_coverage_score_min"] <= 6.0, REPORT_ENGINE_CONFIG["approval"]
assert REPORT_ENGINE_CONFIG["approval"]["evidence_score_min"] <= 6.0, REPORT_ENGINE_CONFIG["approval"]

print(f"Loaded production IFRS report engine. Clean deterministic sections with supported-scope judge scores now approve before destructive revision.")


Loaded production IFRS report engine. Clean deterministic sections with supported-scope judge scores now approve before destructive revision.


## Production final quality reconciliation engine

Adds a reusable, evidence-grounded final quality layer before report assembly. This is not a hardcoded text implementation: it detects cross-section inconsistencies, sparse/missing-looking tables, and stale supported facts from the approved sections, then reconciles them against the payload evidence pack.

In [35]:

# ============================================================
# CELL 12L / 21A — DETERMINISTIC TABLE-HYGIENE + FINAL-QA HELPERS
# (The LLM final-quality reconciliation stage was removed: it added an LLM
#  round-trip with no measured benefit and its deterministic table hygiene is
#  applied by the assembler and the editorial polish. Only the deterministic
#  helpers still used by the editorial stage are retained here.)
# ============================================================
# Why this implementation exists:
# - The supported-scope approval layer solved unnecessary human_review escalation.
# - The remaining issues are final-report quality issues: cross-section
#   inconsistencies and missing-looking table cells such as em dashes.
# - These must be handled generically, not with hardcoded replacements.
#
# Final QA policy:
# - Approved sections remain the unit of generation.
# - Before final assembly, approved sections pass through a reusable final QA
#   reconciler driven by payload evidence and deterministic hygiene rules.
# - The reconciler may remove sparse unsupported table columns/rows, resolve
#   contradictions using evidence, and harmonise repeated metrics.
# - Missing requirements remain audit-only and are never printed in prose.
# ============================================================

SENIOR_IFRS_VERSION = "production_ifrs_report_engine"

DIRS.setdefault("final_quality", OUTPUT_DIR / "13_final_quality")
DIRS["final_quality"].mkdir(parents=True, exist_ok=True)


def _final_qa_bool_env(name: str, default: bool) -> bool:
    """Read a boolean environment variable with explicit validation."""
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return bool(default)
    normalized = raw.strip().lower()
    if normalized in {"1", "true", "yes", "on"}:
        return True
    if normalized in {"0", "false", "no", "off"}:
        return False
    raise ValueError(
        f"Environment variable {name} must be one of "
        "1/0, true/false, yes/no, or on/off."
    )


def _final_qa_int_env(name: str, default: int) -> int:
    """Read an integer environment variable with a useful error message."""
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return int(default)
    try:
        return int(raw.strip())
    except ValueError as exc:
        raise ValueError(f"Environment variable {name} must be an integer; got {raw!r}.") from exc


def _final_qa_float_env(name: str, default: float) -> float:
    """Read a floating-point environment variable with a useful error message."""
    raw = os.getenv(name)
    if raw is None or not raw.strip():
        return float(default)
    try:
        return float(raw.strip())
    except ValueError as exc:
        raise ValueError(f"Environment variable {name} must be numeric; got {raw!r}.") from exc

REPORT_ENGINE_CONFIG.setdefault("final_quality", {})
REPORT_ENGINE_CONFIG["final_quality"].update({
    "enabled": _final_qa_bool_env("IFRS_ENABLE_FINAL_QA_RECONCILIATION", True),
    "strict": _final_qa_bool_env("IFRS_FINAL_QA_STRICT", True),
    "llm_reconciliation_enabled": _final_qa_bool_env("IFRS_FINAL_QA_LLM_RECONCILIATION", True),
    "remove_sparse_table_columns": _final_qa_bool_env("IFRS_FINAL_QA_REMOVE_SPARSE_TABLE_COLUMNS", True),
    "remove_sparse_table_rows": _final_qa_bool_env("IFRS_FINAL_QA_REMOVE_SPARSE_TABLE_ROWS", True),
    "max_evidence_facts": _final_qa_int_env("IFRS_FINAL_QA_MAX_EVIDENCE_FACTS", 1600),
    "max_numeric_claims": _final_qa_int_env("IFRS_FINAL_QA_MAX_NUMERIC_CLAIMS", 400),
    "table_column_missing_ratio_max": _final_qa_float_env("IFRS_TABLE_COLUMN_MISSING_RATIO_MAX", 0.0),
    "table_row_missing_ratio_max": _final_qa_float_env("IFRS_TABLE_ROW_MISSING_RATIO_MAX", 0.0),
    "max_reconciliation_context_chars": _final_qa_int_env("IFRS_FINAL_QA_CONTEXT_CHARS", 110000),
})


_final_qa_demo_md = """| Metric | 2024 value | Note |
| --- | --- | --- |
| Demonstration metric | 10 | — |
"""


_FINAL_QA_MISSING_CELL_VALUES = {
    "", "-", "–", "—", "n/a", "na", "n.a.", "none", "null",
    "not available", "not applicable", "unknown", "missing", "no data"
}


def _final_qa_cell_is_missing_like(cell: Any) -> bool:
    text = re.sub(r"<br\s*/?>", " ", str(cell or ""), flags=re.I).strip()
    text = re.sub(r"\*\*|__|`", "", text).strip()
    return text.lower() in _FINAL_QA_MISSING_CELL_VALUES


def _final_qa_split_md_table_row(line: str) -> List[str]:
    raw = line.strip()
    if raw.startswith("|"):
        raw = raw[1:]
    if raw.endswith("|"):
        raw = raw[:-1]
    return [c.strip() for c in raw.split("|")]


def _final_qa_is_separator_row(line: str) -> bool:
    cells = _final_qa_split_md_table_row(line)
    if not cells:
        return False
    return all(re.fullmatch(r":?-{3,}:?", c.replace(" ", "")) for c in cells)


def _final_qa_table_blocks(markdown: str) -> List[Dict[str, Any]]:
    lines = str(markdown or "").splitlines()
    blocks = []
    i = 0
    while i < len(lines):
        if "|" in lines[i] and i + 1 < len(lines) and _final_qa_is_separator_row(lines[i + 1]):
            start = i
            i += 2
            while i < len(lines) and "|" in lines[i].strip():
                i += 1
            blocks.append({"start": start, "end": i, "lines": lines[start:i]})
        else:
            i += 1
    return blocks


def _final_qa_render_table(header: List[str], rows: List[List[str]]) -> List[str]:
    if not header or not rows:
        return []
    sep = ["---" for _ in header]
    def render(row):
        return "| " + " | ".join(str(c).strip() for c in row) + " |"
    return [render(header), render(sep)] + [render(r) for r in rows]


def _final_qa_clean_sparse_markdown_tables(markdown: str) -> Tuple[str, List[Dict[str, Any]]]:
    """Remove missing-looking table cells generically without knowing the section.

    This does not invent replacement values. If a column or row contains an
    unsupported placeholder/dash, the unsupported disclosure is removed or the
    table is left for the LLM reconciler if deterministic removal would destroy
    the table.
    """
    text = str(markdown or "")
    lines = text.splitlines()
    blocks = _final_qa_table_blocks(text)
    if not blocks:
        return text, []

    changes = []
    new_lines = list(lines)
    offset = 0
    cfg = REPORT_ENGINE_CONFIG.get("final_quality", {})

    for block_idx, block in enumerate(blocks):
        start = block["start"] + offset
        end = block["end"] + offset
        block_lines = new_lines[start:end]
        if len(block_lines) < 3:
            continue
        header = _final_qa_split_md_table_row(block_lines[0])
        rows = [_final_qa_split_md_table_row(line) for line in block_lines[2:]]
        if not header or not rows:
            continue
        width = len(header)
        rows = [r + [""] * (width - len(r)) if len(r) < width else r[:width] for r in rows]

        missing_by_col = []
        for j in range(width):
            ratio = sum(_final_qa_cell_is_missing_like(r[j]) for r in rows) / max(len(rows), 1)
            missing_by_col.append(ratio)

        keep_cols = list(range(width))
        removed_cols = []
        if cfg.get("remove_sparse_table_columns", True) and width > 2:
            for j, ratio in enumerate(missing_by_col):
                if ratio > cfg.get("table_column_missing_ratio_max", 0.0):
                    # Keep the first column if it is the only descriptor column.
                    if j == 0:
                        continue
                    if len(keep_cols) - 1 >= 2:
                        removed_cols.append(header[j])
                        keep_cols.remove(j)

        header2 = [header[j] for j in keep_cols]
        rows2 = [[r[j] for j in keep_cols] for r in rows]

        kept_rows = []
        removed_row_count = 0
        for r in rows2:
            ratio = sum(_final_qa_cell_is_missing_like(c) for c in r) / max(len(r), 1)
            if cfg.get("remove_sparse_table_rows", True) and ratio > cfg.get("table_row_missing_ratio_max", 0.0) and len(rows2) - removed_row_count > 1:
                removed_row_count += 1
                continue
            kept_rows.append(r)

        rendered = _final_qa_render_table(header2, kept_rows)
        if rendered and rendered != block_lines:
            new_lines[start:end] = rendered
            offset += len(rendered) - (end - start)
            changes.append({
                "type": "sparse_table_hygiene",
                "table_index": block_idx,
                "removed_columns": removed_cols,
                "removed_rows": removed_row_count,
            })

    return "\n".join(new_lines).strip() + "\n", changes


def _final_qa_humanize_path(path: str) -> str:
    leaf = str(path or "").split(".")[-1]
    leaf = re.sub(r"\[[0-9]+\]", "", leaf)
    leaf = leaf.replace("_", " ")
    leaf = re.sub(r"\s+", " ", leaf).strip()
    return leaf


def _final_qa_collect_evidence_facts() -> List[Dict[str, Any]]:
    """Build a compact source-of-truth evidence pack from loaded payloads.

    It is intentionally generic: facts are derived from payload paths and values,
    not from section-specific replacement rules.
    """
    facts = []
    max_facts = REPORT_ENGINE_CONFIG.get("final_quality", {}).get("max_evidence_facts", 1600)
    seen = set()

    for section in SECTIONS:
        payload = payloads_by_section.get(section, {})
        flat = flatten_json(payload) if payload else []
        for path, value in flat:
            try:
                if not writer_evidence_path_allowed(path):
                    continue
            except Exception:
                pass
            try:
                if is_missing_like_value(value):
                    continue
            except Exception:
                if _final_qa_cell_is_missing_like(value):
                    continue
            preview = value_preview(value) if "value_preview" in globals() else str(value)
            if not str(preview).strip():
                continue
            if len(str(preview)) > 260:
                continue
            key = (section, str(path), str(preview))
            if key in seen:
                continue
            seen.add(key)
            facts.append({
                "section_source": section,
                "path": str(path),
                "label": _final_qa_humanize_path(str(path)),
                "value": preview,
            })
            if len(facts) >= max_facts:
                return facts
    return facts


def _final_qa_numeric_claims_from_sections(sections: Dict[str, str]) -> List[Dict[str, Any]]:
    claims = []
    max_claims = REPORT_ENGINE_CONFIG.get("final_quality", {}).get("max_numeric_claims", 400)
    sent_split = re.compile(r"(?<=[.!?])\s+")
    for section, md in sections.items():
        plain = re.sub(r"\|", " ", str(md or ""))
        for sent in sent_split.split(plain):
            s = re.sub(r"\s+", " ", sent).strip()
            if not s or not re.search(r"\d", s):
                continue
            nums = extract_numbers(s) if "extract_numbers" in globals() else re.findall(r"\d+(?:[,.]\d+)*", s)
            if not nums:
                continue
            claims.append({"section": section, "numbers": nums[:8], "text": s[:420]})
            if len(claims) >= max_claims:
                return claims
    return claims


def _final_qa_table_missing_locations(sections: Dict[str, str]) -> List[Dict[str, Any]]:
    issues = []
    for section, md in sections.items():
        for idx, block in enumerate(_final_qa_table_blocks(md)):
            lines = block["lines"]
            if len(lines) < 3:
                continue
            header = _final_qa_split_md_table_row(lines[0])
            rows = [_final_qa_split_md_table_row(line) for line in lines[2:]]
            for r_i, row in enumerate(rows):
                for c_i, cell in enumerate(row):
                    if _final_qa_cell_is_missing_like(cell):
                        issues.append({
                            "section": section,
                            "table_index": idx,
                            "row_index": r_i,
                            "column": header[c_i] if c_i < len(header) else f"column_{c_i}",
                            "cell": str(cell),
                        })
    return issues















def _final_qa_load_approved_sections() -> Dict[str, str]:
    """Local loader so final QA can run before the connectivity cell defines load_approved_sections()."""
    approved = {}
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        path = DIRS["approved"] / f"approved_{slug}.md"
        if path.exists():
            approved[section] = read_text(path)
    return approved




# Self-tests: deterministic table hygiene must be generic, not section-specific.
_final_qa_cleaned_demo, _final_qa_demo_changes = _final_qa_clean_sparse_markdown_tables(_final_qa_demo_md)
assert "—" not in _final_qa_cleaned_demo, _final_qa_cleaned_demo
assert _final_qa_demo_changes, "table hygiene self-test did not record a change"

print(f"Loaded production IFRS report engine. Final QA reconciliation will run after section generation and before final assembly.")


Loaded production IFRS report engine. Final QA reconciliation will run after section generation and before final assembly.


## Supported-scope scoring calibration

This cell separates deterministic requirement/evidence coverage from LLM judge opinion. The coverage component uses the coverage matrix produced by the evidence mapper, while the LLM judges remain a separate quality signal. This avoids double-penalising sections for the same issue and gives a more transparent section-generation score.

In [36]:
# ============================================================
# CELL 18G — SUPPORTED-SCOPE SCORING CALIBRATION
# ============================================================
# Production scoring policy:
# - The coverage component comes from the deterministic requirement/evidence
#   coverage matrix.
# - LLM coverage, evidence and style judges remain a separate quality signal.
# - Missing synthetic payload items stay in payload-readiness/audit outputs and
#   do not directly lower the generation score of an evidence-supported section.
# ============================================================

# --- Scoring policy (reporting-only; NOT an approval input) ------------------
# The 0-100 section score is a QUALITY-RANKING signal for reporting, not a gate.
# Approval is decided separately and strictly by composite_approval_gate (a hard
# cascade: deterministic gates must pass, then IFRS-coverage and evidence judges
# must clear their minimum thresholds). Nothing should wire this score back into
# an approval decision -- doing so would let a strong coverage number "buy back" a
# real evidence deficiency, which is unacceptable for a compliance artifact.
#
# The score is a weighted mean of the only two components that VARY continuously
# between a mediocre and a strong section:
#   - supported-scope coverage  (deterministic evidence/coverage matrix)
#   - judge average             (LLM coverage/evidence/style judges)
# The previous formula also averaged in "deterministic gates passed" (0/100) and
# "no missing-data language" (0/100). Both are ENTRY CONDITIONS a scored section
# has already satisfied, so they were near-constant 100 and simply compressed the
# score into a narrow band near the top (~45% of the weight never moved), and they
# double-counted the hard-gate result. They are now reported as pass/fail FLAGS
# beside the score, not averaged into it.
#
# Weight split rationale: the deterministic coverage signal is defensible and
# reproducible; the LLM judges are an unvalidated quality opinion. Coverage is
# therefore weighted well above the judges. Set IFRS_SCORE_JUDGE_WEIGHT=0.0 to
# make the judges advisory-only (coverage becomes the whole score).
REPORT_ENGINE_CONFIG.setdefault("scoring", {})
_score_judge_weight = float(os.getenv("IFRS_SCORE_JUDGE_WEIGHT", "0.25"))
_score_coverage_weight = float(os.getenv("IFRS_SCORE_COVERAGE_WEIGHT", str(round(1.0 - _score_judge_weight, 4))))
REPORT_ENGINE_CONFIG["scoring"].update({
    "coverage_source": os.getenv("IFRS_SCORE_COVERAGE_SOURCE", "deterministic_supported_scope"),
    "coverage_weight": _score_coverage_weight,
    "judge_weight": _score_judge_weight,
})


def _score_float(value: Any, default: float = 0.0) -> float:
    try:
        if value is None:
            return float(default)
        return float(value)
    except Exception:
        return float(default)


def deterministic_supported_scope_coverage_score(section_name: str) -> float:
    """Return the deterministic generation-coverage score for supported requirements.

    This score is based on the evidence/coverage matrix, not on an LLM judge.
    It therefore measures whether the supported disclosure plan has evidence,
    while the judge scores measure writing quality and judgement.
    """
    component = coverage_score_for_section(section_name)
    return round(_score_float(component.get("supported_scope_coverage_score_0_to_100", component.get("coverage_score_0_to_100", 0.0))), 2)


def score_section_generation_output(  # noqa: F811
    section_name: str,
    draft_markdown: str,
    deterministic: Dict[str, Any],
    judges: Optional[Dict[str, Any]],
    approval: Dict[str, Any],
) -> Dict[str, Any]:
    payload_readiness_component = coverage_score_for_section(section_name)
    supported_scope_coverage_score = deterministic_supported_scope_coverage_score(section_name)
    missing_register = missing_registers_by_section.get(section_name, {}) if "missing_registers_by_section" in globals() else {}
    missing_count = missing_register.get("missing_requirements_count", len(missing_register.get("missing_requirements", [])))
    missing_hits = scan_for_missing_data_language(draft_markdown) if "scan_for_missing_data_language" in globals() else []

    # Entry-condition FLAGS (reported beside the score, never averaged into it).
    deterministic_passed = bool(deterministic.get("passed", False))
    is_clean = not missing_hits

    # Continuous quality components (the only parts that vary meaningfully).
    judge_average = _supported_scope_weighted_judge_average_0_to_100(judges)

    weights = REPORT_ENGINE_CONFIG.get("scoring", {})
    coverage_weight = _score_float(weights.get("coverage_weight"), 0.75)
    judge_weight = _score_float(weights.get("judge_weight"), 0.25)
    weight_sum = max(coverage_weight + judge_weight, 1e-9)

    # If judges are unavailable, score on coverage alone rather than dragging the
    # number down with a 0 judge average.
    if judges is None:
        overall = round(supported_scope_coverage_score, 2)
    else:
        overall = round(
            (coverage_weight * supported_scope_coverage_score + judge_weight * judge_average) / weight_sum,
            2,
        )

    return {
        "section_name": section_name,
        "overall_section_generation_score_0_to_100": overall,
        # Continuous components:
        "supported_scope_coverage_score_0_to_100": supported_scope_coverage_score,
        "judge_average_0_to_100": judge_average,
        "judge_scores_0_to_10": _supported_scope_judge_scores(judges) if judges else None,
        # Entry-condition flags (pass/fail, not weighted):
        "deterministic_gates_passed": deterministic_passed,
        "report_is_clean": is_clean,
        # Reported-separately context:
        "payload_readiness_component": payload_readiness_component,
        "missing_requirements_count_flagged": missing_count,
        "missing_requirement_ids_flagged": missing_register.get("missing_requirement_ids", []),
        "missing_data_language_hits": missing_hits,
        "approved": approval.get("approved", False) and not missing_hits,
        "weights": {"coverage": coverage_weight, "judges": judge_weight},
        "scoring_policy": (
            "Reporting-only quality score = weighted mean of deterministic supported-scope "
            "coverage and the LLM judge average, with coverage weighted above the (unvalidated) "
            "judges. Deterministic-gate pass and report cleanliness are entry conditions, "
            "reported as flags rather than averaged in. Approval is decided separately by "
            "composite_approval_gate, not by this score."
        ),
    }

# Semantic self-tests: verify the score DISCRIMINATES and that entry-condition
# flags are NOT averaged into it. These check scoring behaviour, not content.
def _scoring_selftest() -> None:
    det_pass = {"passed": True, "gates": []}
    approval = {"approved": True}
    hi = {"ifrs_coverage_judge": {"ifrs_coverage_score_0_to_10": 9.0},
          "evidence_judge": {"evidence_score_0_to_10": 9.0},
          "style_judge": {"style_score_0_to_10": 9.0}}
    lo = {"ifrs_coverage_judge": {"ifrs_coverage_score_0_to_10": 3.0},
          "evidence_judge": {"evidence_score_0_to_10": 3.0},
          "style_judge": {"style_score_0_to_10": 3.0}}
    s_hi = score_section_generation_output("General Requirements", "text", det_pass, hi, approval)
    s_lo = score_section_generation_output("General Requirements", "text", det_pass, lo, approval)

    # (a) Better judges must produce a strictly higher score (score is not flat).
    assert s_hi["overall_section_generation_score_0_to_100"] > s_lo["overall_section_generation_score_0_to_100"], \
        "score does not discriminate on judge quality"

    # (b) Binary entry conditions are flags, not weighted terms: flipping the
    #     deterministic flag must NOT change the numeric score (same coverage+judges).
    s_detfail = score_section_generation_output("General Requirements", "text", {"passed": False}, hi, {"approved": False})
    assert s_detfail["overall_section_generation_score_0_to_100"] == s_hi["overall_section_generation_score_0_to_100"], \
        "deterministic pass/fail leaked into the weighted score"
    assert s_detfail["deterministic_gates_passed"] is False and s_hi["deterministic_gates_passed"] is True

    # (c) Coverage must outweigh judges: with judges at 0, the score stays close to
    #     coverage (within the judge weight), never collapsing to ~0.
    zero_judges = {"ifrs_coverage_judge": {"ifrs_coverage_score_0_to_10": 0.0},
                   "evidence_judge": {"evidence_score_0_to_10": 0.0},
                   "style_judge": {"style_score_0_to_10": 0.0}}
    s_zero = score_section_generation_output("General Requirements", "text", det_pass, zero_judges, approval)
    cov = s_zero["supported_scope_coverage_score_0_to_100"]
    jw = REPORT_ENGINE_CONFIG["scoring"]["judge_weight"]
    assert s_zero["overall_section_generation_score_0_to_100"] >= cov * (1.0 - jw) - 0.01, \
        "coverage is not the dominant term"


_scoring_selftest()
print("Loaded section scoring: two continuous components (coverage-dominant) + "
      "pass/fail flags. Score is reporting-only; approval is the separate cascade.")


Loaded section scoring: two continuous components (coverage-dominant) + pass/fail flags. Score is reporting-only; approval is the separate cascade.


In [ ]:
# ============================================================
# CELL 19-LG — LANGGRAPH AGENTIC PER-SECTION ORCHESTRATION
# ============================================================
# Re-orchestrates the per-section write->gate->judge->approve/revise flow as a
# LangGraph StateGraph. It calls the SAME deterministic tools and LLM stages as
# the legacy run_section_pipeline (nothing about writing/judging/approval logic
# changes), but replaces the fixed for-loop with a router that reacts to WHY a
# section failed:
#
#   - deterministic gate hard-fail (arithmetic / limitation / omission) -> revise
#   - evidence/grounding judge below threshold                          -> revise
#   - coverage-gap dominant failure                                     -> redraft
#     (re-plan via build_disclosure_plan then re-draft — a different repair than
#      minimal revision, which the linear loop could never take)
#   - repeated identical issue on clean gates, or max loops reached     -> escalate
#
# Determinism principle: the LLM decides ONLY the control flow (which node next).
# Whether a draft is correct/compliant/approved is still decided by the same
# deterministic gates + composite_approval_gate. Approval is never an LLM node.
#
# Additive + safe: legacy run_section_pipeline is untouched. If langgraph is not
# installed, run_section_pipeline_graph falls back to the legacy loop, so the
# notebook always runs. Select at runtime with USE_LANGGRAPH (default on).

try:
    from langgraph.graph import StateGraph, END
    try:
        from typing import TypedDict
    except Exception:
        from typing_extensions import TypedDict
    _LANGGRAPH_AVAILABLE = True
except Exception as _lg_import_exc:  # pragma: no cover
    _LANGGRAPH_AVAILABLE = False
    print("LangGraph not available:", repr(_lg_import_exc))
    print("-> run_section_pipeline_graph will fall back to the legacy loop.")


def _lg_classify_failure(deterministic, judges, approval):
    """Classify a non-approved iteration so the router can pick a repair path."""
    if not (deterministic or {}).get("passed", False):
        return "deterministic"
    appr = REPORT_ENGINE_CONFIG.get("approval", {}) if "REPORT_ENGINE_CONFIG" in globals() else {}
    cov_min = float(appr.get("ifrs_coverage_score_min", 5.0))
    ev_min = float(appr.get("evidence_score_min", appr.get("evidence_score_0_to_10_min", 5.0)))
    j = judges or {}
    cov = (j.get("ifrs_coverage_judge") or {}).get("ifrs_coverage_score_0_to_10")
    ev = (j.get("evidence_judge") or {}).get("evidence_score_0_to_10")
    fails = approval.get("failures", []) or []
    if any(("coverage" in str(f).lower()) or ("missing" in str(f).lower()) for f in fails):
        return "coverage"
    if cov is not None and cov < cov_min and (ev is None or cov <= ev):
        return "coverage"
    if ev is not None and ev < ev_min:
        return "grounding"
    return "revise"


if _LANGGRAPH_AVAILABLE:

    class SectionState(TypedDict, total=False):
        section_name: str
        draft_markdown: str
        claims: dict
        deterministic: dict
        judges: object
        approval: dict
        section_score: dict
        iteration: int
        previous_issue_signature: object
        route: str
        decisions: list
        result: dict

    def _lg_writer(state):
        section = state["section_name"]
        draft = write_section_draft(section)
        md = finalize_section_prose(draft.get("draft_markdown", ""), section)
        return {"draft_markdown": md, "iteration": state.get("iteration", 0),
                "decisions": state.get("decisions", [])}

    def _lg_check(state):
        section = state["section_name"]
        iteration = state.get("iteration", 0)
        md = finalize_section_prose(state["draft_markdown"], section)
        print(f"[graph] {section} iter {iteration} — claims register...")
        claims = repair_claim_evidence_sources(section, build_claims_register(section, md))
        print(f"[graph] {section} iter {iteration} — deterministic gates...")
        deterministic = run_deterministic_gates(section, md, claims)
        judges = None
        if deterministic.get("passed", False):
            print(f"[graph] {section} iter {iteration} — LLM judges...")
            judges = run_llm_judges(section, md, claims)
        else:
            print(f"[graph] {section} iter {iteration} — gates failed, skipping judges.")
            print(deterministic.get("summary", summarize_deterministic_failures(deterministic)))
        approval = composite_approval_gate(section, deterministic, judges)
        section_score = score_section_generation_output(section, md, deterministic, judges, approval)
        approval["section_generation_score"] = section_score
        if section_score.get("missing_data_language_hits"):
            approval["approved"] = False
            approval.setdefault("failures", []).append({
                "gate": "report_cleanliness_missing_data_language",
                "required_fixes": section_score["missing_data_language_hits"],
            })
        save_section_iteration(
            section, iteration,
            {"section_name": section, "draft_markdown": md, "senior_ifrs_version": SENIOR_IFRS_VERSION},
            claims, deterministic, judges, approval,
        )
        print("[graph] approval:", approval.get("approved"),
              "| section score:", section_score.get("overall_section_generation_score_0_to_100"))
        return {"draft_markdown": md, "claims": claims, "deterministic": deterministic,
                "judges": judges, "approval": approval, "section_score": section_score}

    def _lg_route(state):
        approval = state.get("approval", {})
        deterministic = state.get("deterministic", {})
        iteration = state.get("iteration", 0)
        if approval.get("approved"):
            return "finalize_approved"
        sig = same_issue_signature(approval)
        if sig == state.get("previous_issue_signature") and deterministic.get("passed", False):
            print("[graph] repeated issue on clean gates -> escalate")
            return "escalate"
        if iteration >= MAX_REVISION_LOOPS:
            print("[graph] max revision loops reached -> escalate")
            return "escalate"
        ftype = _lg_classify_failure(deterministic, state.get("judges"), approval)
        print(f"[graph] failure type = {ftype}")
        return "redraft" if ftype == "coverage" else "revise"

    def _lg_revise(state):
        section = state["section_name"]
        ftype = _lg_classify_failure(state.get("deterministic"), state.get("judges"), state.get("approval"))
        revised = revise_section_minimally(
            section, state["draft_markdown"], state["claims"],
            state["deterministic"], state.get("judges"), state["approval"],
        )
        md = finalize_section_prose(revised.get("revised_markdown", state["draft_markdown"]), section)
        write_json(revised, DIRS["revisions"] / f"revision_{SECTION_SLUGS[section]}_iter{state.get('iteration',0)}.json")
        decisions = list(state.get("decisions", [])) + [
            {"iteration": state.get("iteration", 0), "route": "revise", "reason": ftype}
        ]
        return {"draft_markdown": md, "iteration": state.get("iteration", 0) + 1,
                "previous_issue_signature": same_issue_signature(state["approval"]),
                "route": "revise", "decisions": decisions}

    def _lg_redraft(state):
        section = state["section_name"]
        try:
            build_disclosure_plan(section)
        except Exception as exc:
            print("[graph] redraft plan refresh failed:", repr(exc))
        draft = write_section_draft(section)
        md = finalize_section_prose(draft.get("draft_markdown", state["draft_markdown"]), section)
        decisions = list(state.get("decisions", [])) + [
            {"iteration": state.get("iteration", 0), "route": "redraft", "reason": "coverage_gap"}
        ]
        return {"draft_markdown": md, "iteration": state.get("iteration", 0) + 1,
                "previous_issue_signature": same_issue_signature(state["approval"]),
                "route": "redraft", "decisions": decisions}

    def _lg_finalize_approved(state):
        section = state["section_name"]
        slug = SECTION_SLUGS[section]
        md = state["draft_markdown"]
        approval = state["approval"]
        section_score = state["section_score"]
        deterministic = state["deterministic"]
        claims = state["claims"]
        approved_md_path = DIRS["approved"] / f"approved_{slug}.md"
        approved_json_path = DIRS["approved"] / f"approved_{slug}.json"
        write_text(md, approved_md_path)
        write_json({
            "section_name": section,
            "status": "approved",
            "draft_markdown": md,
            "claims_register": claims,
            "coverage_matrix_path": str(DIRS["coverage"] / f"coverage_matrix_{slug}.json"),
            "missing_requirements_path": str(DIRS["missing_requirements"] / f"missing_requirements_{slug}.json"),
            "approval": approval,
            "section_generation_score": section_score,
            "deterministic": deterministic,
            "section_metrics": deterministic.get("metrics", {}),
            "senior_ifrs_version": SENIOR_IFRS_VERSION,
        }, approved_json_path)
        result = {
            "section_name": section,
            "status": "approved",
            "approved_markdown_path": str(approved_md_path),
            "approved_json_path": str(approved_json_path),
            "iterations": state.get("iteration", 0),
            "approval": approval,
            "section_generation_score": section_score,
            "decision_log": state.get("decisions", []),
        }
        return {"result": result}

    def _lg_escalate(state):
        section = state["section_name"]
        slug = SECTION_SLUGS[section]
        review_path = DIRS["approved"] / f"human_review_{slug}.md"
        write_text(finalize_section_prose(state["draft_markdown"], section), review_path)
        final_score = state.get("approval", {}).get("section_generation_score", {})
        result = {
            "section_name": section,
            "status": "human_review",
            "markdown_path": str(review_path),
            "approval": state.get("approval", {}),
            "section_generation_score": final_score,
            "senior_ifrs_version": SENIOR_IFRS_VERSION,
            "decision_log": state.get("decisions", []),
        }
        return {"result": result}

    def _build_section_agent_graph():
        g = StateGraph(SectionState)
        g.add_node("writer", _lg_writer)
        g.add_node("check", _lg_check)
        g.add_node("revise", _lg_revise)
        g.add_node("redraft", _lg_redraft)
        g.add_node("finalize_approved", _lg_finalize_approved)
        g.add_node("escalate", _lg_escalate)
        g.set_entry_point("writer")
        g.add_edge("writer", "check")
        g.add_conditional_edges("check", _lg_route, {
            "finalize_approved": "finalize_approved",
            "escalate": "escalate",
            "redraft": "redraft",
            "revise": "revise",
        })
        g.add_edge("redraft", "check")
        g.add_edge("revise", "check")
        g.add_edge("finalize_approved", END)
        g.add_edge("escalate", END)
        return g.compile()

    SECTION_AGENT_GRAPH = _build_section_agent_graph()

    def run_section_pipeline_graph(section_name: str):
        print("=" * 100)
        print("SECTION (LangGraph agent):", section_name)
        print("=" * 100)
        init = {
            "section_name": section_name,
            "draft_markdown": "",
            "claims": {},
            "deterministic": {},
            "judges": None,
            "approval": {"approved": False, "reason": "not_run"},
            "section_score": {},
            "iteration": 0,
            "previous_issue_signature": None,
            "route": "",
            "decisions": [],
            "result": None,
        }
        final = SECTION_AGENT_GRAPH.invoke(init, config={"recursion_limit": 50})
        result = final.get("result")
        if not result:
            result = {"section_name": section_name, "status": "error",
                      "approval": final.get("approval", {}),
                      "decision_log": final.get("decisions", [])}
        return result

else:
    # Fallback: no langgraph -> reuse the legacy loop unchanged.
    def run_section_pipeline_graph(section_name: str):
        return run_section_pipeline(section_name)

print("run_section_pipeline_graph ready | langgraph:", _LANGGRAPH_AVAILABLE)


In [37]:
# ============================================================
# CELL 19 — RUN ALL SECTIONS
# ============================================================

# To test a single section, set SECTION_TO_RUN in .env, e.g. SECTION_TO_RUN=Governance
SECTION_TO_RUN = os.getenv("SECTION_TO_RUN", "").strip()
sections_to_run = [SECTION_TO_RUN] if SECTION_TO_RUN else SECTIONS

USE_LANGGRAPH = os.getenv("USE_LANGGRAPH", "1").strip().lower() not in ("0", "false", "no", "off")
_section_runner = run_section_pipeline_graph if (USE_LANGGRAPH and _LANGGRAPH_AVAILABLE) else run_section_pipeline
print("Section runner:", "LangGraph agent" if _section_runner is run_section_pipeline_graph else "legacy loop")

section_results = []
for section in sections_to_run:
    if section not in SECTIONS:
        raise ValueError(f"Unknown section: {section}")
    result = _section_runner(section)
    section_results.append(result)

write_json(section_results, OUTPUT_DIR / "section_generation_results.json")
display(pd.DataFrame(section_results))

SECTION: General Requirements
Iteration 0 — building claims register...
Iteration 0 — deterministic gates...
Iteration 0 — deterministic gates failed, skipping LLM judges.
- draft_structural_quality_no_templates_no_absence_language: PASS | failures=0 | warnings=0
- draft_depth_quality_no_truncation: PASS | failures=0 | warnings=0
- draft_prose_polish_no_raw_fields_no_booleans_no_unavailable_language: FAIL | failures=1 | warnings=0
  failure: {'type': 'instruction_like_language_in_report', 'required_fix': 'Rewrite instruction-like statements as neutral disclosure prose.'}
- unsupported_generic_boilerplate_and_inferred_thresholds_gate: PASS | failures=0 | warnings=0
- claims_integrity: PASS | failures=0 | warnings=0
- factlock_numbers_entities: PASS | failures=0 | warnings=5
- reference_firewall: FAIL | failures=1 | warnings=0
  failure: {'type': 'forbidden_reference_term', 'value': 'GRC'}
- report_cleanliness_two_tier_limitation_policy: PASS | failures=0 | warnings=0
- engine_config_dri

,section_name,status,approved_markdown_path,approved_json_path,iterations,approval,section_generation_score
0,General Requirements,approved,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,1,"{'section_name': 'General Requirements', 'appr...","{'section_name': 'General Requirements', 'over..."
1,Governance,approved,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,0,"{'section_name': 'Governance', 'approved': Tru...","{'section_name': 'Governance', 'overall_sectio..."
2,Strategy,approved,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,0,"{'section_name': 'Strategy', 'approved': True,...","{'section_name': 'Strategy', 'overall_section_..."
3,Risk Management,approved,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,0,"{'section_name': 'Risk Management', 'approved'...","{'section_name': 'Risk Management', 'overall_s..."
4,Metrics and Targets,approved,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,0,"{'section_name': 'Metrics and Targets', 'appro...","{'section_name': 'Metrics and Targets', 'overa..."


## Final editorial polish and PDF readiness controls

In [38]:
# ============================================================
# CELL 19C — FINAL EDITORIAL POLISH AND PDF READINESS CONTROLS
# ============================================================
# Production policy:
# - Polishing is generic and evidence-preserving.
# - It removes generated-looking structure, internal process wording,
#   excessive numeric precision, and unsupported sparse disclosure patterns.
# - It never inserts new facts. If the editor cannot support a detail, it
#   removes or generalises the wording rather than inventing a replacement.
# - Approved sections are rechecked after polishing before PDF handoff.
# ============================================================

DIRS.setdefault("final_editorial", OUTPUT_DIR / "15_final_editorial")
DIRS["final_editorial"].mkdir(parents=True, exist_ok=True)

REPORT_ENGINE_CONFIG.setdefault("final_editorial", {})
REPORT_ENGINE_CONFIG["final_editorial"].update({
    "enabled": os.getenv("IFRS_ENABLE_FINAL_EDITORIAL_POLISH", "1").strip().lower() not in {"0", "false", "no", "off"},
    "llm_enabled": os.getenv("IFRS_FINAL_EDITORIAL_LLM", "1").strip().lower() not in {"0", "false", "no", "off"},
    "run_deterministic_gate_validation": os.getenv("IFRS_FINAL_EDITORIAL_RUN_GATES", "1").strip().lower() not in {"0", "false", "no", "off"},
    "max_context_chars": int(os.getenv("IFRS_FINAL_EDITORIAL_CONTEXT_CHARS", "115000")),
    "max_tokens": int(os.getenv("IFRS_FINAL_EDITORIAL_MAX_TOKENS", "12000")),
    "large_number_decimal_places": int(os.getenv("IFRS_LARGE_NUMBER_DECIMAL_PLACES", "1")),
    "standard_decimal_places": int(os.getenv("IFRS_STANDARD_DECIMAL_PLACES", "1")),
    "strict_internal_language": os.getenv("IFRS_FINAL_EDITORIAL_STRICT_INTERNAL_LANGUAGE", "1").strip().lower() not in {"0", "false", "no", "off"},
})

_FINAL_EDITORIAL_INTERNAL_LANGUAGE_PATTERNS = [
    r"\bpre[- ]?computed\b",
    r"\bcorrection was applied\b",
    r"\bdata[- ]?preparation\b",
    r"\bdebug\b",
    r"\bpipeline\b",
    r"\braw field\b",
    r"\bdatabase field\b",
    r"\b[A-Za-z][A-Za-z ]{2,35}\s*:\s*Linked\b",
]

_FINAL_EDITORIAL_FORBIDDEN_REPORT_PATTERNS = [
    r"\bpayload\b",
    r"\bsynthetic\b",
    r"\bhuman review\b",
    r"\baudit-only\b",
    # [Fix 1] "missing data" / "data gap" removed from the unconditional block:
    # they are Tier-2 wording, legitimate inside IFRS-sanctioned limitation
    # statements. Unsanctioned Tier-2 usage is flagged via scan_limitation_language().
    r"\[\s*Section\s*:",
]


def _final_editorial_normalize_heading_text(text: str) -> str:
    text = re.sub(r"^#+\s*", "", str(text or "")).strip()
    text = re.sub(r"\*\*|__|`", "", text).strip()
    text = re.sub(r"[^a-zA-Z0-9]+", " ", text).strip().lower()
    return re.sub(r"\s+", " ", text)


def final_editorial_strip_duplicate_section_heading(section_name: str, markdown: str) -> str:
    """Remove a repeated first heading when the assembler already adds the section title."""
    lines = str(markdown or "").splitlines()
    while lines and not lines[0].strip():
        lines.pop(0)
    if not lines:
        return ""
    first = lines[0].strip()
    if re.match(r"^#{1,6}\s+", first):
        first_norm = _final_editorial_normalize_heading_text(first)
        section_norm = _final_editorial_normalize_heading_text(section_name)
        # Generic duplicate detection: exact section title or title plus a short descriptor.
        if first_norm == section_norm or first_norm.startswith(section_norm + " ") or section_norm.startswith(first_norm + " "):
            lines = lines[1:]
            while lines and not lines[0].strip():
                lines.pop(0)
    return "\n".join(lines).strip() + "\n"


def _final_editorial_format_decimal_token(token: str, decimal_places: int) -> str:
    from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
    original = str(token)
    try:
        compact = original.replace(",", "")
        value = Decimal(compact)
        quant = Decimal("1") if decimal_places <= 0 else Decimal("1").scaleb(-decimal_places)
        rounded = value.quantize(quant, rounding=ROUND_HALF_UP)
    except (InvalidOperation, ValueError):
        return original

    sign = "-" if rounded < 0 else ""
    rounded_abs = abs(rounded)
    fixed = f"{rounded_abs:.{decimal_places}f}"
    int_part, _, frac_part = fixed.partition(".")
    int_with_commas = f"{int(int_part):,}"
    if decimal_places <= 0:
        return sign + int_with_commas
    return sign + int_with_commas + "." + frac_part


def final_editorial_normalize_numeric_precision(markdown: str) -> str:
    """Round excessive decimal precision in prose/tables without changing supported facts materially."""
    cfg = REPORT_ENGINE_CONFIG.get("final_editorial", {})
    large_dp = int(cfg.get("large_number_decimal_places", 1))
    standard_dp = int(cfg.get("standard_decimal_places", 1))

    number_re = re.compile(r"(?<![A-Za-z0-9_])(-?(?:\d{1,3}(?:,\d{3})+|\d+)\.\d{3,})(?![A-Za-z0-9_])")

    def repl(match: re.Match) -> str:
        token = match.group(1)
        int_part = token.split(".", 1)[0].replace(",", "").replace("-", "")
        decimal_places = large_dp if len(int_part) >= 4 else standard_dp
        return _final_editorial_format_decimal_token(token, decimal_places)

    return number_re.sub(repl, str(markdown or ""))


def final_editorial_deterministic_section_polish(section_name: str, markdown: str) -> str:
    text = str(markdown or "")
    if "supported_scope_senior_finalize_prose" in globals():
        text = supported_scope_senior_finalize_prose(text, section_name)
    text = final_editorial_strip_duplicate_section_heading(section_name, text)
    text = final_editorial_normalize_numeric_precision(text)
    text, _table_changes = _final_qa_clean_sparse_markdown_tables(text) if "_final_qa_clean_sparse_markdown_tables" in globals() else (text, [])
    # [Fix 3] Drop any heading left empty by sentence/table removals above.
    if "remove_empty_heading_blocks" in globals():
        text = remove_empty_heading_blocks(text)
    return text.strip() + "\n"


def final_editorial_assemble_preview(sections: Dict[str, str]) -> str:
    lines = ["# IFRS S1/S2 Sustainability-Related Financial Disclosures", ""]
    for idx, section in enumerate(SECTIONS, start=1):
        if section not in sections:
            continue
        lines.append(f"# {idx}. {section}")
        lines.append("")
        lines.append(final_editorial_strip_duplicate_section_heading(section, sections[section]).strip())
        lines.append("")
    return "\n".join(lines).strip() + "\n"


def final_editorial_quality_issues(sections: Dict[str, str]) -> List[Dict[str, Any]]:
    issues: List[Dict[str, Any]] = []
    report = final_editorial_assemble_preview(sections)

    # Generated-looking duplicate headings.
    for section, md in sections.items():
        lines = [line.strip() for line in str(md or "").splitlines() if line.strip()]
        if lines and re.match(r"^#{1,6}\s+", lines[0]):
            first_norm = _final_editorial_normalize_heading_text(lines[0])
            section_norm = _final_editorial_normalize_heading_text(section)
            if first_norm == section_norm or first_norm.startswith(section_norm + " ") or section_norm.startswith(first_norm + " "):
                issues.append({"type": "duplicate_section_heading", "section": section, "text": lines[0]})

    # Excessive numeric precision.
    for match in re.finditer(r"(?<![A-Za-z0-9_])(-?(?:\d{1,3}(?:,\d{3})+|\d+)\.\d{3,})(?![A-Za-z0-9_])", report):
        issues.append({"type": "excessive_numeric_precision", "value": match.group(1)})
        if len([i for i in issues if i.get("type") == "excessive_numeric_precision"]) >= 25:
            break

    # Internal/process language.
    for pattern in _FINAL_EDITORIAL_INTERNAL_LANGUAGE_PATTERNS:
        for match in re.finditer(pattern, report, flags=re.I):
            issues.append({"type": "internal_process_language", "pattern": pattern, "text": match.group(0)})
            break

    # Forbidden report language.
    for pattern in _FINAL_EDITORIAL_FORBIDDEN_REPORT_PATTERNS:
        if re.search(pattern, report, flags=re.I):
            issues.append({"type": "forbidden_report_language", "pattern": pattern})

    # [Fix 1] Tier-aware limitation-language scan on the assembled report.
    if "scan_limitation_language" in globals():
        scan = scan_limitation_language(report)
        for hit in scan.get("tier1_hits", []):
            issues.append({"type": "tier1_pipeline_internal_language", "phrase": hit})
        for item in scan.get("unsanctioned_tier2", []):
            issues.append({"type": "unsanctioned_limitation_language", **item})

    # Missing-looking cells remaining after table hygiene.
    if "_final_qa_table_missing_locations" in globals():
        missing_locations = _final_qa_table_missing_locations(sections)
        for loc in missing_locations[:50]:
            issues.append({"type": "missing_looking_table_cell", **loc})

    return issues


def final_editorial_prompt_context(sections: Dict[str, str], issues: List[Dict[str, Any]]) -> Dict[str, Any]:
    evidence_facts = _final_qa_collect_evidence_facts() if "_final_qa_collect_evidence_facts" in globals() else []
    numeric_claims = _final_qa_numeric_claims_from_sections(sections) if "_final_qa_numeric_claims_from_sections" in globals() else []
    return {
        "sections": sections,
        "quality_issues": issues,
        "evidence_facts": evidence_facts[:REPORT_ENGINE_CONFIG.get("final_quality", {}).get("max_evidence_facts", 1600)],
        "numeric_claims": numeric_claims[:REPORT_ENGINE_CONFIG.get("final_quality", {}).get("max_numeric_claims", 400)],
        "editorial_policy": {
            "no_new_facts": True,
            "preserve_numbers_or_round_supported_values_only": True,
            "remove_duplicate_headings": True,
            "remove_internal_process_wording": True,
            "remove_missing_looking_table_cells": True,
            "do_not_mention_payload_or_synthetic_data": True,
        },
    }


def final_editorial_llm_polish_sections(sections: Dict[str, str], issues: List[Dict[str, Any]]) -> Dict[str, Any]:
    context = final_editorial_prompt_context(sections, issues)
    system = (
        "You are a senior IFRS S1/S2 sustainability-report editor. "
        "You perform final editorial polish on already-approved Markdown sections. "
        "You must preserve evidence-supported facts and return JSON only."
    )
    user = f"""
Polish the approved sections for final PDF-ready reporting quality.

Return JSON with exactly these top-level keys:
- revised_sections: object mapping each section name to complete revised Markdown
- changes_made: array
- unresolved_issues: array

Rules:
1. Do not invent new facts, values, dates, metrics, commitments, assurance statements or methodologies.
2. Use only the existing section text and evidence_facts. If a detail is unsupported or editorially risky, remove or generalise it.
3. Remove repeated section headings because the final assembler adds numbered section titles.
4. Remove internal/process wording, including correction/debug/data-preparation wording, database-like association labels, and generated-looking explanations.
5. Round excessive decimal precision consistently where the rounded value remains faithful to the supported number.
6. Remove missing-looking table cells and restructure sparse tables into prose where needed.
7. Improve report flow and IFRS style, but do not materially change the meaning.
8. Never mention pipeline internals (payloads, synthetic data, audit files, human review, placeholders). Preserve IFRS-required limitation disclosures (relief invocation, Scope 3 category basis, estimation/measurement uncertainty, assurance level) where evidence supports them.
9. Return complete Markdown for every section in revised_sections, not diffs.

Context:
{truncate_context(context, max_chars=REPORT_ENGINE_CONFIG.get('final_editorial', {}).get('max_context_chars', 115000))}
""".strip()
    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG.get("minimal_reviser", "strong"),
        temperature=0,
        max_tokens=REPORT_ENGINE_CONFIG.get("final_editorial", {}).get("max_tokens", 12000),
        response_format={"type": "json_object"},
    )
    result = parse_json_response(raw, request_label="final_editorial_polish")
    if not isinstance(result, dict):
        raise ValueError("Final editorial polish returned non-object JSON")
    return result


def final_editorial_validate_section(section_name: str, markdown: str) -> Dict[str, Any]:
    if not REPORT_ENGINE_CONFIG.get("final_editorial", {}).get("run_deterministic_gate_validation", True):
        return {"section_name": section_name, "passed": True, "skipped": True}
    try:
        claims = repair_claim_evidence_sources(section_name, build_claims_register(section_name, markdown))
        deterministic = run_deterministic_gates(section_name, markdown, claims)
        return {
            "section_name": section_name,
            "passed": bool(deterministic.get("passed")),
            "deterministic": deterministic,
        }
    except Exception as exc:
        return {"section_name": section_name, "passed": False, "error": repr(exc)}


def final_editorial_write_sections(sections: Dict[str, str], result: Dict[str, Any]) -> None:
    for section, md in sections.items():
        if section not in SECTIONS:
            continue
        slug = SECTION_SLUGS[section]
        approved_path = DIRS["approved"] / f"approved_{slug}.md"
        if approved_path.exists():
            backup_path = DIRS["final_editorial"] / f"approved_{slug}.pre_editorial.md"
            if not backup_path.exists():
                write_text(read_text(approved_path), backup_path)
        write_text(md.strip() + "\n", approved_path)

        approved_json_path = DIRS["approved"] / f"approved_{slug}.json"
        approved_json = read_json(approved_json_path, default={}) if approved_json_path.exists() else {}
        approved_json.setdefault("final_editorial", {})
        approved_json["final_editorial"].update({
            "polished": True,
            "audit_path": str(DIRS["final_editorial"] / "final_editorial_polish_result.json"),
        })
        write_json(approved_json, approved_json_path)


def run_final_editorial_polish(force: bool = True) -> Dict[str, Any]:
    cfg = REPORT_ENGINE_CONFIG.get("final_editorial", {})
    marker = DIRS["final_editorial"] / "final_editorial_polish_result.json"
    if marker.exists() and not force:
        existing = read_json(marker, default={})
        if existing.get("approved") is True:
            return existing

    if not cfg.get("enabled", True):
        result = {"approved": True, "skipped": True, "reason": "final_editorial_disabled"}
        write_json(result, marker)
        return result

    sections = _final_qa_load_approved_sections() if "_final_qa_load_approved_sections" in globals() else {}
    if not sections:
        result = {"approved": False, "reason": "no_approved_sections_found"}
        write_json(result, marker)
        return result

    deterministic_sections = {
        section: final_editorial_deterministic_section_polish(section, md)
        for section, md in sections.items()
        if section in SECTIONS
    }
    initial_issues = final_editorial_quality_issues(deterministic_sections)

    llm_result = None
    llm_error = None
    candidate_sections = dict(deterministic_sections)
    if cfg.get("llm_enabled", True):
        try:
            llm_result = final_editorial_llm_polish_sections(deterministic_sections, initial_issues)
            revised = llm_result.get("revised_sections", {}) if isinstance(llm_result, dict) else {}
            if isinstance(revised, dict):
                for section in SECTIONS:
                    if section in revised and str(revised[section] or "").strip():
                        candidate_sections[section] = str(revised[section]).strip() + "\n"
        except Exception as exc:
            llm_error = repr(exc)

    polished_sections = {
        section: final_editorial_deterministic_section_polish(section, md)
        for section, md in candidate_sections.items()
        if section in SECTIONS
    }

    final_issues = final_editorial_quality_issues(polished_sections)
    gate_results = [final_editorial_validate_section(section, polished_sections[section]) for section in SECTIONS if section in polished_sections]
    gate_failures = [g for g in gate_results if not g.get("passed")]
    missing_sections = [section for section in SECTIONS if section not in polished_sections or not polished_sections[section].strip()]

    approved = (not final_issues) and (not gate_failures) and (not missing_sections)
    if approved:
        final_editorial_write_sections(polished_sections, {})

    result = {
        "approved": approved,
        "initial_issue_count": len(initial_issues),
        "final_issue_count": len(final_issues),
        "initial_issues": initial_issues[:100],
        "final_issues": final_issues[:100],
        "llm_enabled": cfg.get("llm_enabled", True),
        "llm_error": llm_error,
        "llm_changes_made": llm_result.get("changes_made", []) if isinstance(llm_result, dict) else [],
        "llm_unresolved_issues": llm_result.get("unresolved_issues", []) if isinstance(llm_result, dict) else [],
        "gate_results": gate_results,
        "gate_failure_count": len(gate_failures),
        "missing_sections": missing_sections,
        "output_dir": str(DIRS["final_editorial"]),
        "policy": "Final editorial polish is generic, evidence-preserving and validated before PDF handoff.",
    }
    write_json(result, marker)
    if not approved:
        # Keep current approved sections unchanged when polishing cannot be safely validated.
        write_text(final_editorial_assemble_preview(deterministic_sections), DIRS["final_editorial"] / "deterministic_editorial_preview.md")
    else:
        write_text(final_editorial_assemble_preview(polished_sections), DIRS["final_editorial"] / "editorial_polished_report_preview.md")
    return result


print("Final editorial polish controls loaded.")


Final editorial polish controls loaded.


## Run final editorial polish

In [39]:
# ============================================================
# CELL 19C-RUN — RUN FINAL EDITORIAL POLISH
# ============================================================

final_editorial_result = run_final_editorial_polish(force=True)
write_json(final_editorial_result, DIRS["final_editorial"] / "final_editorial_polish_result.json")
print("Final editorial polish approved:", final_editorial_result.get("approved"))
print("Final editorial polish audit:", DIRS["final_editorial"] / "final_editorial_polish_result.json")
try:
    display(pd.DataFrame(final_editorial_result.get("final_issues", [])))
except Exception:
    pass


Azure strong agent: connection reset/timeout (TimeoutError); retrying attempt 2/6 in 1.4s...
Azure strong agent: connection issue; retrying attempt 3/6 in 2.6s...
Final editorial polish approved: True
Final editorial polish audit: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\15_final_editorial\final_editorial_polish_result.json


""


## Whole-report connectivity judge

Run this after all sections are approved. It checks consistency across sections before PDF assembly.

In [40]:
# ============================================================
# CELL 20 — WHOLE-REPORT CONNECTIVITY JUDGE
# ============================================================


def load_approved_sections() -> Dict[str, str]:
    approved = {}
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        path = DIRS["approved"] / f"approved_{slug}.md"
        if path.exists():
            approved[section] = read_text(path)
    return approved


# The whole-report connectivity judge is an OPTIONAL LLM advisory. It is the only
# component that reasons about SEMANTIC cross-section consistency (numeric
# cross-section consistency is already covered deterministically by
# report_numeric_consistency_issues in the evaluation cell). It is unvalidated
# (LLM grading LLM output, no ground truth) and its output is advisory only —
# nothing downstream consumes it to gate or edit the report. It is therefore OFF
# by default; set IFRS_RUN_CONNECTIVITY_JUDGE=1 to run it as a qualitative second
# opinion. See the seeded-contradiction harness before relying on its verdicts.
def connectivity_judge_enabled() -> bool:
    return os.getenv("IFRS_RUN_CONNECTIVITY_JUDGE", "0").strip().lower() in {"1", "true", "yes", "on"}


def run_connectivity_judge() -> Dict[str, Any]:
    if not connectivity_judge_enabled():
        result = {
            "approved": None,
            "status": "skipped_disabled",
            "reason": "Connectivity judge is an optional advisory and is off by default. "
                      "Set IFRS_RUN_CONNECTIVITY_JUDGE=1 to enable it.",
        }
        write_json(result, DIRS["connectivity"] / "connectivity_judge_result.json")
        return result

    approved_sections = load_approved_sections()
    if len(approved_sections) < 2:
        result = {
            "approved": False,
            "reason": "Not enough approved sections to run connectivity judge.",
            "approved_section_count": len(approved_sections),
        }
        write_json(result, DIRS["connectivity"] / "connectivity_judge_result.json")
        return result

    context = {
        "approved_sections": approved_sections,
        "checks": [
            "Terminology consistency across sections.",
            "Time horizon consistency across Strategy and Risk Management.",
            "Targets in Strategy must not contradict Metrics and Targets.",
            "Governance oversight described in Governance must align with Strategy/Risk Management references.",
            "No duplicated or contradictory claims.",
            "No pipeline-internal wording (payload/synthetic/audit-only/placeholder) in report prose. "
            "IFRS-sanctioned limitation statements (relief invocation, Scope 3 category basis, "
            "estimation/measurement uncertainty, assurance level) are expected and must be consistent across sections.",
        ],
    }
    system = "You are a whole-report IFRS S1/S2 connectivity judge. Return JSON only."
    user = f"""
Review the approved sections for cross-section consistency.

Return JSON with:
- approved
- connectivity_score_0_to_10
- contradictions
- terminology_issues
- target_metric_mismatches
- required_fixes
- summary

Context:
{truncate_context(context, max_chars=90000)}
""".strip()

    raw = azure_chat(
        [{"role": "system", "content": system}, {"role": "user", "content": user}],
        model_tier=MODEL_CONFIG["whole_report_connectivity_judge"],
        temperature=0,
        max_tokens=5000,
        response_format={"type": "json_object"},
    )
    result = parse_json_response(raw, request_label="whole_report_connectivity_judge")
    write_json(result, DIRS["connectivity"] / "connectivity_judge_result.json")
    return result

connectivity_result = run_connectivity_judge()
if connectivity_result.get("status") == "skipped_disabled":
    print("Connectivity judge skipped (optional advisory, off by default). "
          "Set IFRS_RUN_CONNECTIVITY_JUDGE=1 to enable.")
else:
    display(pd.DataFrame([connectivity_result]))

Connectivity judge skipped (optional advisory, off by default). Set IFRS_RUN_CONNECTIVITY_JUDGE=1 to enable.


## Final Markdown and PDF handoff package

This notebook does not use the PDF layout guide for drafting. It creates an approved Markdown report and a handoff manifest for the separate PDF assembly stage.

In [41]:
# ============================================================
# CELL 21 — BUILD FINAL MARKDOWN + PDF HANDOFF MANIFEST
# PRODUCTION RULE: final report cannot contain missing-data/audit wording.
# ============================================================


def assert_no_missing_data_language_in_report(markdown: str) -> None:
    hits = scan_for_missing_data_language(markdown) if "scan_for_missing_data_language" in globals() else []
    if hits:
        audit = {
            "approved": False,
            "reason": "final_report_contains_missing_data_language",
            "hits": hits,
            "policy": "Missing requirements and missing data may appear only in audit outputs, never in approved report prose.",
        }
        write_json(audit, DIRS["handoff"] / "final_report_cleanliness_failure.json")
        raise ValueError(
            "Final report blocked: missing-data/audit wording found in approved prose. "
            f"See {DIRS['handoff'] / 'final_report_cleanliness_failure.json'}"
        )


def assemble_final_markdown() -> Tuple[str, Path]:
    approved_sections = load_approved_sections()
    lines = []
    lines.append("# IFRS S1/S2 Sustainability-Related Financial Disclosures")
    lines.append("")

    for idx, section in enumerate(SECTIONS, start=1):
        if section not in approved_sections:
            continue
        lines.append(f"# {idx}. {section}")
        lines.append("")
        section_markdown = approved_sections[section]
        if "final_editorial_strip_duplicate_section_heading" in globals():
            section_markdown = final_editorial_strip_duplicate_section_heading(section, section_markdown)
        # [Fix 3] Remove any heading left empty by upstream sentence/table removals.
        if "remove_empty_heading_blocks" in globals():
            section_markdown = remove_empty_heading_blocks(section_markdown)
        lines.append(section_markdown.strip())
        lines.append("")

    final_md = "\n".join(lines).strip() + "\n"
    if "final_editorial_normalize_numeric_precision" in globals():
        final_md = final_editorial_normalize_numeric_precision(final_md)
    # [Fix 3] Final safety net at the document level.
    if "remove_empty_heading_blocks" in globals():
        final_md = remove_empty_heading_blocks(final_md)

    assert_no_missing_data_language_in_report(final_md)

    # [Fix 2] Report-level numeric coherence audit: precision drift + labelled-metric
    # conflicts across the assembled sections, plus a re-run of the payload-level
    # arithmetic checks. Written to audit_logs; only raises when IFRS_ARITHMETIC_STRICT=1.
    try:
        report_numeric_issues = (
            report_numeric_consistency_issues(approved_sections)
            if "report_numeric_consistency_issues" in globals() else []
        )
        payload_issues = payload_arithmetic_issues(None) if "payload_arithmetic_issues" in globals() else []
        arithmetic_audit = {
            "report_numeric_consistency_issues": report_numeric_issues,
            "payload_arithmetic_issues": payload_issues,
            "total": len(report_numeric_issues) + len(payload_issues),
            "strict_mode": bool(globals().get("IFRS_ARITHMETIC_STRICT", False)),
        }
        write_json(arithmetic_audit, DIRS["audit_logs"] / "final_report_arithmetic_audit.json")
        print(f"Final arithmetic audit: {arithmetic_audit['total']} issue(s). "
              f"See {DIRS['audit_logs'] / 'final_report_arithmetic_audit.json'}")
        if arithmetic_audit["total"] and globals().get("IFRS_ARITHMETIC_STRICT", False):
            raise ValueError(
                f"Final report blocked: {arithmetic_audit['total']} arithmetic/coherence issue(s) "
                f"with IFRS_ARITHMETIC_STRICT=1. See "
                f"{DIRS['audit_logs'] / 'final_report_arithmetic_audit.json'}"
            )
    except NameError:
        pass

    path = DIRS["handoff"] / "approved_report_markdown.md"
    write_text(final_md, path)
    return final_md, path

final_markdown, final_markdown_path = assemble_final_markdown()

handoff_manifest = {
    "pipeline_mode": PIPELINE_MODE,
    "approved_report_markdown": str(final_markdown_path),
    "approved_sections_dir": str(DIRS["approved"]),
    "coverage_dir": str(DIRS["coverage"]),
    "missing_requirements_dir": str(DIRS["missing_requirements"]),
    "claims_registers_dir": str(DIRS["claims"]),
    "connectivity_judge_result": str(DIRS["connectivity"] / "connectivity_judge_result.json"),
    "rendering_layout_guide": str(RENDERING_DIR / "layout_style_guide.json"),
    "section_generation_results": str(OUTPUT_DIR / "section_generation_results.json"),
    "important_rule": "The PDF assembly stage may use layout_style_guide.json. Drafting agents must not use it. Missing requirements/data are audit-only and must not be printed in the report.",
}
write_json(handoff_manifest, DIRS["handoff"] / "pdf_handoff_manifest.json")

print("Final Markdown:", final_markdown_path)
print("PDF handoff manifest:", DIRS["handoff"] / "pdf_handoff_manifest.json")


Final arithmetic audit: 3 issue(s). See C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\audit_logs\final_report_arithmetic_audit.json
Final Markdown: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\12_pdf_handoff\approved_report_markdown.md
PDF handoff manifest: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\12_pdf_handoff\pdf_handoff_manifest.json


## Scoring schemes: why coverage appears in two places

This pipeline computes coverage under **two intentionally separate** weighting schemes. They answer different questions and neither feeds the other, so this is separation of concerns, not double-counting toward a single decision.

**1. Per-section generation score** (`score_section_generation_output`, defined earlier): `coverage 0.75 / judge 0.25`. This is a **reporting-only quality-ranking signal** emitted for each section during the generation loop. It gates nothing — approval is decided strictly and separately by `composite_approval_gate`. Its purpose is drafting feedback: how well did *this* section cover *its* assigned requirements.

**2. Report-level Quality Index** (the scorecard below): a 5-dimension composite (faithfulness 0.30, compliance 0.25, coherence 0.20, coverage 0.15, non-redundancy 0.10). Here coverage is one of five *whole-report* quality dimensions, weighted 0.15.

The two schemes measure overlapping concepts (both look at coverage) but at different granularities and for different consumers — the section loop vs. the final deliverable scorecard. They are deliberately not unified into one number: collapsing them would conflate per-section drafting feedback with the report-level quality index, and would let one coverage view influence the other. Both are tunable via env knobs (`IFRS_SCORE_JUDGE_WEIGHT` / `IFRS_SCORE_COVERAGE_WEIGHT` for the section score; `IFRS_EVAL_W_*` for the report-level dimensions).

In [42]:
# ============================================================
# CELL 23 — EVALUATION METRICS & SCORECARD
# ============================================================
# Why this cell exists:
#   The pipeline produces a lot of signal (coverage, gates, judges, fact-lock,
#   the two-tier limitation scan, the arithmetic validator) but never turned it
#   into reportable numbers. This cell aggregates that signal into a single,
#   thesis-ready evaluation across five deterministic dimensions plus efficiency,
#   then rolls them into a 0-100 report quality index. It is READ-ONLY: it does
#   not modify the report, and (except for the optional gold-set section) it runs
#   no additional LLM calls, so it is safe to run after every generation.
#
#   Dimensions (all in [0, 1], higher = better; displayed x100):
#     faithfulness  — numeric/entity grounding + claim integrity (fact-lock)
#     compliance    — limitation-policy cleanliness + required-limitation recall
#     coherence     — payload arithmetic + cross-section numeric consistency
#     coverage      — requirement + mandatory-requirement coverage
#     non_redundancy— 1 - inter-section duplication
#   Efficiency (cost/latency/iterations) is reported alongside, not folded into
#   the quality index, because "cheaper" is not "better quality".
# ============================================================

# Weights for the composite index (override via env if desired).
_EVAL_DIMENSION_WEIGHTS = {
    "faithfulness": float(os.getenv("IFRS_EVAL_W_FAITHFULNESS", "0.30")),
    "compliance": float(os.getenv("IFRS_EVAL_W_COMPLIANCE", "0.25")),
    "coherence": float(os.getenv("IFRS_EVAL_W_COHERENCE", "0.20")),
    "coverage": float(os.getenv("IFRS_EVAL_W_COVERAGE", "0.15")),
    "non_redundancy": float(os.getenv("IFRS_EVAL_W_NON_REDUNDANCY", "0.10")),
}


def _eval_load_section_metrics() -> Dict[str, Dict[str, Any]]:
    """Read the per-section metrics persisted by run_section_pipeline."""
    out = {}
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        path = DIRS["approved"] / f"approved_{slug}.json"
        if path.exists():
            try:
                obj = json.loads(read_text(path))
                out[section] = obj.get("section_metrics", {}) or {}
            except Exception:
                out[section] = {}
    return out


def _eval_faithfulness(section_metrics: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    grounding, claim_fail, checked = [], 0, 0
    ent_unsupported = 0
    for m in section_metrics.values():
        if m.get("numbers_checked"):
            grounding.append(m.get("numeric_grounding_rate", 1.0))
            checked += m.get("numbers_checked", 0)
        claim_fail += m.get("claim_integrity_failures", 0)
        ent_unsupported += m.get("unsupported_entities", 0)
    # Token-weighted mean grounding is fairer than a plain section mean.
    numeric_grounding = round(sum(grounding) / len(grounding), 4) if grounding else 1.0
    # Claim penalty: normalise failures against sections (soft).
    claim_score = round(1.0 / (1.0 + claim_fail), 4)
    score = round(0.7 * numeric_grounding + 0.3 * claim_score, 4)
    return {
        "score": score,
        "numeric_grounding_rate": numeric_grounding,
        "numbers_checked": checked,
        "claim_integrity_failures": claim_fail,
        "unsupported_entities": ent_unsupported,
    }


# Which payload evidence signals make each IFRS limitation-disclosure family
# APPLICABLE, and in WHICH section(s) that disclosure belongs. Recall is measured
# per (section, family) pair only where the family is home to that section, so a
# Scope 3 basis is expected in Metrics, not in Governance. This keeps applicability
# precise instead of firing in every section that merely carries proxy data.
_LIMITATION_FAMILY_PAYLOAD_TRIGGERS = {
    "relief_financial_effects": ("relief", "financial_effect", "anticipated_effect",
                                 "not_quantified", "undue_cost", "impracticable"),
    "scope3_category_basis": ("scope3", "scope_3", "financed_emissions", "category",
                              "excluded_categor", "included_categor"),
    "estimation_uncertainty": ("data_gap", "data_quality", "pcaf", "proxy",
                               "estimation", "uncertainty", "assumption"),
    "assurance": ("assurance", "assured", "verification", "verified_by"),
}

# Home sections per family (a disclosure is only "required" where it belongs).
_LIMITATION_FAMILY_HOME_SECTIONS = {
    "relief_financial_effects": {"Strategy"},
    "scope3_category_basis": {"Metrics and Targets"},
    "estimation_uncertainty": {"Metrics and Targets", "General Requirements"},
    "assurance": {"General Requirements"},
}

# Families that are legitimately disclosed ONCE at report level (typically in the
# basis-of-preparation / General Requirements section) rather than repeated in
# every applicable section. For these, a single disclosure anywhere in the report
# satisfies the requirement, so we credit them report-wide instead of per-section.
_REPORT_LEVEL_LIMITATION_FAMILIES = {"assurance"}


def _relief_is_applicable(payloads: Dict[str, Any]) -> bool:
    """Relief (IFRS S2 19-21) applies only when financial effects CANNOT be
    quantified. If the payload carries quantitative effect values, the bank must
    disclose them instead, so relief is NOT the required disclosure and must not
    count against recall.
    """
    if "flatten_json" not in globals():
        return False
    has_effect_fields = False
    has_quantified = False
    for sec_payload in payloads.values():
        for k, v in flatten_json(sec_payload).items():
            kl = str(k).lower()
            if "financial_effect" in kl or "climate_financial_effects" in kl or "relief" in kl:
                has_effect_fields = True
            if "quantitative_effect" in kl:
                try:
                    if v is not None and float(v) != 0.0:
                        has_quantified = True
                except (TypeError, ValueError):
                    pass
    # Applicable only when there are effect fields but NO quantified values, or an
    # explicit relief/undue-cost/impracticable flag is present.
    return has_effect_fields and not has_quantified


def _eval_required_limitation_recall() -> Dict[str, Any]:
    """Per-family recall of IFRS-sanctioned limitation disclosures.

    For each section, the payload determines which limitation-disclosure families
    are APPLICABLE (e.g. proxy/PCAF evidence -> estimation_uncertainty is required).
    The section text is then checked for a matching sanctioned disclosure via
    sanctioned_disclosure_families(). This is the direct measure that Fix 1 works:
    the old banned-everything pipeline scored ~0 here because the writer was
    forbidden from producing these required statements.
    """
    if "sanctioned_disclosure_families" not in globals():
        return {"required_limitation_recall": 1.0, "applicable_pairs": 0,
                "satisfied_pairs": 0, "note": "scanner not loaded"}

    applicable_pairs, satisfied_pairs = 0, 0
    details = []
    report_all = "\n".join(_EVAL_APPROVED.values())
    report_families = sanctioned_disclosure_families(report_all)

    # Whole-payload key blob (for report-level applicability).
    all_keys_blob = " ".join(
        str(k).lower()
        for sec in SECTIONS
        for k in (flatten_json(payloads_by_section.get(sec, {})) if "flatten_json" in globals() else {})
    )

    handled_report_level = set()
    for section in SECTIONS:
        payload = payloads_by_section.get(section, {})
        flat = flatten_json(payload) if "flatten_json" in globals() else {}
        keys_blob = " ".join(str(k).lower() for k in flat)
        sec_text = _EVAL_APPROVED.get(section, "")
        present_families = sanctioned_disclosure_families(sec_text)

        for family, triggers in _LIMITATION_FAMILY_PAYLOAD_TRIGGERS.items():
            # Report-level families are evaluated once, against the whole report.
            if family in _REPORT_LEVEL_LIMITATION_FAMILIES:
                if family in handled_report_level:
                    continue
                if not any(t in all_keys_blob for t in triggers):
                    continue  # not applicable anywhere
                handled_report_level.add(family)
                applicable_pairs += 1
                ok = family in report_families
                satisfied_pairs += 1 if ok else 0
                details.append({"scope": "report", "family": family, "satisfied": ok})
                continue

            home = _LIMITATION_FAMILY_HOME_SECTIONS.get(family, set(SECTIONS))
            if section not in home:
                continue  # this disclosure does not belong in this section
            if family == "relief_financial_effects":
                # Relief only applies when effects are NOT quantified anywhere.
                if not _relief_is_applicable(payloads_by_section):
                    continue
            elif not any(t in keys_blob for t in triggers):
                continue  # payload does not make the family applicable here
            applicable_pairs += 1
            ok = family in present_families
            satisfied_pairs += 1 if ok else 0
            details.append({"section": section, "family": family, "satisfied": ok})

    recall = round(satisfied_pairs / applicable_pairs, 4) if applicable_pairs else 1.0
    return {
        "required_limitation_recall": recall,
        "applicable_pairs": applicable_pairs,
        "satisfied_pairs": satisfied_pairs,
        "report_disclosure_families": sorted(report_families),
        "detail": details,
    }


def _eval_compliance(section_metrics: Dict[str, Dict[str, Any]]) -> Dict[str, Any]:
    total_violations = sum(m.get("limitation_policy_violations", 0) for m in section_metrics.values())
    total_words = sum(m.get("word_count", 0) for m in section_metrics.values())
    violations_per_1k = round(1000.0 * total_violations / total_words, 3) if total_words else 0.0
    # Cleanliness score: 1.0 at zero violations, decaying with density.
    cleanliness = round(1.0 / (1.0 + violations_per_1k), 4)
    recall = _eval_required_limitation_recall()
    score = round(0.5 * cleanliness + 0.5 * recall["required_limitation_recall"], 4)
    return {
        "score": score,
        "limitation_violations": total_violations,
        "violations_per_1k_words": violations_per_1k,
        "cleanliness_score": cleanliness,
        **recall,
    }


def _eval_coherence() -> Dict[str, Any]:
    payload_issues = payload_arithmetic_issues(None) if "payload_arithmetic_issues" in globals() else []
    report_issues = report_numeric_consistency_issues(_EVAL_APPROVED) if "report_numeric_consistency_issues" in globals() else []

    # Rate-based normalisation against the number of CHECKABLE RELATIONSHIPS, not
    # the raw count of numeric fields. Counting every field (thousands, incl. array
    # elements) made the denominator huge and the score generous by construction
    # (0.996 almost regardless of issues). A "checkable relationship" is an actual
    # opportunity for one of the arithmetic rules to fire: an intensity, a baseline,
    # a percentage-of-headcount pair, a distribution group, a progress value. We
    # approximate it as the count of fields the rules key on.
    _RELATIONSHIP_TOKENS = ("intensity", "baseline", "financed", "scope1", "scope_1",
                            "scope2", "scope_2", "pct", "percent", "expertise",
                            "board_size", "directors", "progress", "distribution", "mix")
    checkable_relationships = 0
    if "payloads_by_section" in globals() and "_arith_flat_numeric_index" in globals():
        seen_rel = set()
        for sec in SECTIONS:
            if sec not in payloads_by_section:
                continue
            for path in _arith_flat_numeric_index(payloads_by_section[sec]):
                pl = str(path).lower()
                if any(tok in pl for tok in _RELATIONSHIP_TOKENS):
                    # Dedup the shared governance/targets objects across sections.
                    seen_rel.add(re.sub(r"\[\d+\]", "[]", pl))
        checkable_relationships = len(seen_rel)
    payload_denom = max(checkable_relationships, 1)
    arithmetic_score = round(1.0 - min(len(payload_issues) / payload_denom, 1.0), 4)

    distinct_numbers = len(set(re.findall(r"-?\d[\d,]*\.?\d*", "\n".join(_EVAL_APPROVED.values()))))
    report_denom = max(distinct_numbers, 1)
    consistency_score = round(1.0 - min(len(report_issues) / report_denom, 1.0), 4)

    score = round(0.5 * arithmetic_score + 0.5 * consistency_score, 4)
    return {
        "score": score,
        "payload_arithmetic_issues": len(payload_issues),
        "payload_checkable_relationships": checkable_relationships,
        "report_numeric_consistency_issues": len(report_issues),
        "arithmetic_score": arithmetic_score,
        "cross_section_consistency_score": consistency_score,
    }


def _eval_coverage() -> Dict[str, Any]:
    total, covered, partial = 0, 0, 0
    m_total, m_covered = 0, 0
    for section in SECTIONS:
        for rec in coverage_by_section.get(section, []) if "coverage_by_section" in globals() else []:
            total += 1
            status = rec.get("coverage_status")
            is_mand = bool(rec.get("mandatory", False))
            if status == "covered":
                covered += 1
            elif status == "partially_covered":
                partial += 1
            if is_mand:
                m_total += 1
                if status == "covered":
                    m_covered += 1
    # Lenient rate: a partially covered requirement earns half credit.
    coverage_rate = round((covered + 0.5 * partial) / total, 4) if total else 0.0
    # Strict rate: full coverage only, no partial credit.
    strict_full_rate = round(covered / total, 4) if total else 0.0
    mandatory_rate = round(m_covered / m_total, 4) if m_total else 1.0
    # Does the requirement set actually distinguish a mandatory subset?
    # In the current synthetic data every requirement defaults to mandatory
    # (mandatory missing -> True in load), so m_total == total and a "mandatory
    # coverage" penalty is vacuous -- it is arithmetically identical to the strict
    # full-coverage rate. Applying it under a mandatory label would misrepresent
    # what the number means. So: use a genuine mandatory penalty ONLY when a real
    # subset exists; otherwise blend lenient vs strict full coverage, which still
    # penalises partial coverage without inventing a distinction absent from the data.
    has_mandatory_subset = 0 < m_total < total
    strict_component = mandatory_rate if has_mandatory_subset else strict_full_rate
    score = round(0.4 * coverage_rate + 0.6 * strict_component, 4)
    return {
        "score": score,
        "requirements_total": total,
        "coverage_rate": coverage_rate,
        "strict_full_coverage_rate": strict_full_rate,
        "mandatory_subset_distinguished": has_mandatory_subset,
        "mandatory_requirements_total": m_total,
        "mandatory_coverage_rate": mandatory_rate,
    }


def _eval_shingles(text: str, n: int = 5) -> set:
    tokens = re.findall(r"\b\w+\b", str(text or "").lower())
    return set(tuple(tokens[i:i + n]) for i in range(max(0, len(tokens) - n + 1)))


def _eval_non_redundancy() -> Dict[str, Any]:
    names = [s for s in SECTIONS if s in _EVAL_APPROVED]
    shingles = {s: _eval_shingles(_EVAL_APPROVED[s]) for s in names}
    worst = 0.0
    worst_pair = None
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = shingles[names[i]], shingles[names[j]]
            if not a or not b:
                continue
            jac = len(a & b) / len(a | b)
            if jac > worst:
                worst, worst_pair = jac, (names[i], names[j])
    score = round(1.0 - worst, 4)
    return {
        "score": score,
        "max_inter_section_jaccard": round(worst, 4),
        "most_similar_pair": worst_pair,
    }


def _eval_efficiency() -> Dict[str, Any]:
    telemetry = LLM_TELEMETRY.summary() if "LLM_TELEMETRY" in globals() else {"llm_calls": 0}
    # First-pass yield + iterations from the section results, if available.
    iterations, first_pass = [], 0
    for section in SECTIONS:
        slug = SECTION_SLUGS[section]
        path = DIRS["approved"] / f"approved_{slug}.json"
        if not path.exists():
            continue
        try:
            obj = json.loads(read_text(path))
        except Exception:
            continue
        det = obj.get("deterministic", {})
        if det.get("passed"):
            first_pass += 1
    approved_n = sum(1 for s in SECTIONS if (DIRS["approved"] / f"approved_{SECTION_SLUGS[s]}.json").exists())
    return {
        **telemetry,
        "sections_approved": approved_n,
        "first_pass_clean_sections": first_pass,
        "first_pass_yield": round(first_pass / approved_n, 4) if approved_n else 0.0,
    }


def run_evaluation() -> Dict[str, Any]:
    global _EVAL_APPROVED
    _EVAL_APPROVED = load_approved_sections()
    if not _EVAL_APPROVED:
        print("Evaluation skipped: no approved sections found.")
        return {"status": "no_sections"}

    section_metrics = _eval_load_section_metrics()

    dimensions = {
        "faithfulness": _eval_faithfulness(section_metrics),
        "compliance": _eval_compliance(section_metrics),
        "coherence": _eval_coherence(),
        "coverage": _eval_coverage(),
        "non_redundancy": _eval_non_redundancy(),
    }
    weight_sum = sum(_EVAL_DIMENSION_WEIGHTS.values()) or 1.0
    composite = sum(
        _EVAL_DIMENSION_WEIGHTS.get(dim, 0.0) * dimensions[dim]["score"]
        for dim in dimensions
    ) / weight_sum
    report_quality_index = round(100.0 * composite, 2)

    evaluation = {
        "pipeline_mode": PIPELINE_MODE if "PIPELINE_MODE" in globals() else None,
        "report_quality_index_0_100": report_quality_index,
        "dimension_scores_0_100": {d: round(100.0 * v["score"], 2) for d, v in dimensions.items()},
        "dimension_weights": _EVAL_DIMENSION_WEIGHTS,
        "dimensions": dimensions,
        "per_section_metrics": section_metrics,
        "efficiency": _eval_efficiency(),
    }
    write_json(evaluation, DIRS["audit_logs"] / "evaluation_metrics.json")
    print(f"Report Quality Index: {report_quality_index}/100")
    print("Dimension scores (0-100):")
    for d, v in evaluation["dimension_scores_0_100"].items():
        print(f"  - {d:16s}: {v}")
    eff = evaluation["efficiency"]
    print(f"Efficiency: {eff.get('llm_calls', 0)} LLM calls, "
          f"{eff.get('total_tokens', 0)} tokens, "
          f"${eff.get('total_cost_usd', 0)}, "
          f"first-pass yield {eff.get('first_pass_yield', 0)}")
    print(f"Full metrics: {DIRS['audit_logs'] / 'evaluation_metrics.json'}")
    return evaluation


evaluation_metrics = run_evaluation()
try:
    _eval_rows = [{"dimension": d, "score_0_100": s}
                  for d, s in evaluation_metrics.get("dimension_scores_0_100", {}).items()]
    _eval_rows.append({"dimension": "REPORT QUALITY INDEX",
                       "score_0_100": evaluation_metrics.get("report_quality_index_0_100")})
    display(pd.DataFrame(_eval_rows))
except Exception:
    pass


Report Quality Index: 95.88/100
Dimension scores (0-100):
  - faithfulness    : 100.0
  - compliance      : 100.0
  - coherence       : 98.41
  - coverage        : 75.73
  - non_redundancy  : 98.34
Efficiency: 32 LLM calls, 455567 tokens, $0.0, first-pass yield 1.0
Full metrics: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\audit_logs\evaluation_metrics.json


,dimension,score_0_100
0,faithfulness,100.00
1,compliance,100.00
2,coherence,98.41
3,coverage,75.73
4,non_redundancy,98.34
5,REPORT QUALITY INDEX,95.88


In [43]:
# ============================================================
# CELL 22 — AUDIT SUMMARY
# PRODUCTION RULE: includes missing flags and section-generation scores.
# ============================================================

summary = {
    "pipeline_mode": PIPELINE_MODE,
    "forbid_invention": FORBID_INVENTION,
    "allow_partial_coverage": ALLOW_PARTIAL_COVERAGE,
    "policy": "Missing requirements are audit-only. The approved report must contain no missing-data or payload-unavailable wording.",
    "sections": {},
    "outputs": {name: str(path) for name, path in DIRS.items()},
}

for section in SECTIONS:
    slug = SECTION_SLUGS[section]
    coverage = coverage_by_section.get(section, [])
    missing_register = missing_registers_by_section.get(section, {})
    missing = missing_register.get("missing_requirements", [])
    approved_path = DIRS["approved"] / f"approved_{slug}.md"
    approved_json_path = DIRS["approved"] / f"approved_{slug}.json"
    approved_json = read_json(approved_json_path, default={}) if approved_json_path.exists() else {}
    section_score = approved_json.get("section_generation_score") or approved_json.get("approval", {}).get("section_generation_score") or {}

    summary["sections"][section] = {
        "requirements_total": len(requirements_by_section.get(section, [])),
        "coverage_counts": dict(Counter([c["coverage_status"] for c in coverage])),
        "missing_requirements_count": len(missing),
        "missing_requirement_ids": missing_register.get("missing_requirement_ids", [m.get("requirement_id") for m in missing]),
        "section_readiness_score_0_to_100": missing_register.get("section_readiness_score_0_to_100"),
        "section_generation_score": section_score,
        "approved_markdown_exists": approved_path.exists(),
        "approved_markdown_path": str(approved_path) if approved_path.exists() else None,
        "missing_requirements_path": str(DIRS["missing_requirements"] / f"missing_requirements_{slug}.json"),
    }

write_json(summary, OUTPUT_DIR / "generation_audit_summary.json")

summary_md = [
    "# Agentic IFRS Report Generation Audit Summary",
    "",
    f"- Pipeline mode: `{PIPELINE_MODE}`",
    f"- Forbid invention: `{FORBID_INVENTION}`",
    f"- Allow partial coverage: `{ALLOW_PARTIAL_COVERAGE}`",
    "",
    "## Policy",
    "",
    "The report contains only evidence-supported disclosures. Missing requirements and missing-data explanations are recorded in audit files only and must not appear in report prose.",
    "",
    "## Section summary",
    "",
]

for section, info in summary["sections"].items():
    score = info.get("section_generation_score") or {}
    summary_md.append(f"### {section}")
    summary_md.append(f"- Requirements total: {info['requirements_total']}")
    summary_md.append(f"- Coverage counts: `{info['coverage_counts']}`")
    summary_md.append(f"- Missing requirements count: {info['missing_requirements_count']}")
    summary_md.append(f"- Section readiness score: {info.get('section_readiness_score_0_to_100')}")
    summary_md.append(f"- Section generation score: {score.get('overall_section_generation_score_0_to_100') if isinstance(score, dict) else None}")
    summary_md.append(f"- Approved markdown exists: {info['approved_markdown_exists']}")
    summary_md.append("")

write_text("\n".join(summary_md), OUTPUT_DIR / "generation_audit_summary.md")
print("Saved audit summary:", OUTPUT_DIR / "generation_audit_summary.md")
display(pd.DataFrame(summary["sections"]).T)


Saved audit summary: C:\Users\BV426BP\Documents\IFRS Data\Reporting-Data-\notebooks\gen_data\generated_reports\agentic_ifrs_report\generation_audit_summary.md


,requirements_total,coverage_counts,missing_requirements_count,missing_requirement_ids,section_readiness_score_0_to_100,section_generation_score,approved_markdown_exists,approved_markdown_path,missing_requirements_path
General Requirements,108,"{'partially_covered': 59, 'covered': 43, 'not_...",6,"[IFRS_S1_7_C01, IFRS_S1_60_C01, IFRS_S1_73_C01...",67.13,"{'section_name': 'General Requirements', 'over...",True,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Governance,15,{'covered': 15},0,[],100.0,"{'section_name': 'Governance', 'overall_sectio...",True,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Strategy,70,"{'covered': 60, 'partially_covered': 10}",0,[],92.86,"{'section_name': 'Strategy', 'overall_section_...",True,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Risk Management,17,"{'covered': 10, 'partially_covered': 7}",0,[],79.41,"{'section_name': 'Risk Management', 'overall_s...",True,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
Metrics and Targets,151,"{'partially_covered': 11, 'covered': 128, 'not...",12,"[IFRS_S2_B60_C01, IFRS_S2_B62_C06, IFRS_S2_B62...",88.41,"{'section_name': 'Metrics and Targets', 'overa...",True,C:\Users\BV426BP\Documents\IFRS Data\Reporting...,C:\Users\BV426BP\Documents\IFRS Data\Reporting...
